# PIV-Q27-C1 v2.5A — CANDIDATE-FIRST VERIFIER-FEEDBACK REPAIR

> **Donor / baseline:** canonical v2.4G real run (`source SHA 50b9acb8…`, executed SHA `44934b20…`, evidence bundle SHA `387d4bb7…`). v2.4G established 16/16 standalone final programs + 16/16 fresh verifier executions with 0/16 all-train-exact. The terminal/finalizer path is therefore frozen; v2.5 targets semantic program correctness.
>
> **Single causal mechanism:** after candidate #1 fails fresh train verification, build compact *training-side* verifier feedback and make exactly **one** no-tool, non-thinking structured repair call. Fresh-verify candidate #2. Test execution remains embargoed until all training pairs are exact.
>
> **Feedback contract (v2.5):** per failed train pair report output/expected dimensions; for same-shape grids report wrong-cell count and a small mismatch sample; report the final exception line for execution failures; explicitly flag identity outputs (`output == input`). Feedback is tokenizer-bounded to <=600 tokens. Transport is verified with a canonical `strip()` hash + exact presence sentinel in the rendered repair prompt.
>
> **Important non-change:** ACT-BUDGET3072 keeps `ACTION_RESERVE_TOKENS=768`. Raw v2.4G evidence shows five late forced-close actions hitting exactly 768 tokens and ending `length/malformed_tool`, but changing 768→2048 would alter exploration allocation and become a second solver-policy intervention. v2.5 logs that truncation diagnostic and defers the reserve experiment to a separate follow-up.
>
> **Run discipline:** same model/runtime/C1 engine, frozen SMOKE8/order, LOW + BUDGET3072, same REPL/parser/verifier, same 12,288 generated-token ceiling. No reviewer, multi-sample, DSL, evolution, prompt search, seed diversification, PTQ, NVARC, prefix cache, or custom all-reduce.

---

# PIV-Q27-C1 v2.4G — ONE-SESSION ADAPTIVE TERMINAL-COMMIT SMOKE

> **v2.4G — per-task engine-cap isolation patch over v2.4F.** Real vLLM is expected to respect each request's `max_tokens`, but v2.4F treated a single over-cap RequestOutput as a whole-wave engine exception and terminalized every task in the batch. v2.4G keeps output-cardinality mismatch as a wave-fatal mapping error, while isolating a token-cap violation to the offending task only. The task is recorded infrastructure-invalid with the exact observed token count and requested cap; unaffected requests in the same wave continue. No model, prompt, budget, sampling, terminal-policy, verifier, or arm parameter is changed.


> **v2.4F — PREFLIGHT/runtime-validation patch over v2.4E.** The default ADAPTIVE_SMOKE solver policy is unchanged. This revision fixes the CPU PREFLIGHT terminal-policy assertion, adds explicit coverage for all terminal-trigger predicates, guards against engine outputs exceeding their requested token caps, and corrects TERMINAL_PILOT decision labels. ACT-BUDGET3072 intentionally remains 3,072 thinking tokens: under the 8,192 exploration cap this usually means two full forced-close turns plus at most a partial third; this is a known property of the terminal-policy package, not silently retuned here.


> **v2.4E — runtime-blocker patch over v2.4D.** Two independently reproduced pre-run failures are fixed:
> (1) the preload malformed-tool self-test now tests a *committed* action after `</think>`, while an
> independent regression asserts that unclosed-thinking tool-like text remains scratch `text`; and
> (2) `promotion_eligible_arms` is initialized in the PREFLIGHT branch so recommendation generation
> cannot raise `NameError`. No solver policy, model sampling, budgets, tasks, engine settings, or
> verifier semantics are changed.



> **v2.4D — pre-run correctness patch over v2.4C.** It preserves the one-session solver policy while fixing
> four confirmed scheduler failure paths: incomplete reasoning is handed to terminal commit through a bounded
> scratch note; exhausted exploration retries spend the protected terminal slot; REPL startup/runtime failures
> are propagated as infrastructure evidence; and Proxy48 authorization now requires the real promotion/compute gate.
> Terminal JSON framing is handled without mutating control-token-looking text inside `solution_code` strings.
>
> **v2.4C — final blocker patch over v2.4B.** It preserves the one-session solver policy while fixing two
> remaining evidence-validity issues found in the v2.4B re-audit: verifier infrastructure failures are
> propagated into task/arm infrastructure accounting, and tool feedback is token-truncated against the
> *actual rendered terminal prompt* so the protected 4,096-token finalizer capacity is a checked invariant.
> Partial PILOT2 evidence can no longer populate the matched recommended arm.

> **v2.4B — correctness revision of v2.4A.** Two defects in v2.4A made a real GPU run unusable and
> were invisible to the existing self-tests:
>
> 1. **Terminal artifact never parsed.** Sampling uses `skip_special_tokens=False`, so the decoded
>    finalizer text ends with `<|im_end|>`; `json.loads()` therefore failed on every *correct*
>    answer. The Cell 7 positive control hit it first, so engine bootstrap failed and no real
>    generation ran at all. Fixed in the shared parser now defined in Cell 6.
> 2. **The scoring gate crashed.** `compute_smoke_gate()` did `int(w["phase"])`, but the terminal wave
>    row carries `phase="terminal_finalizer"` → `ValueError`, killing all of Cell 10.
>
> The adaptive interface gate itself was also wrong in both directions (it passed a 100%-broken
> terminal parser and failed a run where the model solved during exploration). See
> `V24B_FIXES` in Cell 14 for the full list.


## Fast competition workflow: embedded safety checks → real positive control → 2-task diagnostic gate → automatic SMOKE8 continuation

**This revision supersedes the serial v2.4 `PREFLIGHT -> upload/review -> TERMINAL_PILOT -> upload/review -> SMOKE8` workflow.**

One Kaggle 4×L4 session now does all of the following without waiting for an external approval in the middle:

1. compact embedded CPU regression checks for the new terminal-policy invariants;
2. one score-excluded real-Qwen positive control for `enable_thinking=False` + no-tools + structured JSON;
3. both approved arms on diagnostic tasks `0934a4d8` and `cb2d8a2c`;
4. an automatic **interface gate** based only on execution integrity and terminal-stage reachability, not puzzle correctness;
5. if that gate passes, the same model session immediately continues the same two arms on the remaining frozen SMOKE8 tasks.

The diagnostic tasks do **not** have to solve. They only test whether the structural terminal path is alive.
If the interface gate fails, the notebook stops before spending the remaining SMOKE8 generation budget and still bundles all evidence.

Proxy-48 is not run inside this notebook. A completed, infrastructure-valid SMOKE8 with at least one fresh-verifier
all-train-exact program and at least one usable test prediction can authorize the next Proxy-48 experiment.

---
# PIV-Q27-C1 v2.4 — TERMINAL-COMMIT PILOT — Protected Finalization + Fresh Exact Verifier
## ARC-AGI-2 | Kaggle 4× NVIDIA L4 | Offline | Qwen3.8-27B-FP8 TP=4

**Purpose of v2.4:** test one solver-policy package only: protect a terminal code-generation budget so exploration cannot consume the entire 12,288-token task budget, then switch structurally to a no-tools, non-thinking terminal commit that emits one standalone `solution(grid)` candidate for the existing fresh verifier.

### v2.4 causal scope
- Total generated-token ceiling remains **12,288**.
- Exploration is hard-capped at **8,192** generated tokens.
- **4,096** tokens are protected for exactly one terminal finalizer request.
- Finalizer uses Qwen `enable_thinking=False`, advertises **no Python tool**, and requests a JSON object containing `solution_code`.
- Exploration prompts, C1 engine settings, REPL, verifier, reasoning arms, and exploration seed policy are otherwise frozen from v2.3.
- This is a **terminal-policy package** experiment; it does not claim to isolate reserve size alone.
- Real pilot tasks are exactly `0934a4d8` and `cb2d8a2c`, for both `ACT-LOW` and `ACT-BUDGET3072`.
- Pilot results can justify a frozen SMOKE8 follow-up, but **cannot authorize Proxy-48 directly**.

### Historical v2.4 serial sequence
Superseded by the v2.4A one-session adaptive workflow above.

---
# PIV-Q27-C1 v2.3 — REAL SOLVER QUALITY SMOKE — Qwen3.8-27B Stateful Python REPL + Fresh Exact Train Verifier
## ARC-AGI-2 | Kaggle 4× NVIDIA L4 | Offline | Qwen3.8-27B-FP8 TP=4

This notebook is the **safe-recovery-hardened preflight for the next decisive solver-quality experiment** after the validated v1.9d reasoning-activation probe and the v1.2.4 throughput probe.

### Frozen causal question

> With the v1.9d PIV solver mechanics frozen, does changing only the serving engine from eager execution to CUDA-graph/hybrid execution (`enforce_eager=True -> False`) allow the two activation-approved reasoning arms (`ACT-LOW`, `ACT-BUDGET3072`) to produce executable, fresh-verifier `all_train_exact` ARC programs and valid test outputs on the same frozen SMOKE8 tasks?

### Canonical donor / evidence

- Donor notebook SHA256: `007c8d68559c39a948dfef9d571fbd2e5da7335492a6f3330645db892648b525`
- Donor lineage: `PIV-Q27-ACT-v1.9d`
- Activation probe run: `20260915T124541Z`
- Approved quality-smoke arms: `ACT-LOW`, `ACT-BUDGET3072`
- Throughput evidence ZIP SHA256: `c34fd3736dd9d2565ef681c6fd1936bb7eb1f99b3a95d222eda419049cd3e2aa`
- Serving intervention: **only** `enforce_eager=False`; custom all-reduce OFF; prefix cache OFF.
- The old greedy-prefix numerics threshold is **not** a solver-quality gate. Real fresh all-train verification and exact ARC outputs are the correctness authority.

### Decision hierarchy

Integrity → executable final program → fresh all-train-exact → valid test execution → competition output-level exact score (strict all-query task exact kept as a diagnostic) → compute efficiency/stability.

Tool engagement is diagnostic only. `ACT-BUDGET3072` is **not gated on ACT-LOW succeeding**.

### Run discipline

Historical v2.3 note: the old quality-smoke sequence has been superseded by the v2.4 `PREFLIGHT -> TERMINAL_PILOT` sequence above.


### Source basis

Primary source snapshot: `gutfeeling/arc-agi-2-submission` at commit
`b3d4494892c6d85fa2a35d548271fb02ea3e191b`.

The P0 mechanism is source-faithful at the **semantic/invariant level**:

1. one reasoning-model lineage per ARC task;
2. one persistent Python REPL namespace across tool turns;
3. the puzzle is preloaded into the REPL and protected from overwrite/whole-puzzle printing;
4. Python stdout/errors/timeouts become model feedback;
5. the model must finish with one self-contained `solution(grid)` program;
6. the final program is executed again in a **fresh process** and must match **every training pair exactly** before any test output is admitted;
7. test grids are produced by execution of the verified program, never transcribed from prose.

### Explicit LOCAL adaptations for Kaggle 4×L4

- Direct `vllm.LLM.generate(...)` + Qwen3.8 native tool-capable chat template instead of an OpenAI/Responses server.
- Lightweight persistent local Python subprocess instead of Docker/IPyBox.
- Hard process timeout + restricted dangerous filesystem/process APIs.
- Bounded P0 budget: one lineage, 6 executed Python calls, at most 12 model turns, up to the remaining 12,288-token task budget per turn (context-bounded), and 12,288 generated tokens/task total.
- Tool output sent back to Qwen is compact; full bounded raw tool events are checkpointed.
- No reviewer, no second lineage, no deterministic PTQ repair loop, no NVARC seeding, no judge and no evolution. Those are later experiments.

v2.4 sequence: `PREFLIGHT -> TERMINAL_PILOT -> frozen SMOKE8 if the interface is clean`. Proxy-48 remains blocked.


### v2.3 safe-recovery and infrastructure-accounting hardening (mechanism unchanged)

Independent review of v2.2 found two related P1 evidence-integrity bugs in the phase-2 exception path. v2.3 fixes them without changing prompts, solver budgets, task selection, reasoning policies, verifier semantics, or the C1 serving intervention:

1. a phase-2 engine exception may salvage phase-1 code **only through the normal length-salvage contract**: the phase-1 text must have a closed `</think>` action region containing a closed, AST-valid `solution()` block. Draft code inside unclosed reasoning is never promoted to a final candidate;
2. engine failure is tracked independently from candidate quality. A recovered train-perfect/test-usable candidate remains useful quality evidence, but the arm is infrastructure-invalid and cannot pass the compute/promotion gate or produce a matched winner;
3. phase-2 output tokens are reported as **unknown/unobserved**, not zero, when `generate()` raises before returning `RequestOutput`; phase-1 observed tokens remain preserved as a lower bound;
4. `provisional_best_valid_arm` is suppressed whenever matched comparison is infrastructure-invalid.

PREFLIGHT includes regressions for unclosed-think draft rejection, legitimate closed-action salvage under infrastructure failure, infrastructure-gate invalidation, and unknown phase-2 token accounting.

### v2.2 failure-path hardening (mechanism unchanged)

Independent review of v2.1 found four remaining evidence-integrity gaps. v2.2 fixes them without changing the solver mechanism or the C1 serving intervention:

1. model import/allocation/warmup failures are captured as bootstrap evidence so downstream reporting/bundling can still finish;
2. a forced-arm phase-2 `generate()` exception preserves phase-1 token/finish evidence, does not destroy already-complete phase-1 final programs in the same batch, and terminalizes only genuinely affected continuations;
3. verifier `started` is true only after a subprocess is actually created; launch failures are distinguishable from started-process failures/timeouts;
4. recommendation logic refuses to call an infrastructure-invalid or incomplete arm a no-quality result.

Verifier timeout semantics are explicit: `VERIFIER_TIMEOUT_SEC` applies **per fresh subprocess stage** (one train stage plus one stage per test input), not to the candidate as a single aggregate 20-second wall-time cap. Any matched C0/C1 comparison must keep this verifier contract identical.

### v1.1 source/donor re-audit hardening

After re-comparing P0 against the frozen Q1 donor and the audited PiVOTools/Qwen sources, v1.1 fixes four issues without adding a new solver mechanism:
1. use the full source agentic prompt family rather than a shortened hybrid prompt;
2. reject length-truncated generations before parsing them as a tool/final answer;
3. use per-request generation caps inside a batch so one near-context-limit task cannot starve the whole wave;
4. harden the lightweight local worker against host/Kaggle filesystem and process/network access.

It also validates the actual mounted Qwen multi-turn tool-history rendering during PREFLIGHT.


### v1.2 additional verifier/accounting correction

A second source-level pass found that PiVOTools verifies training examples first and executes test inputs in a separate step. v1.1's single verifier exception boundary could erase a genuinely train-perfect result if test execution later failed. v1.2 separates those states and counts train-perfect rate over the entire frozen active task set, including missing/failed tasks as non-passes.


### v1.3 external-review verification

An independent review of the earlier v1.0 was checked against this v1.2 notebook and the audited PiVOTools source.
Some findings had already been fixed in v1.2 (per-request SamplingParams, multi-tool handling, finish-reason gate,
full source prompt, Qwen history probe, train-vs-test verifier state). The remaining real issues are fixed here:
kernel-level `copy` import, source-style bounded retries, a less starvation-prone per-turn ceiling, source-tolerant grid
normalization, per-test partial output retention, budget-aware finalization, think-aware action parsing, dead-worker restart,
puzzle reset hardening, narrower dunder blocking, `py` fences, and a scripted CPU preflight agent-loop test.

Two review suggestions are intentionally NOT adopted:
- no raw-text continuation after truncation (the PiVOTools source retries the agentic call; it does not splice partial text);
- no switch from XHIGH to MEDIUM (PIV-Q27-P0 is the planned XHIGH causal experiment).


### v1.4 second external-review re-audit

This revision corrects the remaining mismatch between the local vLLM loop and PiVOTools' two-level retry semantics. Incomplete/length-capped steps are retried first with the same conversation and persistent REPL state; only after step retries are exhausted may the outer source-style attempt restart.

The local per-turn ceiling now equals the remaining total P0 generation budget (subject to context headroom), the closest bounded analogue to the source configuration's lack of an explicit per-step output-token cap.

P0 still uses one Qwen XHIGH lineage and the same 12,288 total generated-token budget. `SMOKE_GPU` now uses the same generation/tool budget as `PROXY48`, so it is a real viability gate: if first-turn length truncation exceeds 50%, do not launch Proxy-48 without revisiting budget/effort/context policy in a separate revision.


### v1.5 third external-review re-audit

The v1.4 first-turn-length SMOKE gate was too weak. v1.5 gates on terminal budget death and empirical wall-clock projection from real generation waves.


### v1.6 deep-audit fixes (mechanism unchanged)

Third-review fixes, all LOCAL harness hygiene — the solver mechanism (one Qwen XHIGH lineage, persistent REPL,
6 executed Python calls, 12 turns, 12,288 total generated tokens, fresh all-train exact verifier, frozen Proxy-48,
Q1-frozen engine) is unchanged:

1. **Safety reserve 30 → 60 min + per-wave projected-time pre-check.** A single wave can run for the remaining
   budget (up to 12,288 tokens ≈ 33 min at the Q1-measured step latency), so a wave starting just before the old
   30-min deadline could cross the 12 h Kaggle limit and lose `/kaggle/working`. The loop now projects each wave
   from measured wave timings (`wave_metrics.jsonl`) and refuses to start one that would cross the deadline
   (`global_deadline_projected`, resumable).
2. **Stratified SMOKE8.** The previous selector took all four Proxy-48 tier-3 tasks (50% vs 8% in Proxy-48), which
   biased the feasibility gate. SMOKE8 now mirrors Proxy-48: 1 tier-3, 2 tier-2, 5 tier-1 (incl. easy_static,
   hard_tier1, multi_query, dynamic), deterministic in Proxy-48 order.
3. **Structured malformed-tool retry counter** resets only after a VALID structured tool turn is accepted (the
   early reset let the max-2 inner retry be bypassed).
4. **Guard-rejected snippets are not charged** to the 6-call tool budget (they never reached Python); fenced
   ```` ```python ```` parameter payloads are unwrapped before the guard.
5. **Tool output is sanitized** before it re-enters the prompt (`<|im_end|>`, `<tool_response>` … cannot break
   the chat structure).
6. **Length-capped final turns with a CLOSED, parseable `solution()` block are accepted** (LOCAL; the block is
   demonstrably complete, retrying would only burn budget).
7. `malformed_tool` requires an actual `<tool_call>`; prose merely mentioning `<function=` is no longer a retry.
8. SMOKE gate additionally requires complete terminal coverage and reports measured step latency + projected
   max wave minutes; `global_deadline_projected` counts as an infrastructure failure.
9. Stale `per-turn 4096` provenance string corrected.
10. PREFLIGHT now drives the **real** `run_real_piv()` scheduler with a scripted fake LLM (happy path, budget
    death, malformed-tool retry accounting, guard-rejection accounting, length salvage, deadline pre-check).

Optional throughput switch (default OFF to keep the Q1 engine fingerprint): `ENABLE_PREFIX_CACHING`.

### v1.7 cross-audit hardening

Independent re-audit confirmed the v1.6 deep-audit changes, but found one scheduler accounting bug:
inside a multi-call assistant turn, an early guard-rejected call did not increment `tool_calls`, yet later calls
were denied by their original positional index. Thus a rejected call could indirectly consume a valid remaining
execution slot. v1.7 makes the budget test dynamic per call.

Two defensive changes do not alter the PIV mechanism:
- SMOKE terminal coverage also requires unique task IDs;
- projected-deadline timing never becomes less conservative than the Q1-derived 0.20 s/output-token prior.


### v1.8/v1.9 — Reasoning-Activation Ablation (ACT)

The v1.7 SMOKE (frozen reference **ACT-XHIGH**) showed 8/8 tasks spending the entire 12,288-token budget inside the
first `<think>` without ever reaching a tool call (tool engagement 0.00, budget death 1.00). Qwen3.8's template places
tool calls **after** `</think>`, so under XHIGH the model must finish a full solve before it can act. This revision
keeps the PIV mechanism (persistent REPL, 6 executed calls, 12 turns, 12,288 total generated tokens, fresh all-train
exact verifier, frozen stratified SMOKE8, Q1-frozen engine) and ablates **only the reasoning-to-action policy**:

| arm | single change vs ACT-XHIGH |
|---|---|
| `ACT-LOW` | `reasoning_effort="low"` (template injects "keep your thinking brief…") |
| `ACT-MEDIUM` | `reasoning_effort="medium"` (template injects no reasoning instruction) |
| `ACT-BUDGET3072` | `xhigh` + Qwen's documented two-stage thinking budget: after 3,072 thinking tokens without `</think>`, the stop-thinking sentence + `</think>` is injected (token-exact continuation of the same request) and the model gets ≤ `ACTION_MAX_TOKENS` to act |

All arms run in one session (one model load) on the same 8 tasks, same prompts, same REPL/verifier, same seeds policy;
each arm has its own lineage, records, traces and work dir. `ACT-XHIGH` is **not rerun** — its v1.7 numbers are the frozen
reference row in `arm_comparison.csv`.

**Pre-registered measurement contract** (`MEASUREMENT_CONTRACT`, written to the bundle before any result is read) and
**pre-registered verdict tiers**: Activation (tool execution ≥ 50% of tasks) → Productivity (final program ≥ 25%, no
infrastructure failure) → Quality (all-train-exact > 0), plus the compute gate (budget death ≤ 50%, projected Proxy-48 with
20% margin ≤ 10 h, complete unique terminal coverage). `recommendation.json` applies these rules mechanically.

The report separates **PROVEN BY PREFLIGHT** (harness correctness, CPU, fake model) from **MEASURED ONLY ON GPU**
(model behaviour). Nothing in PREFLIGHT predicts what the model will do.

### v1.9 — cross-audit fixes (mechanism unchanged)
1. Byte-exact Qwen-documented early-stopping text (`"\n\nConsidering … now.\n</think>\n\n"`).
2. Qwen documented three-way flow: phase 1 `stop` → complete; `length` without `</think>` → inject stop text; `length` **with**
   `</think>` (action cut at the think cap) → token-exact continuation **without** injection (`continuation_mode`).
3. Token-exact continuation is mandatory (`prompt_token_ids`); no text re-render fallback. Warmup validates the token-prompt API on real vLLM.
4. Think/post tokens split exactly on the `</think>` token id; injected stop text never counted as thinking.
5. Phase-2 request uses a deterministic phase-specific seed (`seed + 1,000,000`).
6. Arm-independent wall-clock projection (`ACT_FIXED_SETUP_ELAPSED_SEC` measured once).
7. Ranked winner selection (`recommended_arm` from the ranked list).
8. PROXY48 is locked: `PROXY48_APPROVAL` must carry the SMOKE run id + `recommendation_sha256` from `recommendation.json`.
9. `repeated_tool_calls`, AST stateful-reuse metric, `action_cap_hit_rate`, phase-1/phase-2/combined throughput reported.
10. Template-level assert that `reasoning_effort="high"` is rejected. Frozen ACT-XHIGH reference bound to bundle SHA + run id.
11. Fake-LLM scheduler test: real tokenizer ids in fake outputs; two new lineages (natural-close-cut continuation, repeated tool code).
12. `ACTION_RESERVE_TOKENS=768`: in the forced arm, thinking never consumes the last 768 tokens of the task budget, so the final turn
    always keeps room for an action (SMOKE emulation showed 8/8 budget deaths without it: 3×(3,072+action) left 0 action tokens).
13. `stop_without_think_close` diagnostic (phase 1 ended with EOS inside thinking).
14. **`PROBE_GPU` profile (v1.9c)** — Kaggle-only cheap tier before SMOKE: same 8 tasks / prompts / REPL / verifier, budget
    3,072 per turn · 5,120 total · 2 calls · 2 turns. Prints a per-arm quick readout the moment each arm ends and writes
    `probe_verdict.json` (`smoke_go_arms`). SMOKE_GPU reads `PROBE_APPROVAL` and runs only the arms that showed activation.
    Worst case ≈ 50 min GPU incl. model load; typical far less. Re-runnable in an interactive session (`PROBE_ARMS`).
15. v1.9d (after reading the Kaggle PREFLIGHT bundle 20260915T113233Z): `stop_without_think_close` now reaches the records;
    fake lineage G exercises and asserts the stateful-reuse metric; injected-stop-token assertion uses `STOP_THINKING_TEXT`;
    fake-LLM scheduler traces are bundled under `fake_llm_selftest/<arm>/`.


### v2.1 correctness hardening before GPU quality smoke

This revision does **not** add a solver mechanism. It fixes measurement and failure-handling issues found by independent CPU counterexamples:

1. Proxy promotion now requires verified all-train-exact **and at least one usable test prediction**; tool engagement is diagnostic only.
2. Competition-equivalent quality is the exact rate over **task test outputs**, matching Kaggle scoring. Strict all-query task success is retained only as a diagnostic.
3. vLLM `generate()` exceptions are converted into terminal failed records and an engine-error artifact instead of aborting evidence generation.
4. `verifier_executed` means a verifier subprocess really started; guard rejection is tracked separately.
5. Train verification and test execution are separate fresh-process stages, so a test timeout cannot erase already established train-exact evidence.
6. Phase-2 request seeds are logged explicitly; empty candidate registries retain their schema; the donor prompt runtime text is restored byte-faithfully.

A synchronous CUDA/vLLM call that hard-hangs inside native code still cannot be reliably interrupted from this in-process notebook. That residual risk is recorded explicitly and is not disguised as solved.


In [ ]:
# =====================================================================================
# CELL 1 — PIV-Q27-P0 CONFIG + OFFLINE ENVIRONMENT + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import copy
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "PIV-Q27-C1-v2.5"
GENERATION_CONTRACT_VERSION = "piv-q27-c1-candidate-first-repair-v2.5"
PARSER_CONTRACT_VERSION = "piv-q27-toolxml+terminal-json+repair-json-v2.5"
PROMPT_CONTRACT_VERSION = "piv-q27-source-agentic-prompt-v1.9"

# C1 v2.3 provenance: donor and upstream evidence are frozen inputs to this derivative.
DONOR_NOTEBOOK_SHA256 = "007c8d68559c39a948dfef9d571fbd2e5da7335492a6f3330645db892648b525"
ACTIVATION_PROBE_RUN_ID = "20260915T124541Z"
ACTIVATION_PROBE_BUNDLE_SHA256 = "d400bbb1cbcf3bc05af58b56b5c85040f8362b13e099809c1f49a25d43b2d26a"
THROUGHPUT_SMOKE_RUN_ID = "20260916T091344Z"
THROUGHPUT_EVIDENCE_SHA256 = "c34fd3736dd9d2565ef681c6fd1936bb7eb1f99b3a95d222eda419049cd3e2aa"
C1_EXPECTED_LONG8_TPS = 129.173172
C0_FROZEN_LONG8_TPS = 48.300656

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}


# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "ADAPTIVE_SMOKE"   # default: one-session real GPU workflow

PIV_SOURCE_COMMIT = "b3d4494892c6d85fa2a35d548271fb02ea3e191b"
PIV_SOURCE_REPO = "https://github.com/gutfeeling/arc-agi-2-submission"
PIV_EXTERNAL_SOURCES = {
    "technical_article": "https://pivotools.github.io/pivotools-quarto-blog/posts/agentic_coding_arc_agi/",
    "audited_repo_snapshot": f"https://github.com/gutfeeling/arc-agi-2-submission/tree/{PIV_SOURCE_COMMIT}",
    "public_experiment_traces": "https://huggingface.co/datasets/arcagi2/arcagi2-agentic-coding-publication",
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "qwen_chat_template": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8/blob/main/chat_template.jinja",
    "vllm_019_llm_api": "https://docs.vllm.ai/en/v0.19.0/api/vllm/entrypoints/llm/",
}
PIV_SOURCE_FILES = {
    "clients.py": {"sha": "34a85ed34001923a5efae715f509e5aa7e8e235f", "role": "persistent model/tool loop + reasoning/tool history"},
    "solver.py": {"sha": "cf2ef554cc389322a5689988b300dc2bc05bf36a", "role": "one code turn -> fresh all-train verification -> test execution"},
    "turn.py": {"sha": "fe4dfc69c5963a56dfba84d0bd49821ae0d2e0bc", "role": "last code block + required solution() artifact"},
    "tools/repl_tool.py": {"sha": "691839335b1e65ba0bd2e4b8b84b70c50d559683", "role": "python tool, errors/timeouts as feedback, puzzle protection"},
    "utils/puzzle_utils.py": {"sha": "7e6ec3034e0dc2f94fba3aa9f1d1d4a0896120b4", "role": "all-train/all-test-input serialization; test outputs hidden"},
    "utils/solving_utils.py": {"sha": "197be842b3d6fa08e4e98066e801dc8cf2bc8327", "role": "fresh sandbox train check and test execution"},
    "utils/code_templates/check_solution_on_training_examples.py": {"sha": "c13ea5b89360ea997a30a1eabe2ccb4a67489b44", "role": "deep-copy input + exact train equality"},
    "solver/prompts/agentic_coding_solver.txt": {"sha": "dab87e08bbe3f071920f13eb98f4a4d6f03ada10", "role": "ARC priors, preloaded puzzle, self-contained solution()"},
    "solver/prompts/system_prompt.txt": {"sha": "265131a50fb42deebe0a8da353f466b11d6d7d6b", "role": "stateful Python tool contract"},
    "solver/config/gpt_oss_120b_high.py": {"sha": "8fd563ddcb10962c7564fbfa8cbf3424a7a437d7", "role": "source agentic config example"},
    "sandbox/ipybox_sandbox.py": {"sha": "195e0e66ad7328a4f823974f04778b639a16412a", "role": "source Docker/IPyBox persistent REPL implementation; semantics only, not ported literally"},
    "sandbox/base.py": {"sha": "517f5794ff0cac64024e47c184d116b1f36577ea", "role": "source stateful REPL/Sandbox interface and fresh run_cells semantics"},
    "evaluation/evaluate.py": {"sha": "8b48fb4482dc7d9772ac010ba6dd81f6cc15e0a4", "role": "source multi-sample evaluation / majority aggregation; not copied into this single-sample smoke"},
    "utils/code_templates/produce_output_grid.py": {"sha": "b6ab435ca4f7d77414764e3a4400bb8c6dbead5e", "role": "execute verified solution on all test inputs"},
}

PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "TERMINAL_PILOT": {
        "use_model": True, "task_set": "PILOT2", "active_tasks": 2,
        "agent_batch_size": 2, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_TERMINAL_PILOT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": True,
    },
    "ADAPTIVE_SMOKE": {
        "use_model": True, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_ADAPTIVE_SMOKE_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": True,
    },
}
RUN_PROFILE = str(RUN_PROFILE).strip().upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
AGENT_BATCH_SIZE = int(PROFILE["agent_batch_size"])
MAX_TOOL_CALLS = int(PROFILE["max_tool_calls"])
MAX_MODEL_TURNS = int(PROFILE["max_model_turns"])
MAX_RETRIES = int(PROFILE.get("max_retries", 2))

# =====================================================================================
# v1.9d REASONING-ACTIVATION ARMS (solver mechanics frozen)
# =====================================================================================
# Official Qwen3.8 template values are xhigh / medium / low. 'high' is invalid for this model.
STOP_THINKING_SENTENCE = "Considering the limited time by the user, I have to give the solution based on the thinking directly now."
STOP_THINKING_TEXT = "\n\n" + STOP_THINKING_SENTENCE + "\n</think>\n\n"
assert STOP_THINKING_TEXT == "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
ARM_CONFIG = {
    "ACT-LOW":        {"reasoning_effort": "low",    "think_policy": "natural",      "think_budget": None},
    "ACT-MEDIUM":     {"reasoning_effort": "medium", "think_policy": "natural",      "think_budget": None},
    "ACT-BUDGET3072": {"reasoning_effort": "xhigh",  "think_policy": "forced_close", "think_budget": 3072},  # v2.4F: intentionally frozen; 8,192 exploration tokens typically allow ~2 full forced turns + partial third
}
ACTION_MAX_TOKENS = 2048
ACTION_RESERVE_TOKENS = 768

# v2.4 terminal-commit policy. Total task budget is unchanged; exploration may never consume this reserve.
FINAL_RESERVE_TOKENS = 4096
EXPLORATION_TOKEN_CAP = 12288 - FINAL_RESERVE_TOKENS
FINALIZER_CONTEXT_MARGIN_TOKENS = 512
FINALIZER_CONTEXT_RESERVE_TOKENS = FINAL_RESERVE_TOKENS + FINALIZER_CONTEXT_MARGIN_TOKENS
FINALIZER_MAX_TOKENS = FINAL_RESERVE_TOKENS
FINALIZER_TURN_SLOTS = 1
TERMINAL_HANDOFF_MAX_TOKENS = 1024  # latest incomplete-reasoning tail, fit against full finalizer reserve

# v2.5 candidate-first repair policy. Total generated-token ceiling stays 12,288; repair uses only leftover tokens.
REPAIR_ROUNDS = 1
REPAIR_FEEDBACK_MAX_TOKENS = 600
REPAIR_MAX_TOKENS = FINALIZER_MAX_TOKENS  # actual cap = min(leftover total budget, context headroom, 4096)
REPAIR_SEED_OFFSET = 3_000_000
REPAIR_POLICY_ENABLED = True
# v2.5A PRE-REGISTRATION (measured on the frozen v2.4G run 20260918T104631Z, not a bug):
# the repair round draws from the SAME TOTAL_GENERATED_TOKEN_CAP as exploration, so its cap is
# min(REPAIR_MAX_TOKENS, leftover budget, context headroom). Observed leftovers were 1,229-8,839 (ACT-LOW)
# and 2,175-4,315 (ACT-BUDGET3072). For ACT-LOW 271d71e2 / 2c181942 / 16b78196 the leftover (1,841 / 1,229 /
# 1,305) is SMALLER than candidate #1's own JSON size (2,254 / 2,866 / 2,790), so those repairs cannot even
# re-emit a program of the same length and are expected to end as repair_output_parse_failed (finish=length).
# Read any "repair helped k/N" result against this ceiling; a separate repair budget is a future decision.
REPAIR_BUDGET_PREREGISTRATION = {
    "repair_shares_total_generated_token_cap": True,
    "cap_formula": "min(REPAIR_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP - generated_tokens, MAX_MODEL_LEN - prompt_tokens - CONTEXT_SAFETY_TOKENS)",
    "v24g_leftover_tokens": {"ACT-LOW": [1229, 1305, 1841, 2403, 2888, 3107, 3256, 8839],
                             "ACT-BUDGET3072": [2175, 2299, 2661, 2728, 2788, 2946, 3425, 4315]},
    "v24g_expected_structural_truncations": ["ACT-LOW/271d71e2", "ACT-LOW/2c181942", "ACT-LOW/16b78196"],
}
V24G_CANONICAL_SOURCE_SHA256 = "50b9acb8b62b3c1860914f562c5154dd58e8906b3342cd8ad6d8875fdc882769"
V24G_EXECUTED_SHA256 = "44934b207807f2b4a870304205b579f78daea3f23b64720aa6fec73455ffb66f"
V24G_EVIDENCE_BUNDLE_SHA256 = "387d4bb7383c9e7917c6deff5d2999e0ab83ff9b105d1487aa7dcd7a8cb56837"
# Deliberately frozen for the repair-only experiment. v2.4G observed five BUDGET late phase-2 768-token length hits;
# changing this here would create a second solver-policy intervention.
assert ACTION_RESERVE_TOKENS == 768
TERMINAL_POLICY_ENABLED = (RUN_PROFILE in {"TERMINAL_PILOT","ADAPTIVE_SMOKE"})
TERMINAL_PILOT_TASK_IDS = ["0934a4d8", "cb2d8a2c"]
FINALIZER_JSON_SCHEMA = {
    "type": "object",
    "properties": {"solution_code": {"type": "string"}},
    "required": ["solution_code"],
    "additionalProperties": False,
}
FINALIZER_SAMPLING = {
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0.0,
    "repetition_penalty": 1.0,
}
TERMINAL_FINALIZER_INSTRUCTION = """[TERMINAL COMMIT]
Exploration is over. You have no Python tool and must not continue analysis or ask for more tool use.
Return exactly one JSON object with the single key "solution_code".
The value must be a complete, standalone Python program defining `solution(grid)` and must not depend on variables,
functions, imports, or state that existed only inside the earlier REPL. Do not return Markdown or prose."""
assert EXPLORATION_TOKEN_CAP + FINAL_RESERVE_TOKENS == 12288

# Frozen real-model activation result from v1.9d PROBE_GPU.
PROBE_APPROVAL = {
    "probe_run_id": ACTIVATION_PROBE_RUN_ID,
    "arms": ["ACT-LOW", "ACT-BUDGET3072"],
}
QUALITY_SMOKE_ARMS = ["ACT-LOW", "ACT-BUDGET3072"]
assert PROBE_APPROVAL["probe_run_id"] == "20260915T124541Z"
assert QUALITY_SMOKE_ARMS == PROBE_APPROVAL["arms"]
assert "ACT-MEDIUM" not in QUALITY_SMOKE_ARMS

# PREFLIGHT remains available for debugging, but ADAPTIVE_SMOKE is the default competition workflow.
ARMS = list(ARM_CONFIG) if RUN_PROFILE == "PREFLIGHT" else list(QUALITY_SMOKE_ARMS)
IS_PROBE = False
for _a in ARMS:
    if ARM_CONFIG[_a]["reasoning_effort"] not in ("xhigh", "medium", "low"):
        raise ValueError(f"{_a}: reasoning_effort must be one of xhigh/medium/low for the Qwen3.8 template")

# Frozen reference row = v1.7 SMOKE_GPU (run 20260914T122911Z). Not rerun; used only for deltas/comparison.
FROZEN_XHIGH_REFERENCE = {
    "bundle_sha256": "7435b795691d1542100f7f5cd0764ebcdacc61be6128f236c845f10eb22b143f",   # PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip (v1.7)
    "run_id": "20260914T122911Z", "notebook": "experiment-1-arc-agi-2-with-qwen-3-8 (11).ipynb",
    "task_ids": ["0934a4d8","16b78196","2c181942","271d71e2","136b0064","cb2d8a2c","291dc1e1","28a6681f"],
    "reasoning_effort": "xhigh", "think_policy": "natural", "think_budget": None, "tasks": 8,
    "tool_engagement_rate": 0.0, "final_program_rate": 0.0, "all_train_exact_rate": 0.0, "output_weighted_exact": 0.0,
    "budget_death_rate": 1.0, "infrastructure_failure_rate": 0.0, "first_turn_length_rate": 1.0,
    "natural_close_rate_turns": 0.0, "forced_close_rate_turns": 0.0, "think_tokens_per_turn_median": 12288,
    "executed_calls_per_task_mean": 0.0, "generated_tokens_per_task_mean": 12288.0,
    "sec_per_output_token": 0.16014132123575847, "projected_proxy48_hours_20pct": 4.056694742527999,
    "compute_gate_pass": True, "activation_pass": False, "productivity_pass": False, "quality_signal": False,
    "promotion_candidate": False, "tier_reached": "none",
}
VERDICT_THRESHOLDS = {"activation_tool_engagement_rate": 0.50, "productivity_final_program_rate": 0.25,
                      "compute_budget_death_rate_max": 0.50, "compute_projected_hours_20pct_max": 10.0}

# Pre-registered measurement contract: every effect below is computed after the run from records/traces/waves;
# nothing is added post hoc. 'post_hoc' items need trace inspection and are reported by the external analysis.
MEASUREMENT_CONTRACT = {
 "L1_thinking_behavior":  {"metrics": ["natural_think_close_rate_turns","forced_think_close_rate_turns","think_tokens_per_turn","think_tokens_per_task","first_turn_finish_reasons","tasks_with_any_natural_close"], "computed_by": "notebook"},
 "L2_tool_activation":    {"metrics": ["tool_engagement_rate_tasks","tool_turn_rate_turns","executed_calls_total","executed_calls_per_task","first_tool_turn","first_tool_generated_tokens","tasks_never_calling_tool"], "computed_by": "notebook"},
 "L3_tool_usefulness":    {"metrics": ["tool_ok_total","tool_exception_total","guard_rejections_total","tool_timeouts_total","budget_denials_total","puzzle_keys_only_calls","duplicate_tool_code_rate"], "computed_by": "notebook"},
 "L4_stateful_benefit":   {"metrics": ["worker_restarts_total","tasks_with_2plus_calls","stateful_reuse_events_total","tasks_with_stateful_reuse","repeated_tool_calls_total"], "computed_by": "notebook"},
 "L5_reason_action_transitions": {"metrics": ["transition_counts","turns_per_task","turn_kind_totals"], "computed_by": "notebook"},
 "L6_program_production": {"metrics": ["final_program_rate","final_programs","syntax_valid_program_rate","salvaged_final_on_length_total","final_code_chars"], "computed_by": "notebook"},
 "L7_program_correctness":{"metrics": ["verifier_execution_rate","all_train_exact_rate","all_train_exact_tasks","train_exact_pairs","verifier_status_counts"], "computed_by": "notebook"},
 "L8_test_execution":     {"metrics": ["test_execution_ok_tasks","any_test_output_ok_tasks","valid_grid_rate_outputs"], "computed_by": "notebook"},
 "L9_arc_quality":        {"metrics": ["competition_output_exact_rate","all_queries_exact_task_rate","task_normalized_exact","solved_task_ids_strict"], "computed_by": "notebook"},
 "L10_budget_behavior":   {"metrics": ["generated_tokens_per_task","generated_tokens_per_turn","injected_stop_tokens_total","budget_death_rate","tokens_at_budget_death","context_headroom_deaths","stop_reason_counts"], "computed_by": "notebook"},
 "L11_wall_clock":        {"metrics": ["wave_runtime","projected_proxy48_total_hours","projected_proxy48_total_hours_20pct_margin"], "computed_by": "notebook"},
 "L12_batch_scheduler":   {"metrics": ["waves","active_count_per_wave","idle_tail_fraction_per_wave"], "computed_by": "notebook"},
 "L13_forced_close_effects": {"metrics": ["forced_turns","post_close_action_counts","natural_turn_action_counts","phase2_skipped_reasons","phase2_tokens"], "computed_by": "notebook"},
 "L14_retry_effects":     {"metrics": ["step_retries_total","source_retries_total","tokens_spent_in_retried_turns","retry_reason_counts"], "computed_by": "notebook"},
 "L15_parser_interface":  {"metrics": ["malformed_tool_turns","text_turns_no_action","salvaged_final_on_length"], "computed_by": "notebook"},
 "L16_sandbox_safety":    {"metrics": ["guard_rejections","tool_timeouts","worker_restarts"], "computed_by": "notebook"},
 "L17_diversity":         {"metrics": ["unique_final_code_hashes","unique_predicted_grids","unique_tool_code_hashes_per_task"], "computed_by": "notebook"},
 "L18_eda_slices":        {"metrics": ["category_scores per tier/dynamic/multi_query/high_ratio/extreme_ratio/easy_static/rare"], "computed_by": "notebook"},
 "L19_baseline_deltas":   {"metrics": ["delta vs frozen ACT-XHIGH for engagement/final-program/train-exact/exact/budget-death/first-turn-length/tokens"], "computed_by": "notebook"},
 "L20_provenance":        {"metrics": ["arm","reasoning_effort","think_policy","seed","prompt_sha","template_sha","engine/model/runtime fingerprints","verifier_contract"], "computed_by": "notebook"},
 "L21_matched_arm_quality":{"metrics": ["per-task LOW vs BUDGET3072 executable/train-exact/test-exact/runtime comparison"], "computed_by": "notebook"},
 "L22_terminal_commit":{"metrics": ["finalizer_started","finalizer_trigger_reason","finalizer_prompt_tokens","finalizer_output_tokens","finalizer_finish_reason","terminal_parse_status","feedback_present_in_rendered_prompt","terminal_handoff_source_tokens","terminal_handoff_kept_tokens","terminal_handoff_truncated"], "computed_by": "notebook"},
 "verdict_tiers": {"activation_diagnostic": "tool_engagement_rate >= 0.50 (never a promotion prerequisite)", "productivity_diagnostic": "final_program_rate >= 0.25 and infrastructure_failure_rate == 0",
                   "quality": "all_train_exact_rate > 0", "usable_test": "at least one valid test prediction from a train-exact program",
                   "promotion": "quality and usable_test and compute_gate and no infrastructure failure",
                   "compute_gate": "budget_death_rate <= 0.50 and projected_20pct <= 10h and complete unique terminal coverage"},
}
PER_TURN_MAX_TOKENS = int(PROFILE["per_turn_max_tokens"])
TOTAL_GENERATED_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
if TOTAL_GENERATED_TOKEN_CAP != 12288:
    raise RuntimeError("v2.5 repair experiment requires the frozen 12,288-token total task budget.")
if EXPLORATION_TOKEN_CAP != TOTAL_GENERATED_TOKEN_CAP - FINAL_RESERVE_TOKENS:
    raise RuntimeError("terminal reserve arithmetic drift")
TOOL_TIMEOUT_SEC = float(PROFILE["tool_timeout_sec"])
VERIFIER_TIMEOUT_SEC = float(PROFILE["verifier_timeout_sec"])
VERIFIER_CONTRACT_VERSION = "fresh-train-plus-per-test-stage-v2.2"
VERIFIER_TIMEOUT_CONTRACT = {
    "version": VERIFIER_CONTRACT_VERSION,
    "train_stage_timeout_sec": VERIFIER_TIMEOUT_SEC,
    "per_test_stage_timeout_sec": VERIFIER_TIMEOUT_SEC,
    "aggregate_candidate_timeout_sec": None,
    "semantics": "fresh process for train, then one fresh process per test input; timeout applies independently to each stage",
    "comparison_rule": "Any matched C0/C1 comparison must use this identical verifier contract.",
}
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

MODEL_PATH = Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")
WHEELHOUSE = Path("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3")
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/piv_q27_c1_v25")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
# v2.4C: reset_outputs wipes WORK_ROOT on every Run All, so the checkpoint/resume machinery only
# works WITHIN one session (adaptive stage A -> stage B). Set RESUME_PREVIOUS_SESSION=True to keep a
# previous session's records. NOTE: wave metrics are filtered by RUN_ID, so a resumed run reports
# wall-clock only for the tasks it actually executed in the new session.
RESUME_PREVIOUS_SESSION = False
if PROFILE["reset_outputs"] and WORK_ROOT.exists() and not RESUME_PREVIOUS_SESSION:
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
# v1.6: one wave may run for the remaining P0 budget (up to 12,288 tokens ~= 33 min at the Q1-measured
# ~0.16 s/step). The reserve must exceed the longest possible wave plus bundle time, and each wave is
# additionally pre-checked against a projection built from measured wave timings.
SAFETY_RESERVE_SEC = 60 * 60
DEFAULT_SEC_PER_OUTPUT_TOKEN = 0.20   # projection prior before the first measured wave
WAVE_TIME_SAFETY_FACTOR = 1.25
WAVE_OVERHEAD_SEC = 90.0              # prefill / scheduling / tool + verifier allowance per wave

# Frozen from the successful Q1 / v1.9d lineage. C1 changes only enforce_eager in Cell 7.
MAX_MODEL_LEN = 29696
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90
# C1 terminal pilot keeps prefix caching explicitly OFF. Do not change it in this experiment.
ENABLE_PREFIX_CACHING = False
assert ENABLE_PREFIX_CACHING is False

# EXP-0 diagnostics preserved unchanged.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| Stateful Python REPL + Fresh Exact Train Verifier")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("PIV source commit:", PIV_SOURCE_COMMIT)
print("Agent budget: calls/turns/per-turn/total =", MAX_TOOL_CALLS, MAX_MODEL_TURNS, PER_TURN_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP)
print("ARMS:", ARMS, "| ACTION_MAX_TOKENS:", ACTION_MAX_TOKENS)
print("Terminal policy:", TERMINAL_POLICY_ENABLED, "| explore/final reserve:", EXPLORATION_TOKEN_CAP, FINAL_RESERVE_TOKENS)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT)

required = [MODEL_PATH, WHEELHOUSE, ARC_ROOT, TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
            EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH, TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_RUNTIME_PACKAGES = {"vllm":"0.19.0", "transformers":"4.57.6", "tokenizers":"0.22.2", "huggingface-hub":"0.36.2"}
_loaded_module_names = {"vllm":"vllm", "transformers":"transformers", "tokenizers":"tokenizers", "huggingface-hub":"huggingface_hub"}
before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart Kaggle session before Run All.")

def offline_pip_install(requirements):
    if not requirements: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install(["vllm==0.19.0"])
else:
    repair=[]
    for package in ["transformers","tokenizers","huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package=="huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(f"{package} already imported with wrong version; restart session.")
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)
after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
print("Runtime dependency parity PASS:", after_stack)


In [ ]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


In [ ]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


In [ ]:
# =====================================================================================
# CELL 4 — TOKENIZER + SOURCE-ALIGNED AGENTIC PROMPT + QWEN TOOL TEMPLATE + ACTIVE TASKS
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=False)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

# OpenAI-style schema; Qwen3.8's official chat template serializes it into its XML tool-call contract.
PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "python",
        "description": "Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMA_SHA256 = stable_json_sha256(PYTHON_TOOL)

SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer\u2013color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'

# LOCAL adaptation: source advertises a broad dependency list and 120s IPython timeout.
# The portability audit recommends a smaller P0 stack. Probe the actual current Python
# environment and advertise only approved libraries that really import.
import subprocess as _piv_subprocess
P0_LIBRARY_PROBES = [
    ("numpy","numpy"), ("scipy","scipy"), ("scikit-image","skimage"),
    ("networkx","networkx"), ("Pillow","PIL"),
]
AVAILABLE_P0_LIBRARIES=[]
P0_LIBRARY_PROBE_RESULTS={}
for _label,_module in P0_LIBRARY_PROBES:
    try:
        _cp=_piv_subprocess.run(
            [sys.executable,"-c",f"import {_module}"],
            text=True,capture_output=True,timeout=8,
        )
        _ok=(_cp.returncode==0)
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":_ok,"stderr":(_cp.stderr or "")[-500:]}
        if _ok: AVAILABLE_P0_LIBRARIES.append(_label)
    except Exception as _e:
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":False,"stderr":f"{type(_e).__name__}: {_e}"}

_source_lib_sentence=(
    "The environment supports common third party libraries like numpy, scipy, shapely, networkx, "
    "scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver."
)
_local_lib_sentence=(
    "For this LOCAL P0 port, approved third party libraries verified in the current environment are: "
    + (", ".join(AVAILABLE_P0_LIBRARIES) if AVAILABLE_P0_LIBRARIES else "none beyond the Python standard library")
    + ". Do not assume optional source libraries such as OR-Tools, z3-solver, python-constraint, or shapely are available."
)
SYSTEM_PROMPT = SOURCE_SYSTEM_PROMPT.replace(_source_lib_sentence,_local_lib_sentence).replace(
    "time out after 120.0 seconds",
    f"time out after {TOOL_TIMEOUT_SEC:.1f} seconds",
)
SYSTEM_PROMPT += (
    f"\n\n### Local P0 compute budget\n"
    f"You may execute Python at most {MAX_TOOL_CALLS} times in this task. "
    "Use the calls deliberately. When the final Python call has been used, do not request another tool call; "
    "return the self-contained final `solution(grid)` code block on the next assistant turn."
)

def grid_lines(grid):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)

def puzzle_to_source_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"]):
        parts += [f"### Training example {i}", "Input:", grid_lines(ex["input"]), "Output:", grid_lines(ex["output"])]
    parts.append("## Tests" if len(task["test"]) > 1 else "## Test")
    for i, ex in enumerate(task["test"]):
        if len(task["test"]) > 1:
            parts.append(f"### Test {i}")
        parts += ["Input:", grid_lines(ex["input"])]
    return "\n\n".join(parts)

def build_agent_user_prompt(task):
    # Keep the source prompt intact except for two necessary local substitutions:
    # 1) serialize the current puzzle; 2) clarify that test outputs are absent from the preloaded variable.
    prompt = SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}", puzzle_to_source_text(task))
    prompt = prompt.replace(
        'Each training/test example is a dict containing keys "input" and "output". '
        'The values are the grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
        'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". '
        'The values are grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
    )
    return prompt

SOURCE_AGENTIC_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_AGENTIC_PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
SOURCE_SYSTEM_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_SYSTEM_PROMPT.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"p0_library_probe.json","w") as _f:
    json.dump({"available":AVAILABLE_P0_LIBRARIES,"results":P0_LIBRARY_PROBE_RESULTS},_f,indent=2)
print("P0 approved libraries available:",AVAILABLE_P0_LIBRARIES)

CURRENT_ARM = ARMS[0]

def render_agent_chat(messages, reasoning_effort=None):
    # v1.8: the reasoning effort is the arm's variable; everything else in the render is frozen.
    effort = reasoning_effort or ARM_CONFIG[CURRENT_ARM]["reasoning_effort"]
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        tools=[PYTHON_TOOL],
        enable_thinking=True,
        reasoning_effort=effort,
        preserve_thinking=True,
    )
    try:
        return tokenizer.apply_chat_template(messages, **kwargs), f"qwen_tools+{effort}+preserve_thinking"
    except TypeError as exc:
        # Preserve the causal mechanism if this repacked tokenizer predates the explicit preserve kwarg.
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+{effort}(template-preserve-default); fallback={type(exc).__name__}"

def render_terminal_chat(messages):
    """v2.4 structural terminal renderer: no tools are advertised and thinking is hard-disabled."""
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
        preserve_thinking=True,
    )
    try:
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        method = "qwen_terminal_no_tools+nonthinking+preserve_thinking"
    except TypeError as exc:
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        method = f"qwen_terminal_no_tools+nonthinking(template-preserve-default); fallback={type(exc).__name__}"
    return rendered, method

_terminal_probe_messages=[
    {"role":"system","content":"terminal contract probe"},
    {"role":"user","content":"return final answer only"},
]
_terminal_probe_rendered,_terminal_probe_method=render_terminal_chat(_terminal_probe_messages)
if "<function=python>" in _terminal_probe_rendered or "<tools>" in _terminal_probe_rendered:
    raise RuntimeError("Terminal render unexpectedly advertises the Python tool.")
if _terminal_probe_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Terminal render unexpectedly reopens a thinking block.")
QWEN_TERMINAL_RENDER_SHA256=hashlib.sha256(_terminal_probe_rendered.encode("utf-8")).hexdigest()

# Frozen task selection: same Proxy-48; SMOKE is a deterministic subset of it.
def choose_smoke8_from_proxy():
    """v1.6: stratified to mirror Proxy-48 (tier1 32/48, tier2 12/48, tier3 4/48 -> 5/2/1), deterministic in
    Proxy-48 order. The previous first-match selector took all four tier-3 tasks and biased the feasibility gate."""
    f = proxy_source_feat.set_index("task_id").loc[PROXY48_TASK_IDS].copy()
    chosen=[]
    def add(x):
        if x is not None and x not in chosen and len(chosen)<8: chosen.append(x)
    def first(mask):
        for t in f.index[mask.to_numpy()]:
            if t not in chosen: return t
        return None
    t1=(f["proxy_tier"]==1); t2=(f["proxy_tier"]==2); t3=(f["proxy_tier"]==3)
    add(first(t3))
    add(first(t2)); add(first(t2))
    add(first(t1 & (f["proxy_easy_static"]==1)))
    add(first(t1 & (f["proxy_hard_tier1"]==1)))
    add(first(t1 & (f["proxy_n_queries"]>1)))
    add(first(t1 & (f["proxy_dynamic"]==1)))
    while len(chosen)<8 and first(t1) is not None: add(first(t1))
    for x in PROXY48_TASK_IDS:
        if len(chosen)>=8: break
        add(x)
    return chosen[:8]

if TASK_SET == "PROXY48": ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE8": ACTIVE_TASK_IDS = choose_smoke8_from_proxy()
elif TASK_SET == "PILOT2": ACTIVE_TASK_IDS = list(TERMINAL_PILOT_TASK_IDS)
else: raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")
ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
if len(ACTIVE_TASK_IDS)!=ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET)!=ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task count/uniqueness mismatch")
if not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Active set escaped frozen Proxy-48")
# v2.4B: fail BEFORE the 27B model is allocated if the adaptive pilot is not inside frozen SMOKE8.
# The original check lived in Cell 8, i.e. after model load, so a mismatch wasted a whole GPU session.
if RUN_PROFILE=="ADAPTIVE_SMOKE":
    _smoke8_preview=choose_smoke8_from_proxy()
    if not set(TERMINAL_PILOT_TASK_IDS).issubset(set(_smoke8_preview)):
        raise RuntimeError(f"Adaptive pilot {TERMINAL_PILOT_TASK_IDS} is not a subset of frozen SMOKE8 {_smoke8_preview}")
    print("Adaptive pilot-in-SMOKE8 precheck PASS:",TERMINAL_PILOT_TASK_IDS,"in",_smoke8_preview)

def build_prompt_records(arm):
    """Arm-specific initial prompts: identical user content; only the template-injected reasoning instruction differs."""
    recs=[]
    for task_id, task in eval_challenges.items():
        # v2.4B: only ACTIVE tasks are ever used. Rendering all 120 evaluation prompts on every
        # activate_arm() call (stage A, gate, stage B, score_arm => ~8x per run) was pure overhead.
        if task_id not in ACTIVE_TASK_SET:
            continue
        user_prompt=build_agent_user_prompt(task)
        messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
        rendered, method=render_agent_chat(messages, reasoning_effort=ARM_CONFIG[arm]["reasoning_effort"])
        recs.append({
            "arm":arm,"task_id":task_id, "user_prompt":user_prompt, "initial_messages":messages,
            "prompt":rendered, "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
            "prompt_tokens":len(tokenizer.encode(rendered, add_special_tokens=False)),
            "n_train":len(task["train"]), "n_test":len(task["test"]), "render_method":method,
        })
    df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"prompt","initial_messages","user_prompt"}} for r in recs])
    return recs, df
prompt_records, prompt_df = build_prompt_records(CURRENT_ARM)
prompt_df.to_csv(WORK_ROOT/"prompt_stats.csv", index=False)
PROMPT_RECORD_MAP={r["task_id"]:r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET}
# arm render sanity: every arm must render, and LOW/MEDIUM/XHIGH prompts must differ only in the injected instruction
_arm_probe={a:render_agent_chat([{"role":"system","content":"S"},{"role":"user","content":"U"}],reasoning_effort=ARM_CONFIG[a]["reasoning_effort"])[0] for a in ARM_CONFIG}
assert len({_arm_probe[a] for a in _arm_probe})==len(_arm_probe), "arm renders are not distinct"
print("Arm render probe OK:",{a:len(_arm_probe[a]) for a in _arm_probe})
# v1.9: template-level reasoning-effort contract: 'high' must be REJECTED by the mounted template.
def assert_qwen_reasoning_effort_contract():
    probe=[{"role":"system","content":"effort contract probe"},{"role":"user","content":"reply"}]
    r={e:render_agent_chat(probe,reasoning_effort=e)[0] for e in ("xhigh","medium","low")}
    assert "Reasoning effort is set to xhigh" in r["xhigh"] and "Reasoning effort is set to low" in r["low"]
    assert "Reasoning effort is set to" not in r["medium"]
    try:
        render_agent_chat(probe,reasoning_effort="high"); rejected=False
    except Exception: rejected=True
    if not rejected: raise RuntimeError("Mounted Qwen template unexpectedly accepts reasoning_effort='high'; template contract changed.")
    return {"supported":["xhigh","medium","low"],"high_rejected":True,"medium_injects_no_instruction":True}
QWEN_REASONING_EFFORT_CONTRACT=assert_qwen_reasoning_effort_contract()
print("Qwen reasoning-effort contract:",QWEN_REASONING_EFFORT_CONTRACT)

chat_template=getattr(tokenizer,"chat_template","") or ""
template_sha=hashlib.sha256(chat_template.encode()).hexdigest()
if "<tool_call>" not in chat_template or "tool_response" not in chat_template:
    raise RuntimeError("Mounted Qwen chat template does not expose the required tool-call/tool-response contract.")

# Validate the *actual mounted tokenizer* on the multi-turn structure PIV will use.
_transport_probe_messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"transport probe"},
    {"role":"assistant","content":"","reasoning_content":"PIV_REASONING_SENTINEL",
     "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"x=1\nx+1"}}}]},
    {"role":"tool","content":"2"},
]
_transport_rendered, _transport_method = render_agent_chat(_transport_probe_messages)
for _required in ("PIV_REASONING_SENTINEL", "<tool_call>", "<function=python>", "<parameter=code>", "<tool_response>", "2"):
    if _required not in _transport_rendered:
        raise RuntimeError(f"Qwen multi-turn tool-history rendering contract failed: missing {_required!r}")
if not _transport_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Qwen tool-history render did not reopen thinking for the next assistant turn.")
QWEN_TOOL_HISTORY_TRANSPORT_SHA256 = hashlib.sha256(_transport_rendered.encode("utf-8")).hexdigest()

# v2.4 terminal-history probe: historical tool calls/results must remain renderable even though no tool is advertised
# for the new terminal assistant turn.
_terminal_history_messages=copy.deepcopy(_transport_probe_messages)+[{"role":"user","content":TERMINAL_FINALIZER_INSTRUCTION}]
_terminal_history_rendered,_terminal_history_method=render_terminal_chat(_terminal_history_messages)
if "PIV_REASONING_SENTINEL" not in _terminal_history_rendered or "<tool_response>" not in _terminal_history_rendered or "2" not in _terminal_history_rendered:
    raise RuntimeError("Terminal renderer lost historical reasoning/tool feedback.")
if _terminal_history_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Terminal renderer reopened thinking after historical tool use.")
QWEN_TERMINAL_HISTORY_RENDER_SHA256=hashlib.sha256(_terminal_history_rendered.encode("utf-8")).hexdigest()
# v2.4B diagnostic: with no tools advertised, does the terminal render still carry the historical
# ASSISTANT tool call, or only the tool response? If only the response survives, the finalizer sees
# results without the code that produced them. Recorded, not asserted, so the template stays the authority.
QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS=bool("<function=python>" in _terminal_history_rendered)
if not QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS:
    print("WARNING: terminal render drops historical assistant tool calls; the finalizer sees tool OUTPUT only.")
with open(WORK_ROOT/"chat_template_fingerprint.txt","w") as f:
    f.write(f"sha256={template_sha}\ntool_schema_sha256={TOOL_SCHEMA_SHA256}\n")

profile_task_table=eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table=profile_task_table.merge(proxy_source_feat,on="task_id",how="left")
profile_task_table["prompt_tokens_piv_initial"]=profile_task_table["task_id"].map(prompt_df.set_index("task_id")["prompt_tokens"])
profile_task_table["proxy_order"]=profile_task_table["task_id"].map(proxy48_order)
profile_task_table=profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT/"active_task_selection.csv",index=False)

SOURCE_ALIGNMENT={
    "experiment":EXPERIMENT_ID,
    "piv_source_repo":PIV_SOURCE_REPO,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "piv_source_files":PIV_SOURCE_FILES,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "source_file_sha256":SOURCE_FILE_SHA256,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
    "chat_template_sha256":template_sha,
    "qwen_tool_history_transport_sha256":QWEN_TOOL_HISTORY_TRANSPORT_SHA256,
    "qwen_terminal_render_sha256":QWEN_TERMINAL_RENDER_SHA256,
    "qwen_terminal_history_render_sha256":QWEN_TERMINAL_HISTORY_RENDER_SHA256,
    "qwen_terminal_history_keeps_tool_calls":QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS,
    "source_agentic_prompt_local_sha256":SOURCE_AGENTIC_PROMPT_LOCAL_SHA256,
    "source_system_prompt_local_sha256":SOURCE_SYSTEM_PROMPT_LOCAL_SHA256,
    "source_invariants":[
        "one persistent Python REPL per lineage",
        "puzzle preloaded without test outputs",
        "tool errors returned to model",
        "self-contained solution(grid) final artifact",
        "fresh independent all-train exact verification",
        "test grids produced by verified program execution",
    ],
    "local_adaptations":[
        "direct vLLM + Qwen native tool chat template; no OpenAI Responses server",
        "lightweight subprocess worker instead of Docker/IPyBox",
        "stricter grid/domain/time/file-system safety",
        "bounded 6-execution P0 tool budget and context guard",
        "source-style max_retries=2 reliability retries share the same total token/tool/turn budgets",
        "per-turn ceiling = remaining total P0 budget (12288) bounded by context headroom; total generated-token cap 12288",
        "v2.4 terminal pilot protects 4096 tokens + one turn slot for a no-tools, non-thinking structured terminal commit",
    ],
}
with open(WORK_ROOT/"SOURCE_ALIGNMENT.json","w") as f: json.dump(SOURCE_ALIGNMENT,f,indent=2)
print("Chat template/tool contract PASS | SHA:",template_sha)
print("ACTIVE TASKS:",len(ACTIVE_TASK_IDS),TASK_SET)
display(prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].describe().to_frame("initial_prompt_tokens").round(1))


In [ ]:
# =====================================================================================
# CELL 5 — PIV CONTEXT / COMPUTE SAFETY (preserve history; do not add P3 compaction in P0)
# =====================================================================================
active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
max_initial_prompt=int(active_prompt_tokens.max())
CONTEXT_SAFETY_TOKENS=512
MAX_RENDERED_INPUT_TOKENS=MAX_MODEL_LEN-PER_TURN_MAX_TOKENS-CONTEXT_SAFETY_TOKENS
if max_initial_prompt > MAX_RENDERED_INPUT_TOKENS:
    raise RuntimeError(f"Initial PIV prompt exceeds context: {max_initial_prompt} > {MAX_RENDERED_INPUT_TOKENS}")
MAX_TOOL_OUTPUT_CHARS=4000
MAX_TOOL_RAW_ARTIFACT_CHARS=100000
MAX_CODE_CHARS=30000
print("PIV CONTEXT CONFIG")
print("  max_model_len           :",MAX_MODEL_LEN)
print("  max initial prompt      :",max_initial_prompt)
print("  per-turn max generation:",PER_TURN_MAX_TOKENS)
print("  total generated cap     :",TOTAL_GENERATED_TOKEN_CAP)
print("  max rendered input      :",MAX_RENDERED_INPUT_TOKENS)
print("  max tool output to model:",MAX_TOOL_OUTPUT_CHARS,"chars")
print("  history policy          : PRESERVE (P0); no P3 compact-state intervention")
print("  arms                    :",ARMS)
for _a in ARMS: print(f"    {_a}: {ARM_CONFIG[_a]}")
print("  action cap after forced close:",ACTION_MAX_TOKENS)


In [ ]:
# =====================================================================================
# CELL 6 — QWEN TOOL XML PARSER + PERSISTENT LOCAL REPL + FRESH EXACT VERIFIER + SELF-TESTS
# =====================================================================================
import ast as py_ast
import re, subprocess, selectors, tempfile, uuid, textwrap

DANGEROUS_IMPORT_ROOTS={"os","subprocess","pathlib","shutil","socket","requests","urllib","http","ftplib","paramiko","builtins","importlib","inspect"}
DANGEROUS_CALL_NAMES={"open","exec","eval","compile","__import__","input","breakpoint","globals","locals","vars","getattr","setattr","delattr","exit","quit"}
DANGEROUS_DUNDER_ATTRS={"__subclasses__","__globals__","__builtins__","__code__","__closure__","__getattribute__"}
ALLOWED_SYS_ATTRS={"setrecursionlimit","getrecursionlimit","maxsize"}

def canonical_grid(grid):
    if isinstance(grid,np.ndarray): grid=grid.tolist()
    if not is_rectangular_grid(grid): return None
    return tuple(tuple(int(x) for x in row) for row in grid)
def grids_equal(a,b):
    ca,cb=canonical_grid(a),canonical_grid(b)
    return ca is not None and cb is not None and ca==cb

def code_defines_solution(code):
    try: tree=py_ast.parse(code)
    except Exception: return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

def extract_last_solution_code(text):
    blocks=re.findall(r"```(?:python|py)?\s*\n?(.*?)```", text or "", flags=re.I|re.S)
    for code in reversed(blocks):
        code=code.strip()
        if code_defines_solution(code): return code
    return None

def split_qwen_reasoning(text, action_pos=None):
    text=text or ""
    prefix=text if action_pos is None else text[:action_pos]
    if "</think>" in prefix:
        a,b=prefix.rsplit("</think>",1)
        return a.replace("<think>","",1).strip(), b.strip(), "think_close"
    # Generation prompt already opens <think>; without a close, pre-action text is reasoning.
    return prefix.replace("<think>","",1).strip(), "", "implicit_open_think"

def _action_region(text):
    """Return only the committed action region after the final </think>.

    Qwen's generation prompt opens <think>. Without a close, the entire response remains scratch reasoning,
    even if it contains a fenced solution() draft or tool-like XML.
    """
    text=text or ""
    if "</think>" in text:
        pos=text.rfind("</think>")+len("</think>")
        return text[pos:],pos
    return "",len(text)

def parse_tool_calls(text):
    text=text or ""
    region,offset=_action_region(text)
    calls=[]
    pattern=r"<tool_call>\s*<function=([^>\s]+)>\s*(.*?)</function>\s*</tool_call>"
    for m in re.finditer(pattern,region,re.S):
        name=m.group(1).strip(); body=m.group(2); args={}
        for p in re.finditer(r"<parameter=([^>\s]+)>[ \t]*\r?\n?(.*?)[ \t]*\r?\n?[ \t]*</parameter>",body,re.S):
            val=p.group(2)
            # v1.6: unwrap a fenced ```python payload; the fence itself is not Python.
            _fm=re.fullmatch(r"```(?:python|py)?[ \t]*\n?(.*?)\n?```",val.strip(),flags=re.S|re.I)
            if _fm: val=_fm.group(1)
            # v2.4B: the old `\s*(.*?)\s*` stripped the FIRST line's indentation only, so a uniformly
            # indented XML payload became "x=1\n    print(x)" -> IndentationError inside the REPL.
            # Strip only the framing newline, then remove any uniform leading indent.
            val=textwrap.dedent(val)
            args[p.group(1).strip()]=val
        calls.append({"name":name,"arguments":args,"start":offset+m.start(),"end":offset+m.end(),"raw":m.group(0)})
    return calls

def parse_agent_turn(text):
    text=text or ""
    tcs=parse_tool_calls(text)
    if tcs:
        reasoning,content,rmode=split_qwen_reasoning(text,tcs[0]["start"])
        suffix=text[tcs[-1]["end"]:].strip()
        suffix=re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>)\s*$","",suffix).strip()
        return {"kind":"tool","tool_calls":tcs,"tool_name":tcs[0]["name"],"tool_args":tcs[0]["arguments"],
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":suffix,"final_code":None}
    action_text,_=_action_region(text)
    if "<tool_call>" in action_text:   # v1.6: prose merely mentioning <function= is not a malformed call
        reasoning,content,rmode=split_qwen_reasoning(text,None)
        return {"kind":"malformed_tool","tool_calls":[],"tool_name":None,"tool_args":{},
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":"","final_code":None}
    code=extract_last_solution_code(action_text)
    reasoning,content,rmode=split_qwen_reasoning(text,None)
    return {"kind":"final" if code else "text","tool_calls":[],"tool_name":None,"tool_args":{},
            "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
            "suffix_after_tool":"","final_code":code}

def generation_finish_reason_ok(finish_reason):
    # Mirrors the source client's rule that a truncated response is not a valid turn result.
    return str(finish_reason).lower() == "stop"

# v1.6: tool output is inserted verbatim into the next prompt. Neutralize chat-control strings the model's
# own code could print so they cannot open/close turns or tool blocks.
_CHAT_CONTROL_PATTERNS=[("<|","< |"),("<tool_response>","< tool_response>"),("</tool_response>","< /tool_response>"),
                        ("<tool_call>","< tool_call>"),("</tool_call>","< /tool_call>"),
                        ("<think>","< think>"),("</think>","< /think>")]
def sanitize_tool_output(text):
    text=str(text or "")
    for a,b in _CHAT_CONTROL_PATTERNS: text=text.replace(a,b)
    return text

def salvage_final_code_on_length(text, parsed):
    """v1.6 LOCAL: a length-capped final turn whose action region (after </think>) already contains a CLOSED,
    ast-parseable ```python block defining solution() is demonstrably complete; retrying it would only burn
    budget. Drafts inside an unclosed <think> are never salvaged."""
    if parsed.get("kind")!="final" or parsed.get("reasoning_mode")!="think_close": return None
    return parsed.get("final_code")

def guard_python_code(code, *, final=False):
    if not isinstance(code,str) or not code.strip(): return False,"empty_code"
    if len(code)>MAX_CODE_CHARS: return False,"code_too_long"
    try: tree=py_ast.parse(code)
    except SyntaxError as e: return False,f"syntax_error:{e.msg}"

    # `sys.setrecursionlimit(...)` is common in DFS/flood-fill ARC solutions.
    # Permit only a narrow sys surface; block module-table/process introspection.
    sys_aliases=set()
    for _n in py_ast.walk(tree):
        if isinstance(_n,py_ast.Import):
            for _a in _n.names:
                if _a.name=="sys": sys_aliases.add(_a.asname or "sys")
        elif isinstance(_n,py_ast.ImportFrom) and (_n.module or "")=="sys":
            imported={_a.name for _a in _n.names}
            if not imported.issubset(ALLOWED_SYS_ATTRS):
                return False,f"blocked_sys_importfrom:{sorted(imported)}"

    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name) and node.id=="__builtins__":
            return False,"blocked_builtins_escape"
        if isinstance(node,py_ast.Attribute) and node.attr in DANGEROUS_DUNDER_ATTRS:
            return False,f"blocked_dunder_attribute:{node.attr}"
        if isinstance(node,py_ast.Attribute) and isinstance(node.value,py_ast.Name) and node.value.id in sys_aliases:
            if node.attr not in ALLOWED_SYS_ATTRS:
                return False,f"blocked_sys_attribute:{node.attr}"
        if isinstance(node,py_ast.Constant) and isinstance(node.value,str) and "/kaggle" in node.value.lower():
            return False,"blocked_host_path_literal"
        if isinstance(node,(py_ast.Import,py_ast.ImportFrom)):
            names=[a.name.split('.')[0] for a in node.names] if isinstance(node,py_ast.Import) else [(node.module or '').split('.')[0]]
            if any(x in DANGEROUS_IMPORT_ROOTS for x in names): return False,f"blocked_import:{names}"
        if isinstance(node,py_ast.Call):
            if isinstance(node.func,py_ast.Name) and node.func.id in DANGEROUS_CALL_NAMES:
                return False,f"blocked_call:{node.func.id}"
            if isinstance(node.func,py_ast.Attribute) and node.func.attr in DANGEROUS_DUNDER_ATTRS:
                return False,f"blocked_dunder_call:{node.func.attr}"
        if isinstance(node,py_ast.Raise):
            exc=node.exc
            if isinstance(exc,py_ast.Name) and exc.id=="SystemExit":
                return False,"blocked_system_exit"
            if isinstance(exc,py_ast.Call) and isinstance(exc.func,py_ast.Name) and exc.func.id=="SystemExit":
                return False,"blocked_system_exit"
        if isinstance(node,(py_ast.Assign,py_ast.AnnAssign,py_ast.AugAssign,py_ast.NamedExpr)):
            targets=[]
            if isinstance(node,py_ast.Assign): targets=node.targets
            else: targets=[node.target]
            def rooted_at_puzzle(t):
                while isinstance(t,(py_ast.Subscript,py_ast.Attribute)):
                    t=t.value
                return isinstance(t,py_ast.Name) and t.id=="puzzle"
            for t in targets:
                if rooted_at_puzzle(t): return False,"protected_puzzle_assignment"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Attribute):
            base=node.func.value
            while isinstance(base,(py_ast.Subscript,py_ast.Attribute)): base=base.value
            if isinstance(base,py_ast.Name) and base.id=="puzzle" and node.func.attr in {"update","clear","pop","popitem","setdefault","append","extend","insert","remove"}:
                return False,"protected_puzzle_mutation"
        if isinstance(node,py_ast.Expr) and isinstance(node.value,py_ast.Name) and node.value.id=="puzzle":
            return False,"blocked_whole_puzzle_display"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Name) and node.func.id=="print":
            if any(isinstance(a,py_ast.Name) and a.id=="puzzle" for a in node.args):
                return False,"blocked_whole_puzzle_print"
    if final and not code_defines_solution(code): return False,"missing_solution_function"
    return True,"ok"

def compact_tool_output(text,limit=MAX_TOOL_OUTPUT_CHARS):
    text=str(text or "")
    if len(text)<=limit: return text
    keep=max(1,(limit-120)//2)
    return text[:keep]+f"\n...[tool output truncated; original chars={len(text)}]...\n"+text[-keep:]

WORKER_SCRIPT = r"""
import sys,json,traceback,io,contextlib,copy,ast,os
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV sandbox blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
original_puzzle=None
for line in sys.stdin:
    try:
        msg=json.loads(line); op=msg.get("op")
        if op=="init":
            original_puzzle=copy.deepcopy(msg["puzzle"])
            ns={"__name__":"__main__","puzzle":copy.deepcopy(original_puzzle)}
            out={"ok":True,"text":"REPL initialized."}
        elif op=="exec":
            # Restore the protected puzzle binding before every cell. Other REPL state persists.
            ns["puzzle"]=copy.deepcopy(original_puzzle)
            buf=io.StringIO()
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    # Jupyter-like semantics: execute statements, then display repr(last expression).
                    tree=ast.parse(msg["code"],mode="exec")
                    if tree.body and isinstance(tree.body[-1],ast.Expr):
                        prefix=ast.Module(body=tree.body[:-1],type_ignores=[])
                        if prefix.body: exec(compile(prefix,"<piv-repl>","exec"),ns,ns)
                        value=eval(compile(ast.Expression(tree.body[-1].value),"<piv-repl>","eval"),ns,ns)
                        if value is not None: print(repr(value))
                    else:
                        exec(compile(tree,"<piv-repl>","exec"),ns,ns)
                out={"ok":True,"text":buf.getvalue() or "Code executed successfully. No output was returned."}
            except BaseException:
                out={"ok":False,"text":buf.getvalue()+traceback.format_exc(limit=8)}
        elif op=="close":
            print(json.dumps({"ok":True,"text":"closing"}),flush=True); break
        else: out={"ok":False,"text":"Unknown worker op"}
    except BaseException:
        out={"ok":False,"text":traceback.format_exc(limit=8)}
    print(json.dumps(out,ensure_ascii=False),flush=True)
"""

class PersistentPythonWorker:
    def __init__(self,puzzle,timeout_sec=10.0):
        self.puzzle=puzzle; self.timeout_sec=float(timeout_sec); self.proc=None; self.successful_cells=[]; self.restart_count=0
        self.start()
    def start(self):
        self.close(silent=True)
        self.proc=subprocess.Popen([sys.executable,"-u","-c",WORKER_SCRIPT],stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=subprocess.DEVNULL,text=True,bufsize=1,cwd=tempfile.mkdtemp(prefix="piv_repl_"))
        r=self._request({"op":"init","puzzle":self.puzzle},timeout=max(2.0,self.timeout_sec))
        if not r.get("ok"): raise RuntimeError(f"REPL init failed: {r}")
        # Replay only previously successful bounded cells after a timeout/crash.
        if self.successful_cells:
            cells=list(self.successful_cells); self.successful_cells=[]
            for c in cells:
                rr=self.execute(c,record=True,replay=True)
                if not rr.get("ok"): break
    def _request(self,msg,timeout):
        if self.proc is None or self.proc.poll() is not None: raise RuntimeError("REPL process is not alive")
        self.proc.stdin.write(json.dumps(msg,ensure_ascii=False)+"\n"); self.proc.stdin.flush()
        sel=selectors.DefaultSelector(); sel.register(self.proc.stdout,selectors.EVENT_READ)
        events=sel.select(timeout); sel.close()
        if not events: raise TimeoutError(f"REPL tool exceeded {timeout}s")
        line=self.proc.stdout.readline()
        if not line: raise RuntimeError("REPL worker closed stdout")
        return json.loads(line)
    def execute(self,code,record=True,replay=False):
        ok,reason=guard_python_code(code,final=False)
        if not ok: return {"ok":False,"text":f"Tool code rejected by local safety guard: {reason}","guard_rejected":True,"restarted":False}
        try:
            r=self._request({"op":"exec","code":code},self.timeout_sec); r["restarted"]=False
            if r.get("ok") and record: self.successful_cells.append(code)
            return r
        except TimeoutError:
            self.restart_count+=1
            old=list(self.successful_cells)
            self.successful_cells=old
            try: self.start()
            except Exception: pass
            return {"ok":False,"text":f"Code execution terminated after {self.timeout_sec}s. Persistent worker was restarted; prior successful cells were replayed when possible.","timeout":True,"restarted":True}
        except Exception as e:
            self.restart_count+=1
            restarted=False
            try:
                self.start(); restarted=True
            except Exception:
                restarted=False
            return {"ok":False,"text":f"REPL infrastructure error: {type(e).__name__}: {e}. "
                                      f"Worker restart attempted; prior successful cells replayed when possible.",
                    "infrastructure_error":True,"restarted":restarted}
    def close(self,silent=False):
        p=getattr(self,"proc",None)
        if p is not None and p.poll() is None:
            try: p.terminate(); p.wait(timeout=1)
            except Exception:
                try: p.kill()
                except Exception: pass
        self.proc=None

VERIFIER_STAGE_SCRIPT = r"""
import sys,json,copy,traceback,numbers,os
payload=json.loads(sys.stdin.read()); code=payload["code"]; mode=payload["mode"]
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV verifier blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
def clean(g):
    try:
        import numpy as np
        if isinstance(g,np.ndarray): g=g.tolist()
    except Exception:
        pass
    try: rows=list(g)
    except Exception: raise ValueError("invalid_grid_rows")
    if not (1<=len(rows)<=30): raise ValueError("invalid_grid_rows")
    out=[]
    for r in rows:
        try:
            if hasattr(r,"tolist"): r=r.tolist()
            rr0=list(r)
        except Exception: raise ValueError("invalid_grid_rows")
        out.append(rr0)
    widths=[len(r) for r in out]
    if not widths or min(widths)<1 or max(widths)>30 or len(set(widths))!=1: raise ValueError("invalid_grid_shape")
    clean_out=[]
    for r in out:
        rr=[]
        for x in r:
            if isinstance(x,bool) or not isinstance(x,numbers.Integral): raise ValueError("non_integer_cell")
            x=int(x)
            if x<0 or x>9: raise ValueError("cell_out_of_domain")
            rr.append(x)
        clean_out.append(rr)
    return clean_out
RESULT_SENTINEL="PIV_RESULT:"
def emit(obj):
    sys.__stdout__.write("\n"+RESULT_SENTINEL+json.dumps(obj)+"\n"); sys.__stdout__.flush()
try:
    exec(code,ns,ns); fn=ns.get("solution")
    if not callable(fn): raise ValueError("solution_not_callable")
except BaseException:
    emit({"stage":mode,"ok":False,"code_error":traceback.format_exc(limit=10)}); raise SystemExit(0)
if mode=="train":
    task=payload["task"]
    train=[]
    for i,ex in enumerate(task["train"]):
        try:
            pred=clean(fn(copy.deepcopy(ex["input"])))
            exact=(pred==ex["output"])
            train.append({"index":i,"exact":exact,"pred":pred if not exact else None,"error":None})
        except BaseException:
            train.append({"index":i,"exact":False,"pred":None,"error":traceback.format_exc(limit=8)})
    emit({"stage":"train","ok":True,"train":train,"all_train_exact":bool(train) and all(x["exact"] for x in train)})
elif mode=="test":
    try:
        pred=clean(fn(copy.deepcopy(payload["test_input"])))
        emit({"stage":"test","ok":True,"pred":pred,"error":None})
    except BaseException:
        emit({"stage":"test","ok":False,"pred":None,"error":traceback.format_exc(limit=10)})
else:
    emit({"stage":mode,"ok":False,"code_error":"unknown_verifier_mode"})
"""

def task_without_test_solutions(task):
    return {"train":copy.deepcopy(task["train"]),"test":[{"input":copy.deepcopy(x["input"])} for x in task["test"]]}

def _run_verifier_stage(payload, timeout_sec):
    """Run one fresh verifier subprocess and return (started, result).

    `started` becomes True only after Popen succeeds. This distinguishes launch failures
    (no verifier process ever existed) from timeout/process/protocol failures after launch.
    """
    workdir=tempfile.mkdtemp(prefix="piv_verify_")
    proc=None; started=False
    stdout=""; stderr=""
    try:
        proc=subprocess.Popen([sys.executable,"-u","-c",VERIFIER_STAGE_SCRIPT],
                              stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                              text=True,cwd=workdir)
        started=True
        stdout,stderr=proc.communicate(input=json.dumps(payload),timeout=float(timeout_sec))
        base={"verifier_started":True,"returncode":proc.returncode}
        if proc.returncode!=0:
            return True,{**base,"ok":False,"error":(stderr or "")[-4000:] or f"verifier_returncode={proc.returncode}","stage_status":"process_error"}
        sentinel="PIV_RESULT:"
        payload_lines=[ln[len(sentinel):] for ln in (stdout or "").splitlines() if ln.startswith(sentinel)]
        if not payload_lines:
            return True,{**base,"ok":False,"error":"missing_verifier_result_sentinel","stage_status":"protocol_error","stdout_tail":(stdout or "")[-2000:]}
        r=json.loads(payload_lines[-1]); r.update(base); r["stage_status"]="ok"
        return True,r
    except subprocess.TimeoutExpired:
        if proc is not None:
            try: proc.kill()
            except Exception: pass
            try:
                _out,_err=proc.communicate(timeout=2)
                stdout=(stdout or "")+(_out or ""); stderr=(stderr or "")+(_err or "")
            except Exception: pass
        return bool(started),{"verifier_started":bool(started),"ok":False,"error":"verifier_timeout","stage_status":"timeout",
                              "stdout_tail":(stdout or "")[-2000:]}
    except Exception as e:
        if proc is not None:
            try: proc.kill()
            except Exception: pass
            try: proc.communicate(timeout=2)
            except Exception: pass
        _status="error" if started else "launch_error"
        return bool(started),{"verifier_started":bool(started),"ok":False,"error":f"{type(e).__name__}: {e}","stage_status":_status}
    finally:
        shutil.rmtree(workdir,ignore_errors=True)

VERIFIER_INFRA_STAGE_STATUSES={"launch_error","process_error","protocol_error","error"}

def _verifier_stage_is_infrastructure_failure(stage_status):
    return str(stage_status or "").strip().lower() in VERIFIER_INFRA_STAGE_STATUSES

def fresh_verify_solution(code,task,timeout_sec=VERIFIER_TIMEOUT_SEC):
    """Fresh two-stage verifier. Train evidence is finalized before any test subprocess starts.

    v2.4C keeps candidate-code failures distinct from verifier transport/process failures. A launch/process/
    protocol/host-side verifier failure invalidates infrastructure evidence; ordinary code rejection and timeouts
    remain candidate outcomes unless another infrastructure signal exists.
    """
    ok,reason=guard_python_code(code,final=True)
    if not ok:
        return {"ok":False,"verifier_started":False,"train_verifier_started":False,"test_verifier_started_count":0,
                "all_train_exact":False,"train":[],"test_outputs":[],"test_errors":[],"test_execution_ok":False,
                "any_test_output_ok":False,"error":reason,"verifier_status":"guard_rejected",
                "infrastructure_failure":False,"infrastructure_errors":[]}
    safe_task=task_without_test_solutions(task)
    train_started,tr=_run_verifier_stage({"mode":"train","code":code,"task":{"train":safe_task["train"]}}, timeout_sec)
    base={"ok":False,"verifier_started":bool(train_started),"train_verifier_started":bool(train_started),"test_verifier_started_count":0,
          "train":tr.get("train",[]),"all_train_exact":bool(tr.get("all_train_exact")),"test_outputs":[],"test_errors":[],
          "test_execution_ok":False,"any_test_output_ok":False,"train_error":None,"test_error":None,"error":None,
          "infrastructure_failure":False,"infrastructure_errors":[]}
    if tr.get("code_error"):
        base["train_error"]=tr.get("code_error"); base["error"]=tr.get("code_error"); base["verifier_status"]="train_code_error"; return base
    if tr.get("stage_status")!="ok":
        _st=str(tr.get("stage_status"))
        base["train_error"]=tr.get("error"); base["error"]=tr.get("error"); base["verifier_status"]="train_"+_st
        if _verifier_stage_is_infrastructure_failure(_st):
            base["infrastructure_failure"]=True
            base["infrastructure_errors"].append(f"train_verifier_{_st}: {tr.get('error')}")
        return base
    if not base["all_train_exact"]:
        base["verifier_status"]="train_rejected"; return base
    tests=[]; errors=[]; started_count=0
    for _test_idx,ex in enumerate(safe_task["test"]):
        started,te=_run_verifier_stage({"mode":"test","code":code,"test_input":ex["input"]}, timeout_sec)
        started_count += int(bool(started))
        _stage=str(te.get("stage_status") or "")
        if _stage=="ok" and te.get("ok") and te.get("pred") is not None:
            tests.append(te.get("pred")); errors.append(None)
        else:
            _err=te.get("error") or te.get("code_error") or _stage or "test_execution_failed"
            tests.append(None); errors.append(_err)
            if _verifier_stage_is_infrastructure_failure(_stage):
                base["infrastructure_failure"]=True
                base["infrastructure_errors"].append(f"test[{_test_idx}]_verifier_{_stage}: {_err}")
    base["test_verifier_started_count"]=started_count
    base["test_outputs"]=tests; base["test_errors"]=errors
    base["test_execution_ok"]=bool(tests) and all(x is not None for x in tests)
    base["any_test_output_ok"]=any(x is not None for x in tests)
    base["ok"]=base["any_test_output_ok"] and not base["infrastructure_failure"]
    if not base["test_execution_ok"]: base["test_error"]="\n".join(str(e) for e in errors if e)
    if base["infrastructure_failure"]:
        base["verifier_status"]="train_exact_test_infrastructure_failed"
    elif base["test_execution_ok"]: base["verifier_status"]="train_exact_test_ok"
    elif base["any_test_output_ok"]: base["verifier_status"]="train_exact_test_partial"
    else: base["verifier_status"]="train_exact_test_failed"
    return base

def verifier_candidate_status(verifier):
    """Single candidate-status authority; never relabel verifier infrastructure failure as model train rejection."""
    v=verifier or {}
    if v.get("infrastructure_failure"): return "verifier_infrastructure_failed"
    if v.get("all_train_exact") and v.get("test_execution_ok"): return "verified"
    if v.get("all_train_exact") and v.get("any_test_output_ok"): return "verified_partial_test"
    if v.get("all_train_exact"): return "test_execution_failed"
    return "train_rejected"

# ---------------------------- v2.4B TERMINAL ARTIFACT PARSER ----------------------------
# Moved here from Cell 8: Cell 7's GPU positive control must use the identical parser, and Cell 7
# runs before Cell 8. Keeping one implementation means the control really tests the production path.
_MODEL_CONTROL_TOKEN_RE=re.compile(r"<\|[^|>]*\|>")
def strip_model_control_tokens(text):
    """vLLM is called with skip_special_tokens=False (needed for </think> accounting), so the decoded
    text keeps chat-control tokens such as <|im_end|>. They are never part of the structured artifact."""
    return _MODEL_CONTROL_TOKEN_RE.sub("",str(text or ""))

def parse_terminal_json_output(text):
    """Parse terminal JSON without mutating its payload.

    Control tokens may surround the JSON with skip_special_tokens=False. Find the first JSON object and
    raw-decode it; do not regex-strip the whole string because solution_code may contain a literal like
    <|im_end|> that must survive byte-for-byte.
    """
    raw=str(text or "").strip()
    out={"ok":False,"status":"empty_output","solution_code":None,"json_object":None}
    if not raw or not strip_model_control_tokens(raw).strip():
        return out
    start=raw.find("{")
    if start<0:
        out["status"]="json_object_not_found"
        return out
    try:
        # raw_decode tolerates trailing whitespace/junk after a complete JSON object.
        obj,_end=json.JSONDecoder().raw_decode(raw[start:])
    except Exception as exc:
        out["status"]=f"json_parse_error:{type(exc).__name__}"
        return out
    out["json_object"]=obj
    if not isinstance(obj,dict) or set(obj.keys())!={"solution_code"}:
        out["status"]="json_schema_shape_mismatch"
        return out
    code=obj.get("solution_code")
    if not isinstance(code,str) or not code.strip():
        out["status"]="solution_code_missing_or_empty"
        return out
    try:
        tree=py_ast.parse(code)
    except Exception as exc:
        out["status"]=f"python_syntax_error:{type(exc).__name__}"
        return out
    if not any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body):
        out["status"]="missing_top_level_solution_function"
        return out
    safe,why=guard_python_code(code,final=True)
    if not safe:
        out["status"]=f"final_guard_rejected:{why}"
        return out
    out.update({"ok":True,"status":"ok","solution_code":code})
    return out

# ---------------------------- PRELOAD SELF-TESTS ----------------------------
self_tests={}
t=parse_agent_turn("reasoning\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>")
assert t["kind"]=="tool" and len(t["tool_calls"])==1 and t["tool_name"]=="python" and "puzzle.keys" in t["tool_args"]["code"]
t_multi=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nx=1\n</parameter>\n</function>\n</tool_call>\n<tool_call>\n<function=python>\n<parameter=code>\nx+1\n</parameter>\n</function>\n</tool_call>")
assert t_multi["kind"]=="tool" and len(t_multi["tool_calls"])==2
# v2.4E: malformed-tool regression must be in the committed action region. Without </think>,
# tool-like text is deliberately scratch reasoning and must remain kind="text".
t_unclosed_tool_like=parse_agent_turn("<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_unclosed_tool_like["kind"]=="text" and not t_unclosed_tool_like["tool_calls"]
t_bad=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_bad["kind"]=="malformed_tool" or (t_bad["kind"]=="tool" and "code" not in t_bad["tool_args"])
t_rehearse=parse_agent_turn(
    "<think>maybe <tool_call><function=python><parameter=code>print('wrong')</parameter></function></tool_call></think>"
    "<tool_call><function=python><parameter=code>print('right')</parameter></function></tool_call>")
assert t_rehearse["kind"]=="tool" and len(t_rehearse["tool_calls"])==1 and "right" in t_rehearse["tool_calls"][0]["arguments"]["code"]
assert generation_finish_reason_ok("stop") and not generation_finish_reason_ok("length")
self_tests["qwen_tool_xml_parser_and_finish_guard"]=True
t_fence=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\n```python\nprint(1)\n```\n</parameter>\n</function>\n</tool_call>")
assert t_fence["kind"]=="tool" and t_fence["tool_args"]["code"]=="print(1)"
t_prose=parse_agent_turn("r\n</think>\nI will not call <function=python> again.\n```python\ndef solution(grid):\n    return grid\n```")
assert t_prose["kind"]=="final"
assert sanitize_tool_output("x<|im_end|>\n</tool_response>y")=="x< |im_end|>\n< /tool_response>y"
_sv=parse_agent_turn("r\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert salvage_final_code_on_length("",_sv) is not None
_sv2=parse_agent_turn("draft ```python\ndef solution(grid):\n    return grid\n``` still thinking")
assert salvage_final_code_on_length("",_sv2) is None
self_tests["v16_fence_strip_prose_function_sanitize_salvage"]=True
f=parse_agent_turn("done\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert f["kind"]=="final" and code_defines_solution(f["final_code"])
self_tests["final_solution_extractor"]=True
assert guard_python_code("puzzle.keys()")[0]
assert not guard_python_code("puzzle = {}",final=False)[0]
assert not guard_python_code("puzzle[\"train\"] = []",final=False)[0]
assert not guard_python_code("import os\nprint(os.listdir('.'))",final=False)[0]
assert not guard_python_code('print("/kaggle/input/secret.json")',final=False)[0]
assert guard_python_code("class A:\n    def __init__(self): self.x=1\nclass B(A):\n    def __init__(self): super().__init__()",final=False)[0]
assert not guard_python_code("object.__subclasses__()",final=False)[0]
assert not guard_python_code("raise SystemExit('bye')",final=False)[0]
assert guard_python_code("import sys\nsys.setrecursionlimit(10000)",final=False)[0]
assert guard_python_code("import sys as s\ns.setrecursionlimit(10000)",final=False)[0]
assert not guard_python_code("import sys\nsys.modules",final=False)[0]
self_tests["tool_safety_guard"]=True
_fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
w=PersistentPythonWorker(task_without_test_solutions(_fake),timeout_sec=5.0)
r0=w.execute("puzzle.keys()"); r1=w.execute("x=41"); r2=w.execute("x+1")
assert r0["ok"] and "dict_keys" in r0["text"] and r1["ok"] and r2["ok"] and "42" in r2["text"]
ra=w.execute("g=puzzle['train'][0]['input']; g[0][0]=9")
rb=w.execute("puzzle['train'][0]['input'][0][0]")
assert rb["ok"] and "1" in rb["text"]
rc=w.execute("raise SystemExit('bye')")
rd=w.execute("1+1")
assert (not rc["ok"]) and rd["ok"] and "2" in rd["text"]
w.close()
self_tests["persistent_repl_state_and_expression_display"]=True
v=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)
assert v["all_train_exact"] and v["test_outputs"]==[[[2]]]
v_noise=fresh_verify_solution(
    "print('noise', end='')\ndef solution(grid):\n    return [row[:] for row in grid]",
    _fake,timeout_sec=5)
assert v_noise["all_train_exact"] and v_noise["test_outputs"]==[[[2]]]
self_tests["verifier_result_sentinel"]=True
self_tests["fresh_exact_verifier"]=True
v2=fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5)
assert not v2["all_train_exact"] and v2["test_outputs"]==[]
self_tests["train_fail_blocks_test_execution"]=True
v3=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('test-only failure')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake,timeout_sec=5)
assert v3["all_train_exact"] and not v3["test_execution_ok"] and v3["test_outputs"]==[None]
self_tests["train_exact_survives_test_execution_failure"]=True
_fake_multi={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]},{"input":[[3]]}]}
v4=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('one test fails')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake_multi,timeout_sec=5)
assert v4["all_train_exact"] and v4["test_outputs"][0] is None and v4["test_outputs"][1]==[[3]]
self_tests["partial_multiquery_test_outputs_preserved"]=True
# v2.1 regression: a test-only timeout must preserve already established train-exact evidence.
v_timeout=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]:\n        while True: pass\n    return [row[:] for row in grid]",
    _fake,timeout_sec=2.0)
assert v_timeout["verifier_started"] and v_timeout["all_train_exact"] and not v_timeout["test_execution_ok"] and v_timeout["test_outputs"]==[None]
self_tests["train_exact_survives_test_timeout"]=True
# v2.1 regression: guard rejection must not claim a verifier subprocess actually started.
v_guard=fresh_verify_solution("import os\ndef solution(grid): return grid",_fake,timeout_sec=1)
assert v_guard["verifier_status"]=="guard_rejected" and not v_guard["verifier_started"]
self_tests["guard_reject_does_not_overstate_verifier_execution"]=True
# v2.2 regression: a process-launch failure must NOT claim verifier_started=True.
_orig_popen=subprocess.Popen
def _launch_boom(*a,**k):
    raise FileNotFoundError("synthetic verifier launch failure")
try:
    subprocess.Popen=_launch_boom
    _started,_launch_res=_run_verifier_stage({"mode":"train","code":"def solution(grid): return grid","task":{"train":[]}},1.0)
finally:
    subprocess.Popen=_orig_popen
assert (not _started) and (not _launch_res.get("verifier_started")) and _launch_res.get("stage_status")=="launch_error", _launch_res
self_tests["verifier_launch_failure_not_counted_as_started"]=True
# v2.4C regression: launch failure must propagate through fresh_verify_solution as infrastructure-invalid evidence.
_orig_popen_v24c=subprocess.Popen
def _launch_boom_v24c(*a,**k):
    raise FileNotFoundError("synthetic fresh-verifier launch failure")
try:
    subprocess.Popen=_launch_boom_v24c
    _v24c_vi=fresh_verify_solution("def solution(grid): return [row[:] for row in grid]",_fake,timeout_sec=1)
    assert _v24c_vi["verifier_status"]=="train_launch_error"
    assert _v24c_vi["infrastructure_failure"] and _v24c_vi["infrastructure_errors"]
    assert verifier_candidate_status(_v24c_vi)=="verifier_infrastructure_failed"
finally:
    subprocess.Popen=_orig_popen_v24c
self_tests["v24c_verifier_infrastructure_propagation"]=True

# ---- v2.4B regressions -------------------------------------------------------------------
# 1) Uniformly indented XML payloads must stay valid Python.
_t_indent=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>\n    x=1\n    print(x)\n</parameter></function></tool_call>")
assert _t_indent["kind"]=="tool" and _t_indent["tool_args"]["code"]=="x=1\nprint(x)", _t_indent["tool_args"]
_t_inline=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>")
assert _t_inline["tool_args"]["code"]=="puzzle.keys()", _t_inline["tool_args"]
# genuinely indented bodies (a function defined inside the payload) must NOT be flattened
_t_body=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>\ndef f():\n    return 1\nprint(f())\n</parameter></function></tool_call>")
assert _t_body["tool_args"]["code"]=="def f():\n    return 1\nprint(f())", _t_body["tool_args"]
py_ast.parse(_t_indent["tool_args"]["code"]); py_ast.parse(_t_body["tool_args"]["code"])
self_tests["v24b_parameter_dedent"]=True
# 2) The terminal parser must survive skip_special_tokens=False decoding.
_good_json=json.dumps({"solution_code":"def solution(grid):\n    return [row[:] for row in grid]"})
assert parse_terminal_json_output(_good_json)["ok"]
assert parse_terminal_json_output(_good_json+"<|im_end|>")["ok"], "control tokens must not break terminal parsing"
assert parse_terminal_json_output(_good_json+"\n<|endoftext|>")["ok"]
assert parse_terminal_json_output("<|im_start|>assistant\n"+_good_json+"<|im_end|>")["ok"]
assert parse_terminal_json_output('{"solution_code":"def solution(grid):')["status"].startswith("json_parse_error")
assert parse_terminal_json_output("")["status"]=="empty_output"
assert parse_terminal_json_output("<|im_end|>")["status"]=="empty_output"
assert parse_terminal_json_output(json.dumps({"solution_code":"def other(g): return g"}))["status"]=="missing_top_level_solution_function"
assert parse_terminal_json_output(json.dumps({"solution_code":"import os\ndef solution(g): return g"}))["status"].startswith("final_guard_rejected")
_literal_code='def solution(grid):\n    marker="<|im_end|>"\n    return [row[:] for row in grid]'
_literal_parsed=parse_terminal_json_output(json.dumps({"solution_code":_literal_code})+"<|im_end|>")
assert _literal_parsed["ok"] and _literal_parsed["solution_code"]==_literal_code
_unclosed_draft="<think>still reasoning\n```python\ndef solution(grid):\n    return grid\n```"
_unclosed_parsed=parse_agent_turn(_unclosed_draft)
assert _unclosed_parsed["kind"]=="text" and _unclosed_parsed["final_code"] is None and _unclosed_parsed["reasoning_mode"]=="implicit_open_think"
self_tests["v24d_terminal_literal_preservation"]=True
self_tests["v24d_unclosed_reasoning_not_final"]=True
self_tests["v24b_terminal_parser_control_tokens"]=True
# 3) Every verifier return path exposes "ok".
for _vr in (fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5),
            fresh_verify_solution("import os\ndef solution(grid): return grid",_fake,timeout_sec=5),
            fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)):
    assert "ok" in _vr, _vr
self_tests["v24b_verifier_ok_key_always_present"]=True

with open(WORK_ROOT/"preload_self_tests.json","w") as f: json.dump(self_tests,f,indent=2)
print("PIV preload self-tests PASS:",self_tests)


# =====================================================================================
# v2.5 — CANDIDATE-FIRST REPAIR HELPERS (defined before model allocation)
# =====================================================================================

def _v25_exception_last_line(err, max_chars=220):
    lines=[x.strip() for x in str(err or "").splitlines() if x.strip()]
    out=(lines[-1] if lines else "execution_error")
    return out[:int(max_chars)]

def _v25_grid_dims(g):
    try:
        if not isinstance(g,list) or not g: return None
        if not all(isinstance(r,list) for r in g): return None
        widths=[len(r) for r in g]
        if not widths or min(widths)<1 or len(set(widths))!=1: return None
        return (len(g),widths[0])
    except Exception:
        return None

def _v25_pair_diag(entry, ex, mismatch_limit=4):
    idx=int(entry.get("index",0))
    if entry.get("exact"):
        return {"pair":idx,"kind":"exact"}
    err=entry.get("error")
    if err:
        return {"pair":idx,"kind":"exception","error":_v25_exception_last_line(err)}
    pred=entry.get("pred")
    expected=ex.get("output")
    identity=bool(pred==ex.get("input")) if pred is not None else False
    got_dims=_v25_grid_dims(pred); exp_dims=_v25_grid_dims(expected)
    if got_dims is None:
        return {"pair":idx,"kind":"invalid_output","got_dims":None,"expected_dims":exp_dims,"identity":identity}
    if got_dims!=exp_dims:
        return {"pair":idx,"kind":"dims_mismatch","got_dims":got_dims,"expected_dims":exp_dims,"identity":identity}
    H,W=got_dims
    mism=[]; wrong=0
    for r in range(H):
        for c in range(W):
            if pred[r][c]!=expected[r][c]:
                wrong+=1
                if len(mism)<int(mismatch_limit):
                    mism.append((r,c,expected[r][c],pred[r][c]))
    return {"pair":idx,"kind":"cells_wrong","got_dims":got_dims,"expected_dims":exp_dims,
            "wrong_cells":int(wrong),"total_cells":int(H*W),"mismatches":mism,"identity":identity}

def _v25_feedback_bounded(txt, max_feedback_tokens):
    ids=list(tokenizer.encode(txt,add_special_tokens=False))
    truncated=False
    if len(ids)>int(max_feedback_tokens):
        txt=tokenizer.decode(ids[:int(max_feedback_tokens)],skip_special_tokens=False).strip()
        truncated=True
    return txt,len(tokenizer.encode(txt,add_special_tokens=False)),truncated

def _v25_feedback_text(verifier, task, max_feedback_tokens=REPAIR_FEEDBACK_MAX_TOKENS):
    train=list((verifier or {}).get("train") or [])
    exact=sum(1 for x in train if x.get("exact"))
    n=len(task.get("train",[]))
    if not train:
        # v2.5A: train_timeout / train_code_error return no per-pair results. Report the verifier's own error
        # instead of claiming an exception on every pair.
        _vs=str((verifier or {}).get("verifier_status") or "unknown")
        _ve=_v25_exception_last_line((verifier or {}).get("train_error") or (verifier or {}).get("error") or "no verifier error text")
        _txt="\n".join([f"candidate #1 fresh-train result: 0/{n} exact.",
                        f"VERIFIER: {_vs}; {_ve}",
                        "The verifier returned no per-pair result, so the whole program failed before any pair was scored.",
                        "Return a self-contained program that terminates quickly and avoids the reported failure.",
                        "Repair the standalone solution(grid) program. Do not rely on REPL variables or helper state from exploration."]).strip()
        _txt,_tok,_tr=_v25_feedback_bounded(_txt,max_feedback_tokens)
        return {"text":_txt,"tokens":_tok,"truncated":_tr,"train_exact_count":0,"n_train":int(n),
                "identity_pairs":[],"pair_diags":[],"verifier_level_failure":True}
    diags=[]
    by_index={int(x.get("index",i)):x for i,x in enumerate(train)}
    for i,ex in enumerate(task.get("train",[])):
        ent=by_index.get(i,{"index":i,"exact":False,"error":"missing verifier pair result"})
        diags.append(_v25_pair_diag(ent,ex))
    identity_pairs=[d["pair"] for d in diags if d.get("identity")]
    lines=[f"candidate #1 fresh-train result: {exact}/{n} exact."]
    if identity_pairs:
        joined=",".join(map(str,identity_pairs))
        lines.append(f"IDENTITY FAILURE: your program returned the input unchanged on train pair(s) {joined}; no transformation was applied there.")
    for d in diags:
        if d["kind"]=="exact":
            continue
        i=d["pair"]
        if d["kind"]=="exception":
            lines.append(f"pair {i}: EXCEPTION {d['error']}")
        elif d["kind"]=="dims_mismatch":
            gd=d.get("got_dims"); ed=d.get("expected_dims"); ident="; output==input" if d.get("identity") else ""
            lines.append(f"pair {i}: DIMS got {gd[0]}x{gd[1]} / expected {ed[0]}x{ed[1]}{ident}")
        elif d["kind"]=="invalid_output":
            ed=d.get("expected_dims"); es=(f"{ed[0]}x{ed[1]}" if ed else "unknown")
            lines.append(f"pair {i}: INVALID OUTPUT / expected dims {es}")
        else:
            gd=d["got_dims"]
            mm="; ".join(f"({r},{c}): expected {e}, got {g}" for r,c,e,g in d.get("mismatches",[]))
            ident="; output==input" if d.get("identity") else ""
            lines.append(f"pair {i}: dims got {gd[0]}x{gd[1]} / expected {gd[0]}x{gd[1]}; wrong cells {d['wrong_cells']}/{d['total_cells']}{ident}; first mismatches: {mm}")
    lines.append("Repair the standalone solution(grid) program. Do not rely on REPL variables or helper state from exploration.")
    txt="\n".join(lines).strip()
    ids=list(tokenizer.encode(txt,add_special_tokens=False))
    truncated=False
    if len(ids)>int(max_feedback_tokens):
        txt=tokenizer.decode(ids[:int(max_feedback_tokens)],skip_special_tokens=False).strip()
        truncated=True
    return {"text":txt,"tokens":len(tokenizer.encode(txt,add_special_tokens=False)),"truncated":truncated,
            "train_exact_count":int(exact),"n_train":int(n),"identity_pairs":identity_pairs,"pair_diags":diags}

def _v25_repair_feedback_marker(feedback_text):
    canonical=str(feedback_text or "").strip()
    sha=hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return canonical,sha,f"[V25_REPAIR_FEEDBACK_SHA256={sha}]"

def _v25_build_repair_messages(task, candidate_code, feedback_text):
    canonical,sha,marker=_v25_repair_feedback_marker(feedback_text)
    safe_task=task_without_test_solutions(task)
    puzzle_text=puzzle_to_source_text(safe_task)
    system=("You repair one ARC Python solution after fresh training verification. "
            "Do not use tools. Do not expose chain-of-thought. Return only the required JSON object with key solution_code.")
    user=("Repair candidate #1 into one complete standalone def solution(grid) program. Test solutions are unavailable. "
          "The puzzle below contains training input/output pairs and test inputs only.\n\n"
          f"PUZZLE:\n{puzzle_text}\n\nCANDIDATE #1:\n```python\n{candidate_code}\n```\n\n"
          f"{marker}\n{canonical}\n[/V25_REPAIR_FEEDBACK]\n\n"
          "Use the verifier evidence to correct the transformation. Return the full corrected program, not a patch or explanation.")
    msgs=[{"role":"system","content":system},{"role":"user","content":user}]
    rendered,method=render_terminal_chat(msgs)
    # Qwen's template may trim message-edge whitespace. Canonical stripped feedback is the transport authority.
    stripped_present=bool(canonical and canonical in rendered)
    marker_present=bool(marker in rendered)
    raw_present=bool(str(feedback_text or "") in rendered)
    return {"messages":msgs,"rendered":rendered,"render_method":method,"feedback_sha256":sha,
            "feedback_raw_sha256":hashlib.sha256(str(feedback_text or "").encode("utf-8")).hexdigest(),
            "feedback_present_raw":raw_present,"feedback_present_stripped":stripped_present,
            "feedback_marker_present":marker_present}

def _v25_repair_admission(prompt_tokens, generated_tokens):
    remaining=max(0,int(TOTAL_GENERATED_TOKEN_CAP)-int(generated_tokens))
    safe=max(0,int(MAX_MODEL_LEN)-int(prompt_tokens)-int(CONTEXT_SAFETY_TOKENS))
    cap=min(int(REPAIR_MAX_TOKENS),remaining,safe)
    return {"cap":max(0,int(cap)),"remaining_total_tokens":remaining,"context_headroom":safe}

def _v25_train_exact_count(verifier):
    return int(sum(1 for x in list((verifier or {}).get("train") or []) if x.get("exact")))

def _v25_choose_candidate(code1,v1,code2,v2):
    """Quality-preserving selection. Candidate #2 is selected only if it strictly improves train exact count or reaches all-train-exact."""
    c1=_v25_train_exact_count(v1); c2=_v25_train_exact_count(v2)
    if bool((v2 or {}).get("all_train_exact")):
        return 2,code2,v2
    if c2>c1:
        return 2,code2,v2
    return 1,code1,v1

def _v25_repair_output_cap_violation(observed_tokens,requested_cap):
    return int(observed_tokens)>int(requested_cap)

def run_v25_repair_helper_selftests():
    fake_task={"train":[
        {"input":[[1,0],[0,1]],"output":[[0,1],[1,0]]},
        {"input":[[1,2,3]],"output":[[3,2,1]]},
        {"input":[[9]],"output":[[8]]},
        {"input":[[4]],"output":[[5]]},
    ],"test":[{"input":[[2,2]]}]}
    fake_v={"train":[
        {"index":0,"exact":False,"pred":[[1,0],[0,1]],"error":None},
        {"index":1,"exact":False,"pred":[[3],[2],[1]],"error":None},
        {"index":2,"exact":False,"pred":None,"error":"Traceback...\nTypeError: NoneType not subscriptable\n"},
        {"index":3,"exact":True,"pred":None,"error":None},
    ],"all_train_exact":False,"verifier_status":"train_rejected"}
    fb=_v25_feedback_text(fake_v,fake_task)
    assert fb["tokens"]<=REPAIR_FEEDBACK_MAX_TOKENS,fb
    assert 0 in fb["identity_pairs"],fb
    assert "IDENTITY FAILURE" in fb["text"] and "DIMS got 3x1 / expected 1x3" in fb["text"] and "TypeError:" in fb["text"],fb
    # v2.5A: identity with CHANGED dims is dims_mismatch; identity with SAME dims is cells_wrong. Both must set identity.
    _dim_identity=_v25_pair_diag({"index":0,"exact":False,"pred":[[1,2,3]],"error":None},{"input":[[1,2,3]],"output":[[1],[2],[3]]})
    assert _dim_identity["kind"]=="dims_mismatch" and _dim_identity["identity"],_dim_identity
    _cells_identity=_v25_pair_diag({"index":1,"exact":False,"pred":fake_task["train"][1]["input"],"error":None},fake_task["train"][1])
    assert _cells_identity["kind"]=="cells_wrong" and _cells_identity["identity"],_cells_identity
    # Trailing whitespace must not create a false negative in the transport assertion.
    tr=_v25_build_repair_messages(fake_task,"def solution(grid):\n    return grid",fb["text"]+"\n\n")
    assert tr["feedback_present_stripped"] and tr["feedback_marker_present"],tr
    # v2.5A: verifier-level failure (no per-pair results) must surface the verifier error, not fake exceptions.
    _vl=_v25_feedback_text({"train":[],"all_train_exact":False,"verifier_status":"train_timeout",
                            "train_error":"verifier stage timeout after 20s"},fake_task)
    assert _vl["verifier_level_failure"] and "VERIFIER: train_timeout" in _vl["text"] and "timeout after 20s" in _vl["text"],_vl
    assert "missing verifier pair result" not in _vl["text"],_vl
    # Failure injection: repair prompt overflow -> no admission.
    adm=_v25_repair_admission(MAX_MODEL_LEN+100,0)
    assert adm["cap"]==0,adm
    # Failure injection: verifier timeout in repair must not erase a stronger candidate #1.
    v1={"train":[{"exact":True},{"exact":False}],"all_train_exact":False,"verifier_status":"train_rejected"}
    v2={"train":[],"all_train_exact":False,"verifier_status":"train_timeout"}
    sel=_v25_choose_candidate("c1",v1,"c2",v2)
    assert sel[0]==1,sel
    # Failure injection: over-cap detection.
    assert _v25_repair_output_cap_violation(101,100) and not _v25_repair_output_cap_violation(100,100)
    payload={"ok":True,"feedback_tokens":fb["tokens"],"identity_flag":True,"dims_diff":True,
             "exception_last_line":True,"transport_stripped_presence":True,"prompt_overflow_injection":True,
             "repair_verifier_timeout_preserves_candidate1":True,"overcap_injection":True,
             "verifier_level_failure_feedback":True}
    with open(WORK_ROOT/"v25_repair_helper_selftest.json","w") as f: json.dump(payload,f,indent=2)
    print("v2.5 repair helper self-tests PASS:",payload)
    return payload

V25_REPAIR_HELPER_SELFTEST=run_v25_repair_helper_selftests()



In [ ]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD (C1: CUDA graph / no custom AR)
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None
# v2.4B: defined unconditionally so downstream cells never depend on how far bootstrap got.
TERMINAL_POSITIVE_CONTROL = {"ran": False, "ok": None}
ENGINE_BOOTSTRAP_OK = (not USE_MODEL)
ENGINE_BOOTSTRAP_STAGE = "not_requested" if not USE_MODEL else "not_started"
ENGINE_BOOTSTRAP_ERROR = None
ENGINE_BOOTSTRAP_TRACEBACK = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": False,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": bool(ENABLE_PREFIX_CACHING),
    "seed": SEED,
}
# C1 causal-regression firewall: compare against the frozen donor engine.
DONOR_ENGINE_CONFIG = dict(ENGINE_CONFIG)
DONOR_ENGINE_CONFIG["enforce_eager"] = True
_engine_diffs = {
    k: (DONOR_ENGINE_CONFIG.get(k), ENGINE_CONFIG.get(k))
    for k in sorted(set(DONOR_ENGINE_CONFIG) | set(ENGINE_CONFIG))
    if DONOR_ENGINE_CONFIG.get(k) != ENGINE_CONFIG.get(k)
}
assert _engine_diffs == {"enforce_eager": (True, False)}, _engine_diffs
assert ENGINE_CONFIG["disable_custom_all_reduce"] is True
assert ENGINE_CONFIG["enable_prefix_caching"] is False
assert "disable_log_stats" not in ENGINE_CONFIG  # throughput instrumentation is NOT copied into solver.
assert ENGINE_CONFIG["tensor_parallel_size"] == 4
assert ENGINE_CONFIG["max_num_seqs"] == 16
assert ENGINE_CONFIG["max_num_batched_tokens"] == 32768
assert ENGINE_CONFIG["max_model_len"] == 29696
assert abs(float(ENGINE_CONFIG["gpu_memory_utilization"]) - 0.90) < 1e-12
print("C1 causal engine assertion PASS:", _engine_diffs)
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    StructuredOutputsParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by TERMINAL_PILOT.")
else:
    ENGINE_BOOTSTRAP_OK = False
    ENGINE_BOOTSTRAP_STAGE = "real_runtime_import"
    ENGINE_BOOTSTRAP_ERROR = None
    ENGINE_BOOTSTRAP_TRACEBACK = None
    _bootstrap_t0 = time.perf_counter()
    try:
        import torch
        import vllm
        from vllm import LLM, SamplingParams
        from vllm.sampling_params import StructuredOutputsParams
        RUNTIME_VLLM_VERSION = vllm.__version__
        RUNTIME_TORCH_VERSION = torch.__version__
        RUNTIME_CUDA_VERSION = torch.version.cuda
        print("=" * 116); print("REAL vLLM / CUDA PREFLIGHT"); print("=" * 116)
        print("PyTorch :", RUNTIME_TORCH_VERSION); print("CUDA    :", RUNTIME_CUDA_VERSION); print("vLLM    :", RUNTIME_VLLM_VERSION)
        print("Transformers:", pkg_version("transformers"))
        if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]: raise RuntimeError("vLLM runtime drift after import.")
        if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]: raise RuntimeError("Transformers runtime drift after vLLM setup.")
        ENGINE_BOOTSTRAP_STAGE = "sampling_api_probe"
        _sampling_api_probe = SamplingParams(temperature=1.0,top_p=0.95,top_k=20,min_p=0.0,presence_penalty=0.0,
                                             repetition_penalty=1.0,max_tokens=16,ignore_eos=False,seed=SEED,
                                             skip_special_tokens=False,spaces_between_special_tokens=False)
        _structured_api_probe = StructuredOutputsParams(json=FINALIZER_JSON_SCHEMA)
        _structured_sampling_probe = SamplingParams(
            temperature=FINALIZER_SAMPLING["temperature"], top_p=FINALIZER_SAMPLING["top_p"],
            top_k=FINALIZER_SAMPLING["top_k"], min_p=FINALIZER_SAMPLING["min_p"],
            presence_penalty=FINALIZER_SAMPLING["presence_penalty"],
            repetition_penalty=FINALIZER_SAMPLING["repetition_penalty"],
            max_tokens=64, structured_outputs=_structured_api_probe,
        )
        print("SamplingParams + StructuredOutputsParams API self-test: PASS")
        ENGINE_BOOTSTRAP_STAGE = "cuda_topology_check"
        print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4"); print("GPUs visible:", torch.cuda.device_count())
        if not torch.cuda.is_available(): raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
        if torch.cuda.device_count() != 4: raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")
        for i in range(4):
            cc=torch.cuda.get_device_capability(i); mem=torch.cuda.get_device_properties(i).total_memory/1024**3; name=torch.cuda.get_device_name(i)
            gpu_info.append({"gpu":i,"name":name,"memory_gib":mem,"cc":f"{cc[0]}.{cc[1]}"}); print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")
        ENGINE_BOOTSTRAP_STAGE = "model_allocation"
        load_start=time.perf_counter(); arc_llm=LLM(**ENGINE_CONFIG); model_load_sec=time.perf_counter()-load_start
        print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")
        ENGINE_BOOTSTRAP_STAGE = "warmup_generate"
        _active_prompts=[r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET]
        _med=float(np.median([r["prompt_tokens"] for r in _active_prompts])); probe_record=min(_active_prompts,key=lambda r:abs(r["prompt_tokens"]-_med))
        warm_params=SamplingParams(temperature=0.0,max_tokens=32,ignore_eos=False,skip_special_tokens=False)
        t0=time.perf_counter(); _probe_ids=tokenizer.encode(probe_record["prompt"],add_special_tokens=False)
        warm=arc_llm.generate([probe_record["prompt"],{"prompt_token_ids":list(_probe_ids)}],[warm_params,warm_params],use_tqdm=False)
        warmup_sec=time.perf_counter()-t0
        assert len(warm)==2 and all(len(w.outputs)>=1 for w in warm)
        if not list(getattr(warm[0],"prompt_token_ids",None) or []): raise RuntimeError("RequestOutput.prompt_token_ids unavailable in this vLLM build; token-exact continuation cannot run.")

        # v2.4 GPU-only, score-excluded terminal positive control: tests non-thinking/no-tools render,
        # structured-output backend, JSON parsing, and standalone solution() syntax on the real stack.
        if RUN_PROFILE in {"TERMINAL_PILOT","ADAPTIVE_SMOKE"}:
            ENGINE_BOOTSTRAP_STAGE="terminal_positive_control"
            _pc_messages=[
                {"role":"system","content":"Return only the requested structured final artifact."},
                {"role":"user","content":"Return a standalone identity ARC program defining solution(grid) that returns a copy of grid."},
            ]
            _pc_prompt,_pc_method=render_terminal_chat(_pc_messages)
            _pc_schema=StructuredOutputsParams(json=FINALIZER_JSON_SCHEMA)
            _pc_params=SamplingParams(
                temperature=FINALIZER_SAMPLING["temperature"], top_p=FINALIZER_SAMPLING["top_p"],
                top_k=FINALIZER_SAMPLING["top_k"], min_p=FINALIZER_SAMPLING["min_p"],
                presence_penalty=FINALIZER_SAMPLING["presence_penalty"],
                repetition_penalty=FINALIZER_SAMPLING["repetition_penalty"],
                max_tokens=256, ignore_eos=False, seed=SEED+9_000_000,
                skip_special_tokens=False, spaces_between_special_tokens=False,
                structured_outputs=_pc_schema,
            )
            _pc_t0=time.perf_counter()
            _pc_req=arc_llm.generate([_pc_prompt],[_pc_params],use_tqdm=False)[0]
            _pc_sec=time.perf_counter()-_pc_t0
            _pc_text=_pc_req.outputs[0].text if getattr(_pc_req,"outputs",None) else ""
            # v2.4B CRITICAL FIX: sampling uses skip_special_tokens=False, so the decoded text ends with
            # <|im_end|>. The old json.loads(_pc_text) therefore raised JSONDecodeError on every
            # structurally CORRECT answer, failing engine bootstrap and skipping all real generation.
            # Use the same hardened parser the scheduler uses, so the control tests the production path.
            _pc_parsed=parse_terminal_json_output(_pc_text)
            if not _pc_parsed["ok"]:
                raise RuntimeError(f"terminal positive control failed at stage {_pc_parsed['status']!r}; raw tail={_pc_text[-200:]!r}")
            _pc_code=_pc_parsed["solution_code"]
            _pc_verify_task={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
                             "test":[{"input":[[5,6],[7,8]]}]}
            _pc_verify=fresh_verify_solution(_pc_code,_pc_verify_task,timeout_sec=min(5.0,VERIFIER_TIMEOUT_SEC))
            if not (_pc_verify.get("all_train_exact") and _pc_verify.get("test_execution_ok")):
                raise RuntimeError(f"terminal positive control fresh verifier failed: {_pc_verify}")
            TERMINAL_POSITIVE_CONTROL={
                "ran":True,"ok":True,"render_method":_pc_method,
                "prompt_sha256":hashlib.sha256(_pc_prompt.encode()).hexdigest(),
                "output_sha256":hashlib.sha256(_pc_text.encode()).hexdigest(),
                "output_tokens":len(getattr(_pc_req.outputs[0],"token_ids",[]) or []),
                "finish_reason":str(getattr(_pc_req.outputs[0],"finish_reason",None)),
                "elapsed_sec":float(_pc_sec),"score_excluded":True,
                "parse_status":_pc_parsed["status"],"fresh_verifier_status":_pc_verify.get("verifier_status"),
                "raw_text_had_control_tokens":bool(_pc_text!=strip_model_control_tokens(_pc_text)),
            }
            (WORK_ROOT/"terminal_positive_control.json").write_text(json.dumps(TERMINAL_POSITIVE_CONTROL,indent=2),encoding="utf-8")
            print("TERMINAL POSITIVE CONTROL PASS:",TERMINAL_POSITIVE_CONTROL)

        ENGINE_BOOTSTRAP_OK=True; ENGINE_BOOTSTRAP_STAGE="complete"
        print(f"REAL MODEL WARMUP (text + token-prompt continuation API + prompt_token_ids): PASS ({warmup_sec:.2f} sec)")
    except BaseException as _bootstrap_exc:
        ENGINE_BOOTSTRAP_ERROR=f"{type(_bootstrap_exc).__name__}: {_bootstrap_exc}"; ENGINE_BOOTSTRAP_TRACEBACK=traceback.format_exc(limit=40)
        _bootstrap_payload={"run_id":RUN_ID,"run_profile":RUN_PROFILE,"ok":False,"failed_stage":ENGINE_BOOTSTRAP_STAGE,
                            "error":ENGINE_BOOTSTRAP_ERROR,"traceback":ENGINE_BOOTSTRAP_TRACEBACK,
                            "elapsed_sec":float(time.perf_counter()-_bootstrap_t0),"engine_config_sha256":ENGINE_CONFIG_SHA256,
                            "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256}
        (WORK_ROOT/"ENGINE_BOOTSTRAP_FAILURE.json").write_text(json.dumps(_bootstrap_payload,indent=2),encoding="utf-8")
        print("ENGINE BOOTSTRAP FAILURE CAPTURED:",ENGINE_BOOTSTRAP_ERROR)
        print("Downstream cells will write terminal infrastructure-failure records and still build the evidence bundle.")
        arc_llm=None

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
ENGINE_BOOTSTRAP_STATUS = {
    "run_id":RUN_ID,"run_profile":RUN_PROFILE,"use_model":USE_MODEL,"ok":bool(ENGINE_BOOTSTRAP_OK),
    "stage":ENGINE_BOOTSTRAP_STAGE,"error":ENGINE_BOOTSTRAP_ERROR,"model_load_sec":float(model_load_sec),
    "warmup_sec":float(warmup_sec),"gpu_info":gpu_info,"engine_config_sha256":ENGINE_CONFIG_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
}
(WORK_ROOT/"ENGINE_BOOTSTRAP_STATUS.json").write_text(json.dumps(ENGINE_BOOTSTRAP_STATUS,indent=2,default=str),encoding="utf-8")
print("Engine bootstrap status    :",{k:ENGINE_BOOTSTRAP_STATUS[k] for k in ("ok","stage","error")})
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


In [ ]:
# =====================================================================================
# CELL 8 — PIV-Q27-P0 BOUNDED STATEFUL AGENT LOOP + CONTRACT-SAFE CHECKPOINTS
# =====================================================================================
# v1.8: artifact paths are arm-scoped; set_arm_paths()/activate_arm() (defined below) point them at WORK_ROOT/arms/<arm>/.
RAW_GENERATIONS_PATH=WORK_ROOT/"raw_generations.jsonl"
TRACE_DIR=WORK_ROOT/"traces"; TRACE_DIR.mkdir(exist_ok=True)
WAVE_METRICS_PATH=WORK_ROOT/"wave_metrics.jsonl"
AGENT_LOOP_RUNTIME_PATH=WORK_ROOT/"agent_loop_runtime.json"
ARM_DIR=WORK_ROOT

PIV_SAMPLING={
    "temperature":1.0,"top_p":0.95,"top_k":20,"min_p":0.0,
    "presence_penalty":0.0,"repetition_penalty":1.0,"seed":SEED+3000,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,
}
QWEN_OFFICIAL_SAMPLING_REFERENCE={"thinking":{k:PIV_SAMPLING[k] for k in ["temperature","top_p","top_k","min_p","presence_penalty","repetition_penalty"]}}
SAMPLING_CONFIG_SHA256=stable_json_sha256(PIV_SAMPLING)
AGENT_BUDGET={
    "agent_batch_size":AGENT_BATCH_SIZE,"max_tool_calls":MAX_TOOL_CALLS,"max_model_turns":MAX_MODEL_TURNS,
    "max_retries":MAX_RETRIES,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,"total_generated_token_cap":TOTAL_GENERATED_TOKEN_CAP,
    "tool_timeout_sec":TOOL_TIMEOUT_SEC,"verifier_timeout_sec":VERIFIER_TIMEOUT_SEC,
    "verifier_contract_version":VERIFIER_CONTRACT_VERSION,"verifier_timeout_contract":VERIFIER_TIMEOUT_CONTRACT,
    "max_tool_output_chars":MAX_TOOL_OUTPUT_CHARS,"history_policy":"preserve_thinking_and_tool_history",
    "arms":{a:ARM_CONFIG[a] for a in ARMS},"action_max_tokens":ACTION_MAX_TOKENS,"stop_thinking_sentence_sha256":hashlib.sha256(STOP_THINKING_SENTENCE.encode()).hexdigest(),
    "terminal_commit":{"enabled":TERMINAL_POLICY_ENABLED,"final_reserve_tokens":FINAL_RESERVE_TOKENS,"exploration_token_cap":EXPLORATION_TOKEN_CAP,"finalizer_max_tokens":FINALIZER_MAX_TOKENS,"finalizer_turn_slots":FINALIZER_TURN_SLOTS,"context_reserve_tokens":FINALIZER_CONTEXT_RESERVE_TOKENS,"handoff_max_tokens":TERMINAL_HANDOFF_MAX_TOKENS,"sampling":FINALIZER_SAMPLING,"json_schema":FINALIZER_JSON_SCHEMA},
}
AGENT_BUDGET_SHA256=stable_json_sha256(AGENT_BUDGET)

ENGINE_RUNTIME_CONTRACT={
    "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
    "engine_config_sha256":ENGINE_CONFIG_SHA256,
    "chat_template_sha256":template_sha,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
}

def generation_contract_payload(task_id):
    pr=PROMPT_RECORD_MAP[task_id]
    return {
        "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
        "prompt_contract_version":PROMPT_CONTRACT_VERSION,"experiment_id":EXPERIMENT_ID,"run_profile":RUN_PROFILE,
        "task_set":TASK_SET,"task_id":task_id,"n_train":pr["n_train"],"n_test":pr["n_test"],
        "initial_prompt_sha256":pr["prompt_sha256"],"piv_source_commit":PIV_SOURCE_COMMIT,
        "sampling_config_sha256":SAMPLING_CONFIG_SHA256,"agent_budget_sha256":AGENT_BUDGET_SHA256,
        "verifier_contract_version":VERIFIER_CONTRACT_VERSION,"verifier_timeout_contract":VERIFIER_TIMEOUT_CONTRACT,
        "arm":CURRENT_ARM,"reasoning_effort":ARM_CONFIG[CURRENT_ARM]["reasoning_effort"],"think_policy":ARM_CONFIG[CURRENT_ARM]["think_policy"],
        "think_budget":ARM_CONFIG[CURRENT_ARM]["think_budget"],"action_max_tokens":ACTION_MAX_TOKENS,
        "terminal_policy_enabled":TERMINAL_POLICY_ENABLED,"exploration_token_cap":EXPLORATION_TOKEN_CAP,"final_reserve_tokens":FINAL_RESERVE_TOKENS,"finalizer_json_schema_sha256":stable_json_sha256(FINALIZER_JSON_SCHEMA),
        **ENGINE_RUNTIME_CONTRACT,
    }
def current_contract_sha(task_id): return stable_json_sha256(generation_contract_payload(task_id))

def append_record(rec):
    with open(RAW_GENERATIONS_PATH,"a",encoding="utf-8") as f:
        f.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")

def read_existing_records():
    if not RAW_GENERATIONS_PATH.exists(): return []
    out=[]
    for i,line in enumerate(RAW_GENERATIONS_PATH.read_text(encoding="utf-8").splitlines()):
        if not line.strip(): continue
        try:
            r=json.loads(line); r["_checkpoint_order"]=i; out.append(r)
        except Exception: pass
    return out

def record_compatible(r):
    tid=r.get("task_id")
    if tid not in ACTIVE_TASK_SET: return False,"foreign_task"
    if r.get("generation_contract_sha256") != current_contract_sha(tid): return False,"contract_mismatch"
    return True,"ok"

def audit_checkpoint_records(records):
    compatible=[]; stale=[]; foreign=[]
    for r in records:
        ok,reason=record_compatible(r)
        if ok: compatible.append(r)
        elif reason=="foreign_task": foreign.append({"task_id":r.get("task_id"),"record_id":r.get("record_id")})
        else: stale.append({"task_id":r.get("task_id"),"record_id":r.get("record_id"),"reason":reason})
    return compatible,{"raw_rows":len(records),"compatible_rows":len(compatible),"stale_rows":len(stale),"foreign_rows":len(foreign),
                       "stale_preview":stale[:20],"foreign_preview":foreign[:20]}

def terminal_completed_task_ids(records):
    out=set()
    for r in records:
        ok,_=record_compatible(r)
        if ok and r.get("status")=="complete": out.add(r["task_id"])
    return out

def save_event(task_id,event):
    p=TRACE_DIR/f"{task_id}.jsonl"
    with open(p,"a",encoding="utf-8") as f: f.write(json.dumps(event,ensure_ascii=False,default=str)+"\n")

def assistant_tool_message(parsed):
    calls=[]
    for tc in parsed.get("tool_calls",[]):
        calls.append({"type":"function","function":{"name":tc["name"],"arguments":tc["arguments"]}})
    return {
        "role":"assistant","content":parsed.get("content","") or "",
        "reasoning_content":parsed.get("reasoning_content","") or "",
        "tool_calls":calls,
    }

def complete_record(state, task, verifier, candidate_status, stop_reason, error=""):
    test_outputs=verifier.get("test_outputs",[]) if verifier else []
    code=state.get("final_code")
    _verifier_infra=bool(verifier and verifier.get("infrastructure_failure"))
    _verifier_infra_errors=list((verifier or {}).get("infrastructure_errors") or [])
    _state_infra_errors=list(state.get("infrastructure_errors",[]) or [])
    _merged_infra_errors=_state_infra_errors + [x for x in _verifier_infra_errors if x not in _state_infra_errors]
    try:
        _final_tree=py_ast.parse(code) if code else None
        _final_syntax_valid=bool(_final_tree and any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in _final_tree.body))
    except Exception:
        _final_syntax_valid=False
    rec={
        "run_id":RUN_ID,"record_id":str(uuid.uuid4()),"task_id":state["task_id"],"branch":f"piv_{CURRENT_ARM}","arm":CURRENT_ARM,
        "reasoning_effort":ARM_CONFIG[CURRENT_ARM]["reasoning_effort"],"think_policy":ARM_CONFIG[CURRENT_ARM]["think_policy"],"status":"complete",
        "candidate_status":candidate_status,"stop_reason":stop_reason,"error":error,
        "infrastructure_failure":bool(state.get("infrastructure_failure",False) or _verifier_infra),
        "infrastructure_errors":_merged_infra_errors,
        "verifier_infrastructure_failure":_verifier_infra,
        "verifier_infrastructure_errors":_verifier_infra_errors,
        "generated_tokens_observed_lower_bound":bool(state.get("generated_tokens_observed_lower_bound",False)),
        "phase2_tokens_unknown":bool(state.get("phase2_tokens_unknown",False)),
        "finalizer_started":bool(state.get("finalizer_started",False)),
        "finalizer_trigger_reason":state.get("finalizer_trigger_reason"),
        "finalizer_prompt_tokens":state.get("finalizer_prompt_tokens"),
        "finalizer_output_tokens":state.get("finalizer_output_tokens"),
        "finalizer_finish_reason":state.get("finalizer_finish_reason"),
        "terminal_parse_status":state.get("terminal_parse_status"),
        "terminal_render_method":state.get("terminal_render_method"),
        "terminal_prompt_sha256":state.get("terminal_prompt_sha256"),
        "feedback_present_in_rendered_prompt":state.get("feedback_present_in_rendered_prompt"),
        "terminal_handoff_source_tokens":state.get("terminal_handoff_source_tokens",0),
        "terminal_handoff_kept_tokens":state.get("terminal_handoff_kept_tokens",0),
        "terminal_handoff_truncated":bool(state.get("terminal_handoff_truncated",False)),
        "n_train":len(task["train"]),"n_test":len(task["test"]),"seed":PIV_SAMPLING["seed"],
        "sampling_seeds":state.get("sampling_seeds",[]),"phase2_sampling_seeds":state.get("phase2_sampling_seeds",[]),"source_retries":state.get("source_retries",0),
        "step_retries_total":state.get("step_retries_total",0),"retry_reasons":state.get("retry_reasons",[]),
        "budget_denials":state.get("budget_denials",0),
        "guard_rejections":state.get("guard_rejections",0),"salvaged_final_on_length":state.get("salvaged_final_on_length",0),
        # v1.8 measurement-contract counters
        "think_tokens_total":state.get("think_tokens_total",0),"natural_think_closes":state.get("natural_think_closes",0),
        "forced_think_closes":state.get("forced_think_closes",0),"injected_stop_tokens_total":state.get("injected_stop_tokens_total",0),
        "action_tokens_total":state.get("action_tokens_total",0),"first_tool_turn":state.get("first_tool_turn"),
        "first_tool_generated_tokens":state.get("first_tool_generated_tokens"),"tool_ok_count":state.get("tool_ok_count",0),
        "tool_exception_count":state.get("tool_exception_count",0),"unique_tool_code_hashes":len(state.get("tool_code_hashes",[])),
        "turn_kinds":state.get("turn_kinds",[]),"post_close_actions":state.get("post_close_actions",[]),
        "train_exact_count":(sum(1 for x in (verifier.get("train") or []) if x.get("exact")) if verifier else None),
        "repeated_tool_calls":state.get("repeated_tool_calls",0),"stateful_reuse_events":state.get("stateful_reuse_events",0),
        "stateful_reused_symbols":state.get("stateful_reused_symbols",[]),"natural_continuations":state.get("natural_continuations",0),
        "phase2_length_turns":state.get("phase2_length_turns",0),"stop_without_think_close":state.get("stop_without_think_close",0),
        "model_turns":state["model_turns"],"tool_calls":state["tool_calls"],"tool_errors":state["tool_errors"],
        "tool_timeouts":state["tool_timeouts"],"worker_restarts":state["worker_restarts"],
        "generated_tokens":state["generated_tokens"],"max_rendered_prompt_tokens":state["max_prompt_tokens"],
        "finish_reasons":state["finish_reasons"],"final_code":code,"final_code_sha256":hashlib.sha256((code or "").encode()).hexdigest() if code else None,
        "final_code_syntax_valid":_final_syntax_valid,"verifier_invoked":bool(verifier is not None),
        "verifier_executed":bool(verifier and verifier.get("verifier_started")),
        "all_train_exact":bool(verifier and verifier.get("all_train_exact")),
        "test_execution_ok":bool(verifier and verifier.get("test_execution_ok")),
        "any_test_output_ok":bool(verifier and verifier.get("any_test_output_ok")),
        "test_errors":verifier.get("test_errors",[]) if verifier else [],
        "verifier_status":verifier.get("verifier_status") if verifier else None,
        "verifier_error":(verifier.get("train_error") or verifier.get("test_error") or verifier.get("error")) if verifier else None,
        "test_outputs":test_outputs,
        "initial_prompt_sha256":PROMPT_RECORD_MAP[state["task_id"]]["prompt_sha256"],
        "generation_contract_sha256":current_contract_sha(state["task_id"]),"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
        "agent_budget_sha256":AGENT_BUDGET_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,**ENGINE_RUNTIME_CONTRACT,
    }
    return rec

def step_retry_available(state):
    return (
        state.get("step_retry_count",0) < MAX_RETRIES
        and state.get("generated_tokens",0) < TOTAL_GENERATED_TOKEN_CAP
        and state.get("model_turns",0) < MAX_MODEL_TURNS
    )

def register_step_retry(state, reason):
    """PiVOTools inner tool-loop retry: same messages + same persistent REPL state."""
    if not step_retry_available(state):
        return False
    state["step_retry_count"] = int(state.get("step_retry_count",0)) + 1
    state["step_retries_total"] = int(state.get("step_retries_total",0)) + 1
    state["sampling_retry_nonce"] = int(state.get("sampling_retry_nonce",0)) + 1
    state.setdefault("retry_reasons",[]).append(f"step:{reason}")
    return True

def run_retry_policy_preflight_test():
    st={"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
        "generated_tokens":100,"model_turns":1,"tool_calls":3,
        "messages":[{"role":"user","content":"sentinel-history"}],"retry_reasons":[]}
    before_messages=copy.deepcopy(st["messages"]); before_calls=st["tool_calls"]
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert st["messages"]==before_messages and st["tool_calls"]==before_calls
    assert st["step_retry_count"]==1 and st["sampling_retry_nonce"]==1
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert not register_step_retry(st,"invalid_finish_reason:length")
    return {"ok":True,"step_retries_total":st["step_retries_total"],
            "history_preserved":st["messages"]==before_messages,"tool_calls_preserved":st["tool_calls"]==before_calls}

def run_fake_scheduler_preflight_test():
    """Scripted length -> same-history retry -> Python tool -> final program -> fresh verifier."""
    fake={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
          "test":[{"input":[[5,6],[7,8]]}]}
    st={"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
        "generated_tokens":200,"model_turns":1,"tool_calls":0,
        "messages":[{"role":"user","content":"fake"}],"retry_reasons":[]}
    history_before=copy.deepcopy(st["messages"])
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert st["messages"]==history_before
    worker=PersistentPythonWorker(task_without_test_solutions(fake),timeout_sec=5.0)
    try:
        tool_text="inspect\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>"
        p=parse_agent_turn(tool_text); assert p["kind"]=="tool"
        st["step_retry_count"]=0
        rr=worker.execute(p["tool_calls"][0]["arguments"]["code"]); assert rr["ok"]
        st["tool_calls"]+=1
        final_text="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
        pf=parse_agent_turn(final_text); assert pf["kind"]=="final"
        vr=fresh_verify_solution(pf["final_code"],fake,timeout_sec=5)
        assert vr["all_train_exact"] and vr["test_execution_ok"]
    finally:
        worker.close()
    return {"ok":True,"tool_calls":st["tool_calls"],"verifier_status":vr["verifier_status"]}

def run_scripted_preflight_agent_loop_test():
    """CPU-only scripted tool->feedback->final-code path using the same parser/worker/verifier pieces."""
    fake={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
          "test":[{"input":[[5,6],[7,8]]}]}
    worker=PersistentPythonWorker(task_without_test_solutions(fake),timeout_sec=5.0)
    try:
        raw1="inspect\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>"
        p1=parse_agent_turn(raw1)
        assert p1["kind"]=="tool" and len(p1["tool_calls"])==1
        rr=worker.execute(p1["tool_calls"][0]["arguments"]["code"])
        assert rr["ok"] and "dict_keys" in rr["text"]
        raw2="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
        p2=parse_agent_turn(raw2)
        assert p2["kind"]=="final"
        vr=fresh_verify_solution(p2["final_code"],fake,timeout_sec=5)
        assert vr["all_train_exact"] and vr["test_outputs"]==[[[5,6],[7,8]]]
        payload={"ok":True,"tool_feedback":rr["text"],"verifier_status":vr["verifier_status"]}
    finally:
        worker.close()
    with open(WORK_ROOT/"scripted_agent_loop_selftest.json","w") as f:
        json.dump(payload,f,indent=2)
    print("Scripted CPU agent-loop self-test PASS:",payload)

def run_multicall_budget_accounting_selftest():
    max_calls=1
    executed=0
    outcomes=[]
    for guard_rejected in (True,False):
        if executed>=max_calls:
            outcomes.append("denied")
            continue
        if guard_rejected:
            outcomes.append("guard_rejected")
        else:
            executed+=1
            outcomes.append("executed")
    assert outcomes==["guard_rejected","executed"] and executed==1
    return {"ok":True,"outcomes":outcomes,"executed_calls":executed}

def append_wave_metric(row):
    with open(WAVE_METRICS_PATH,"a",encoding="utf-8") as _f:
        _f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")

# ---- v1.8: arm-aware paths -------------------------------------------------------------------
def token_count(text):
    """Approximate token length of a text span with the mounted tokenizer (used for think-token accounting)."""
    if not text: return 0
    return len(tokenizer.encode(str(text), add_special_tokens=False))

THINK_CLOSE_IDS = list(tokenizer.encode("</think>", add_special_tokens=False))
def _find_subseq(seq, sub):
    if not sub or len(sub) > len(seq): return -1
    for i in range(len(seq) - len(sub) + 1):
        if seq[i:i + len(sub)] == sub: return i
    return -1

def split_think_post_tokens(token_ids, text):
    """Exact split of model-generated ids into (think, post-close) using the </think> token id sequence.
    Returns (think_n, post_n, natural_close). Falls back to a text-based estimate only if the id sequence is hidden."""
    ids = list(token_ids or [])
    idx = _find_subseq(ids, THINK_CLOSE_IDS)
    if idx >= 0:
        end = idx + len(THINK_CLOSE_IDS); return end, max(0, len(ids) - end), True
    if "</think>" in (text or ""):
        n = min(len(ids), token_count((text or "").split("</think>", 1)[0] + "</think>")); return n, max(0, len(ids) - n), True
    return len(ids), 0, False

def code_state_effects(state, code):
    """Deterministic proxy for stateful-REPL benefit: names loaded now that were defined by an earlier call."""
    try: tree = py_ast.parse(code)
    except Exception: return {"reuse": [], "defined": []}
    loaded = set(); defined = set()
    for node in py_ast.walk(tree):
        if isinstance(node, py_ast.Name):
            if isinstance(node.ctx, py_ast.Load): loaded.add(node.id)
            elif isinstance(node.ctx, (py_ast.Store, py_ast.Param)): defined.add(node.id)
        elif isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef, py_ast.ClassDef)): defined.add(node.name)
        elif isinstance(node, py_ast.Import):
            for a in node.names: defined.add(a.asname or a.name.split(".")[0])
        elif isinstance(node, py_ast.ImportFrom):
            for a in node.names: defined.add(a.asname or a.name)
    prior = set(state.get("defined_symbols", []))
    reuse = sorted(loaded & prior)
    state["defined_symbols"] = sorted(prior | defined)
    if reuse:
        state["stateful_reuse_events"] = int(state.get("stateful_reuse_events", 0)) + 1
        state["stateful_reused_symbols"] = sorted(set(state.get("stateful_reused_symbols", [])) | set(reuse))
    return {"reuse": reuse, "defined": sorted(defined)}

def set_arm_paths(arm):
    """All checkpoint/trace/wave/runtime artifacts live under WORK_ROOT/arms/<arm>/."""
    global RAW_GENERATIONS_PATH, TRACE_DIR, WAVE_METRICS_PATH, AGENT_LOOP_RUNTIME_PATH, ARM_DIR
    ARM_DIR = WORK_ROOT / "arms" / arm
    ARM_DIR.mkdir(parents=True, exist_ok=True)
    RAW_GENERATIONS_PATH = ARM_DIR / "raw_generations.jsonl"
    TRACE_DIR = ARM_DIR / "traces"; TRACE_DIR.mkdir(exist_ok=True)
    WAVE_METRICS_PATH = ARM_DIR / "wave_metrics.jsonl"
    AGENT_LOOP_RUNTIME_PATH = ARM_DIR / "agent_loop_runtime.json"
    return ARM_DIR

def activate_arm(arm):
    """Set the current arm, re-render the arm-specific initial prompts and point all artifact paths at the arm dir."""
    global CURRENT_ARM, PROMPT_RECORD_MAP, prompt_records, prompt_df
    if arm not in ARM_CONFIG: raise ValueError(f"unknown arm {arm!r}")
    CURRENT_ARM = arm
    prompt_records, prompt_df = build_prompt_records(arm)
    PROMPT_RECORD_MAP = {r["task_id"]: r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET}
    d = set_arm_paths(arm)
    prompt_df.to_csv(d / "prompt_stats.csv", index=False)
    return d

# v2.4B: parse_terminal_json_output() and strip_model_control_tokens() now live in Cell 6 so that the
# Cell 7 GPU positive control can use the identical parser. Keeping a second copy here would let the
# control and the scheduler drift apart, which is exactly how the v2.4A <|im_end|> defect stayed hidden.

def _latest_tool_feedback(messages):
    for m in reversed(list(messages or [])):
        if m.get("role")=="tool":
            return str(m.get("content") or "")
    return None

def _feedback_render_diag(messages, rendered):
    fb=_latest_tool_feedback(messages)
    if fb is None:
        return {"latest_feedback_sha256":None,"feedback_present_in_rendered_prompt":None}
    # v2.5 instrumentation fix only: Qwen chat templates may trim message-edge whitespace.
    # Keep the RAW hash for provenance, but use stripped exact presence as the transport assertion.
    canonical=fb.strip()
    return {
        "latest_feedback_sha256":hashlib.sha256(fb.encode("utf-8")).hexdigest(),
        "feedback_present_in_rendered_prompt":bool(canonical and canonical in rendered),
    }

def engine_output_cap_violations(outs, caps):
    """Pure detector: return {request_index: {observed_tokens, requested_cap}}.
    Cardinality is checked separately because a missing/extra RequestOutput destroys request mapping."""
    if len(outs) != len(caps):
        raise ValueError(f"output/cap cardinality mismatch: {len(outs)} outputs / {len(caps)} caps")
    bad={}
    for i,(req,cap) in enumerate(zip(outs,caps)):
        _outputs=list(getattr(req,"outputs",[]) or [])
        if not _outputs:
            continue
        nt=len(getattr(_outputs[0],"token_ids",[]) or [])
        if nt>int(cap):
            bad[i]={"observed_tokens":int(nt),"requested_cap":int(cap)}
    return bad

def exploration_cap_v24(*, generated_tokens, prompt_tokens, forced_policy=False, phase2=False):
    """Hard admission invariant: exploration can never consume terminal token/context reserve."""
    remaining_explore=max(0,int(EXPLORATION_TOKEN_CAP)-int(generated_tokens))
    context_allow=max(0,int(MAX_MODEL_LEN)-int(prompt_tokens)-int(CONTEXT_SAFETY_TOKENS)-int(FINALIZER_CONTEXT_RESERVE_TOKENS))
    cap=min(remaining_explore,context_allow)
    if forced_policy and not phase2:
        cap=min(cap,max(0,remaining_explore-int(ACTION_RESERVE_TOKENS)))
    if forced_policy and not phase2:
        # v2.4B: use the CURRENT arm's think budget. The hard-coded "ACT-BUDGET3072" only happened to be
        # correct because it is the single forced_close arm today; any second one would silently mis-cap.
        _tb=ARM_CONFIG.get(CURRENT_ARM,{}).get("think_budget")
        cap=min(cap,int(_tb or cap))
    elif not phase2:
        cap=min(cap,int(PER_TURN_MAX_TOKENS))
    else:
        cap=min(cap,int(ACTION_MAX_TOKENS))
    return max(0,int(cap))

def fit_tool_feedback_to_terminal_reserve(messages_before_tool, raw_tool, suffix=""):
    """Return model-visible tool feedback that *provably* leaves FINALIZER_MAX_TOKENS context capacity.

    v2.4B reserved context before exploration but appended up to 4,000 tool-output characters afterwards.
    That character cap could consume more than the 512-token margin. v2.4C fits feedback against the actual
    rendered terminal prompt using the mounted tokenizer, preserving as much head/tail evidence as possible.
    """
    max_terminal_prompt_tokens=int(MAX_MODEL_LEN-CONTEXT_SAFETY_TOKENS-FINALIZER_MAX_TOKENS)
    raw_compact=sanitize_tool_output(compact_tool_output(str(raw_tool or ""),limit=MAX_TOOL_OUTPUT_CHARS))
    raw_ids=list(tokenizer.encode(raw_compact,add_special_tokens=False))
    marker="\n...[tool feedback token-truncated to preserve 4096-token terminal reserve]...\n"
    marker_ids=list(tokenizer.encode(marker,add_special_tokens=False))

    def _body_for_keep(keep_tokens):
        keep_tokens=max(0,int(keep_tokens))
        if len(raw_ids)<=keep_tokens:
            body=raw_compact
        elif keep_tokens<=len(marker_ids)+2:
            body="[tool feedback omitted to preserve terminal finalizer context]"
        else:
            payload=max(0,keep_tokens-len(marker_ids))
            h=payload//2; t=payload-h
            chosen=raw_ids[:h] + marker_ids + (raw_ids[-t:] if t else [])
            body=tokenizer.decode(chosen,skip_special_tokens=False)
            body=sanitize_tool_output(body)
        return body + str(suffix or "")

    def _terminal_prompt_tokens(content):
        _msgs=copy.deepcopy(messages_before_tool)+[{"role":"tool","content":content},
                                                   {"role":"user","content":TERMINAL_FINALIZER_INSTRUCTION}]
        _rendered,_method=render_terminal_chat(_msgs)
        return len(tokenizer.encode(_rendered,add_special_tokens=False)),_method

    full_content=_body_for_keep(len(raw_ids))
    full_pt,method=_terminal_prompt_tokens(full_content)
    if full_pt<=max_terminal_prompt_tokens:
        return full_content,{"fits":True,"truncated":False,"terminal_prompt_tokens":full_pt,
                             "max_terminal_prompt_tokens":max_terminal_prompt_tokens,"render_method":method,
                             "raw_feedback_tokens":len(raw_ids),"kept_feedback_tokens":len(raw_ids)}

    # Binary search the largest feedback-token budget that preserves the exact rendered terminal capacity.
    lo,hi=0,len(raw_ids); best=None
    while lo<=hi:
        mid=(lo+hi)//2
        content=_body_for_keep(mid)
        pt,method=_terminal_prompt_tokens(content)
        if pt<=max_terminal_prompt_tokens:
            best=(content,pt,method,mid); lo=mid+1
        else:
            hi=mid-1
    if best is None:
        minimal=_body_for_keep(0)
        pt,method=_terminal_prompt_tokens(minimal)
        return minimal,{"fits":False,"truncated":True,"terminal_prompt_tokens":pt,
                        "max_terminal_prompt_tokens":max_terminal_prompt_tokens,"render_method":method,
                        "raw_feedback_tokens":len(raw_ids),"kept_feedback_tokens":0}
    content,pt,method,kept=best
    return content,{"fits":True,"truncated":True,"terminal_prompt_tokens":pt,
                    "max_terminal_prompt_tokens":max_terminal_prompt_tokens,"render_method":method,
                    "raw_feedback_tokens":len(raw_ids),"kept_feedback_tokens":int(kept)}

def _bounded_reasoning_handoff(text, max_tokens=TERMINAL_HANDOFF_MAX_TOKENS):
    """Keep the tail of the latest discarded incomplete reasoning as scratch context for terminal commit."""
    clean=sanitize_tool_output(str(text or "").strip())
    if not clean:
        return "",{"source_tokens":0,"kept_tokens":0,"truncated":False}
    ids=list(tokenizer.encode(clean,add_special_tokens=False))
    keep=min(len(ids),max(0,int(max_tokens)))
    chosen=ids[-keep:] if keep else []
    out=tokenizer.decode(chosen,skip_special_tokens=False) if chosen else ""
    out=sanitize_tool_output(out).strip()
    return out,{"source_tokens":len(ids),"kept_tokens":keep,"truncated":len(ids)>keep}

def _terminal_messages_with_reasoning_handoff(state):
    """Carry incomplete reasoning while preserving the full protected finalizer output capacity."""
    base=copy.deepcopy(state.get("messages") or [])
    handoff=str(state.get("terminal_handoff_reasoning") or "").strip()
    max_prompt=int(MAX_MODEL_LEN-CONTEXT_SAFETY_TOKENS-FINALIZER_MAX_TOKENS)
    label="\n\n[SCRATCH HANDOFF — latest incomplete exploration reasoning; not a committed answer]\n"
    if not handoff:
        msgs=base+[{"role":"user","content":TERMINAL_FINALIZER_INSTRUCTION}]
        rendered,method=render_terminal_chat(msgs)
        return msgs,rendered,method,{"present":False,"kept_tokens":0,"source_tokens":0,"prompt_tokens":len(tokenizer.encode(rendered,add_special_tokens=False)),"max_prompt_tokens":max_prompt}
    ids=list(tokenizer.encode(handoff,add_special_tokens=False)); lo,hi=0,min(len(ids),int(TERMINAL_HANDOFF_MAX_TOKENS)); best=None
    while lo<=hi:
        mid=(lo+hi)//2
        snippet=tokenizer.decode(ids[-mid:],skip_special_tokens=False) if mid else ""
        snippet=sanitize_tool_output(snippet).strip()
        inst=TERMINAL_FINALIZER_INSTRUCTION+(label+snippet if snippet else "")
        msgs=base+[{"role":"user","content":inst}]
        rendered,method=render_terminal_chat(msgs); pt=len(tokenizer.encode(rendered,add_special_tokens=False))
        if pt<=max_prompt: best=(msgs,rendered,method,mid,pt); lo=mid+1
        else: hi=mid-1
    if best is None:
        msgs=base+[{"role":"user","content":TERMINAL_FINALIZER_INSTRUCTION}]
        rendered,method=render_terminal_chat(msgs); pt=len(tokenizer.encode(rendered,add_special_tokens=False))
        return msgs,rendered,method,{"present":False,"kept_tokens":0,"source_tokens":len(ids),"prompt_tokens":pt,"max_prompt_tokens":max_prompt,"omitted_for_context":True}
    msgs,rendered,method,kept,pt=best
    return msgs,rendered,method,{"present":bool(kept),"kept_tokens":int(kept),"source_tokens":len(ids),"prompt_tokens":pt,"max_prompt_tokens":max_prompt,"truncated":len(ids)>int(kept)}

def _request_terminal_after_exhaustion(state, reason):
    """When exploration retries are exhausted, use the already-protected terminal slot."""
    if not TERMINAL_POLICY_ENABLED or state.get("done"): return False
    state["terminal_requested"]=True; state["finalizer_trigger_reason"]=str(reason); return True

def _terminal_trigger_reason(state):
    """Return the scheduler's immediate terminal-transition reason, or None.
    This helper is used by production and by PREFLIGHT so the trigger matrix cannot drift."""
    if not TERMINAL_POLICY_ENABLED or state.get("done"):
        return None
    if state.get("terminal_requested"):
        return str(state.get("finalizer_trigger_reason") or "terminal_requested")
    if int(state.get("generated_tokens") or 0) >= int(EXPLORATION_TOKEN_CAP):
        return "exploration_token_cap"
    if int(state.get("model_turns") or 0) >= int(MAX_MODEL_TURNS - FINALIZER_TURN_SLOTS):
        return "final_turn_slot"
    if int(state.get("tool_calls") or 0) >= int(MAX_TOOL_CALLS):
        return "tool_budget_exhausted"
    return None

def run_terminal_policy_preflight_selftest():
    # Overshoot counterexample from independent review: 5,000 total remain, but only 904 are exploration-admissible.
    generated=TOTAL_GENERATED_TOKEN_CAP-5000
    cap=exploration_cap_v24(generated_tokens=generated,prompt_tokens=1000,forced_policy=False)
    assert cap<=904,(cap,generated)
    assert generated+cap<=EXPLORATION_TOKEN_CAP
    # Context reserve is independently protected.
    near_context=MAX_MODEL_LEN-CONTEXT_SAFETY_TOKENS-FINALIZER_CONTEXT_RESERVE_TOKENS-100
    cap_ctx=exploration_cap_v24(generated_tokens=0,prompt_tokens=near_context,forced_policy=False)
    assert 0<=cap_ctx<=100,cap_ctx
    # Parser funnel: valid JSON/Python vs incomplete JSON vs REPL-dependent code.
    good=json.dumps({"solution_code":"def solution(grid):\n    return [row[:] for row in grid]"})
    pg=parse_terminal_json_output(good); assert pg["ok"] and pg["status"]=="ok"
    bad=parse_terminal_json_output('{"solution_code":"def solution(grid):')
    assert not bad["ok"] and bad["status"].startswith("json_parse_error")
    dep=json.dumps({"solution_code":"def solution(grid):\n    return helper(grid)"})
    pdp=parse_terminal_json_output(dep)
    # Syntax/guard can pass even if semantic dependency is unresolved; fresh verifier remains authority.
    assert pdp["ok"],pdp
    # Terminal render must be non-thinking/no-tool on a fresh probe.
    probe,_=render_terminal_chat([{"role":"system","content":"S"},{"role":"user","content":"U"}])
    assert "<function=python>" not in probe and "<tools>" not in probe
    assert not probe.rstrip().endswith("<think>")
    # v2.4C: pathological 4,000-char tool feedback must be token-truncated so a full terminal request still fits.
    _hist=[{"role":"system","content":"S"},{"role":"user","content":"U"},
           {"role":"assistant","content":"","reasoning_content":"","tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"print(1)"}}}]}]
    _fitted,_fitdiag=fit_tool_feedback_to_terminal_reserve(_hist,"[]{}()<>=,:;0123456789"*600)
    assert _fitdiag["fits"] and _fitdiag["terminal_prompt_tokens"]<=_fitdiag["max_terminal_prompt_tokens"],_fitdiag
    _handoff,_hdiag=_bounded_reasoning_handoff("V24D_REASONING_HANDOFF_SENTINEL "+("hypothesis "*1200))
    _hst={"messages":[{"role":"system","content":"S"},{"role":"user","content":"U"}],"terminal_handoff_reasoning":_handoff}
    _hmsgs,_hrender,_hmethod,_hfit=_terminal_messages_with_reasoning_handoff(_hst)
    assert _hfit["prompt_tokens"]<=_hfit["max_prompt_tokens"],_hfit
    assert "hypothesis" in _hrender and _hfit["kept_tokens"]>0
    # v2.4F: PREFLIGHT's profile-level TERMINAL_POLICY_ENABLED is False by design, but this
    # self-test validates the shipped terminal helper. Exercise it under an explicit temporary ON state.
    _orig_terminal_policy=TERMINAL_POLICY_ENABLED
    try:
        globals()["TERMINAL_POLICY_ENABLED"]=True
        _rst={"done":False,"terminal_requested":False,"finalizer_trigger_reason":None}
        assert _request_terminal_after_exhaustion(_rst,"retry_exhausted") and _rst["terminal_requested"]
        _trigger_cases={
            "exploration_token_cap":{"done":False,"terminal_requested":False,"generated_tokens":EXPLORATION_TOKEN_CAP,
                                     "model_turns":0,"tool_calls":0},
            "final_turn_slot":{"done":False,"terminal_requested":False,"generated_tokens":0,
                               "model_turns":MAX_MODEL_TURNS-FINALIZER_TURN_SLOTS,"tool_calls":0},
            "tool_budget_exhausted":{"done":False,"terminal_requested":False,"generated_tokens":0,
                                     "model_turns":0,"tool_calls":MAX_TOOL_CALLS},
            "already_requested":{"done":False,"terminal_requested":True,"finalizer_trigger_reason":"retry_exhausted",
                                 "generated_tokens":0,"model_turns":0,"tool_calls":0},
        }
        assert _terminal_trigger_reason(_trigger_cases["exploration_token_cap"])=="exploration_token_cap"
        assert _terminal_trigger_reason(_trigger_cases["final_turn_slot"])=="final_turn_slot"
        assert _terminal_trigger_reason(_trigger_cases["tool_budget_exhausted"])=="tool_budget_exhausted"
        assert _terminal_trigger_reason(_trigger_cases["already_requested"])=="retry_exhausted"
        _admission_cap=exploration_cap_v24(generated_tokens=EXPLORATION_TOKEN_CAP-ACTION_RESERVE_TOKENS+1,
                                           prompt_tokens=1000,forced_policy=True,phase2=False)
        assert _admission_cap<32, _admission_cap
        # v2.4G: one over-cap response must be identifiable without poisoning its healthy neighbour.
        class _CapOut:
            def __init__(self,n): self.token_ids=list(range(n))
        class _CapReq:
            def __init__(self,n): self.outputs=[_CapOut(n)]
        _cap_bad=engine_output_cap_violations([_CapReq(10),_CapReq(21)],[10,20])
        assert _cap_bad=={1:{"observed_tokens":21,"requested_cap":20}}, _cap_bad
    finally:
        globals()["TERMINAL_POLICY_ENABLED"]=_orig_terminal_policy
    payload={"ok":True,"overshoot_cap":cap,"context_cap":cap_ctx,"good_parse":pg["status"],
             "incomplete_parse":bad["status"],"semantic_dependency_left_to_fresh_verifier":True,
             "terminal_render_sha256":hashlib.sha256(probe.encode()).hexdigest(),
             "terminal_safe_tool_feedback":_fitdiag,"reasoning_handoff_fit":_hfit,"terminal_trigger_matrix":["exploration_token_cap","final_turn_slot","tool_budget_exhausted","retry_exhausted","exploration_admission_reserve"],"per_task_engine_cap_detector":True}
    (WORK_ROOT/"terminal_policy_selftest.json").write_text(json.dumps(payload,indent=2),encoding="utf-8")
    print("v2.4 terminal-policy PREFLIGHT self-test PASS:",payload)
    return payload

# v2.5A: module-level request factories. CELL 8B (repair round) calls _mk_finalizer_params from module scope,
# so these must not be nested inside run_real_piv().
def _mk_params(cap, seed):
    return SamplingParams(
        temperature=PIV_SAMPLING["temperature"], top_p=PIV_SAMPLING["top_p"], top_k=PIV_SAMPLING["top_k"],
        min_p=PIV_SAMPLING["min_p"], presence_penalty=PIV_SAMPLING["presence_penalty"],
        repetition_penalty=PIV_SAMPLING["repetition_penalty"], max_tokens=int(cap), ignore_eos=False,
        seed=int(seed), skip_special_tokens=False, spaces_between_special_tokens=False)

def _mk_finalizer_params(cap, seed):
    if StructuredOutputsParams is None:
        raise RuntimeError("StructuredOutputsParams unavailable in real terminal-pilot profile")
    _so=StructuredOutputsParams(json=FINALIZER_JSON_SCHEMA)
    return SamplingParams(
        temperature=FINALIZER_SAMPLING["temperature"], top_p=FINALIZER_SAMPLING["top_p"],
        top_k=FINALIZER_SAMPLING["top_k"], min_p=FINALIZER_SAMPLING["min_p"],
        presence_penalty=FINALIZER_SAMPLING["presence_penalty"],
        repetition_penalty=FINALIZER_SAMPLING["repetition_penalty"], max_tokens=int(cap), ignore_eos=False,
        seed=int(seed), skip_special_tokens=False, spaces_between_special_tokens=False,
        structured_outputs=_so)

def run_real_piv():
    """v1.8 scheduler. Identical to v1.7 for natural-close arms; the forced_close arm generates each turn in two
    phases: (1) thinking up to THINK budget; if no </think> appeared, (2) token-exact continuation of the same
    request with the documented Qwen stop-thinking sentence + '</think>' injected, capped at ACTION_MAX_TOKENS.
    Injected tokens are accounted separately from model-generated tokens."""
    arm = CURRENT_ARM; acfg = ARM_CONFIG[arm]
    forced_policy = (acfg["think_policy"] == "forced_close")
    think_budget = int(acfg["think_budget"] or 0)
    stop_text = STOP_THINKING_TEXT   # v1.9: byte-exact Qwen-documented early-stopping text
    stop_ids = list(tokenizer.encode(stop_text, add_special_tokens=False))
    loop_t0 = time.perf_counter()
    pre_loop_elapsed_sec = float(ACT_FIXED_SETUP_ELAPSED_SEC)   # v1.9: model load + setup, identical for every arm
    global_wave_no = 0
    wave_spt_samples = []
    stop_all = False
    existing = read_existing_records(); completed = terminal_completed_task_ids(existing)
    pending = [t for t in ACTIVE_TASK_IDS if t not in completed]
    print(f"[{arm}] Resume: completed", len(completed), "pending", len(pending))

    def _terminalize_engine_exception(phase, affected_tids, exc):
        """Preserve terminal evidence when synchronous vLLM generate raises. Native hard-hangs remain a documented residual risk."""
        err=f"{type(exc).__name__}: {exc}"
        for _tid in affected_tids:
            st=states.get(_tid)
            if st is None or st.get("done"):
                continue
            st["infrastructure_failure"]=True
            st.setdefault("infrastructure_errors",[]).append({"phase":phase,"error":err})
            st["generated_tokens_observed_lower_bound"]=True
            st["done"]=True
            rec=complete_record(st,eval_challenges[_tid],None,"failed",f"engine_generate_exception_{phase}",err)
            rec["status"]="failed"; append_record(rec)
            save_event(_tid,{"run_id":RUN_ID,"arm":arm,"task_id":_tid,"parsed_kind":"engine_exception",
                             "phase":phase,"error":err,"terminalized":True})
        _err_path=ARM_DIR/f"engine_error_{phase}.json"
        _err_path.write_text(json.dumps({"run_id":RUN_ID,"arm":arm,"phase":phase,"task_ids":list(affected_tids),
                                         "error":err,"traceback":traceback.format_exc(limit=20)},indent=2),encoding="utf-8")
        print(f"[{arm}] ENGINE GENERATE EXCEPTION ({phase}): {err}; terminalized {len(affected_tids)} tasks")

    for batch_no, start_idx in enumerate(range(0, len(pending), AGENT_BATCH_SIZE), 1):
        tids = pending[start_idx:start_idx + AGENT_BATCH_SIZE]
        batch_wave_no = 0
        if stop_all:
            print("Projected-deadline stop already triggered; not starting batch", batch_no); break
        if time.perf_counter() >= deadline:
            print("Safety deadline reached before batch", batch_no); break
        workers = {}; states = {}

        def start_fresh_attempt(tid, st, reason=None):
            # PiVOTools AbstractTurn retries invalid/incomplete agentic calls from the initial prompt.
            # LOCAL fairness: retries share the same total token/tool/turn budgets; they do not create extra candidates.
            if reason is not None:
                if st["source_retries"] >= MAX_RETRIES:
                    return False
                if st["generated_tokens"] >= TOTAL_GENERATED_TOKEN_CAP or st["model_turns"] >= MAX_MODEL_TURNS:
                    return False
                st["source_retries"] += 1
                st["sampling_retry_nonce"] += 1
                st["retry_reasons"].append(str(reason))
            old = workers.get(tid)
            if old is not None:
                st["worker_restart_offset"] += int(getattr(old, "restart_count", 0))
                st["worker_restarts"] = st["worker_restart_offset"]
                try: old.close()
                except Exception: pass
            st["messages"] = copy.deepcopy(PROMPT_RECORD_MAP[tid]["initial_messages"])
            st["attempt_turns"] = 0
            st["step_retry_count"] = 0
            st["budget_nudged"] = False
            st["final_code"] = None
            try:
                workers[tid] = PersistentPythonWorker(task_without_test_solutions(eval_challenges[tid]), TOOL_TIMEOUT_SEC)
            except Exception as exc:
                err=f"{type(exc).__name__}: {exc}"
                st["infrastructure_failure"]=True
                st.setdefault("infrastructure_errors",[]).append({"phase":"repl_start","error":err})
                st["done"]=True
                rec=complete_record(st,eval_challenges[tid],None,"failed","repl_start_failure",err)
                rec["status"]="failed"; append_record(rec)
                save_event(tid,{"run_id":RUN_ID,"arm":arm,"task_id":tid,"parsed_kind":"repl_start_failure","error":err,"terminalized":True})
                return True
            return True

        def _terminalize_remaining(stop_reason, error_text):
            """v2.4B: when the batch loop aborts (engine exception / projected deadline) every task that
            has no terminal record yet would silently disappear from coverage. Record them explicitly."""
            _n=0
            for _tid in tids:
                _st=states.get(_tid)
                if _st is None or _st.get("done"): continue
                _st["infrastructure_failure"]=True
                _st.setdefault("infrastructure_errors",[]).append({"phase":stop_reason,"error":error_text})
                _st["generated_tokens_observed_lower_bound"]=True
                _st["done"]=True
                _rec=complete_record(_st,eval_challenges[_tid],None,"failed",stop_reason,error_text)
                _rec["status"]="failed"; append_record(_rec); _n+=1
            if _n: print(f"[{arm}] terminalized {_n} remaining task(s) as {stop_reason}")
            return _n

        def _assert_engine_output_count(phase, prompts, outs, caps=None):
            """Cardinality mismatch is wave-fatal because request/output mapping is no longer trustworthy.
            Per-request token-cap violations are isolated by _terminalize_overcap_outputs()."""
            if len(outs)!=len(prompts):
                raise RuntimeError(f"engine returned {len(outs)} outputs for {len(prompts)} {phase} prompts")
            if caps is not None and len(caps)!=len(prompts):
                raise RuntimeError(f"internal cap cardinality mismatch for {phase}: {len(caps)} caps / {len(prompts)} prompts")

        def _terminalize_overcap_outputs(phase, request_tids, outs, caps, turn_results=None):
            """Quarantine only RequestOutputs that exceed their own max_tokens cap.
            Healthy neighbours in the same vLLM batch remain usable. Returns the offending task-id set."""
            bad_idx=engine_output_cap_violations(outs,caps)
            bad_tids=set()
            for idx,diag in bad_idx.items():
                tid=request_tids[idx]; bad_tids.add(tid)
                st=states[tid]
                if st.get("done"):
                    continue
                req=outs[idx]
                _outputs=list(getattr(req,"outputs",[]) or [])
                _out=_outputs[0] if _outputs else None
                nt=int(diag["observed_tokens"]); cap=int(diag["requested_cap"])
                fr=str(getattr(_out,"finish_reason",None)) if _out is not None else "no_output"
                err=f"engine output exceeded max_tokens in {phase}: {nt} > {cap}"
                st["infrastructure_failure"]=True
                st.setdefault("infrastructure_errors",[]).append(
                    {"phase":phase,"error":err,"observed_tokens":nt,"requested_cap":cap})
                # Preserve exact observed accounting rather than pretending the offending output did not exist.
                if phase=="phase1":
                    st["model_turns"]+=1; st["attempt_turns"]+=1
                    st["generated_tokens"]+=nt; st["finish_reasons"].append(fr)
                    if _out is not None:
                        _text=getattr(_out,"text","") or ""
                        _ids=list(getattr(_out,"token_ids",[]) or [])
                        _think,_post,_nat=split_think_post_tokens(_ids,_text)
                        st["think_tokens_total"]+=int(_think); st["action_tokens_total"]+=int(_post)
                        if _nat: st["natural_think_closes"]+=1
                elif phase=="phase2":
                    tr=(turn_results or {}).get(tid,{})
                    n1=int(tr.get("phase1_tokens",0) or 0)
                    st["model_turns"]+=1; st["attempt_turns"]+=1
                    st["generated_tokens"]+=n1+nt; st["finish_reasons"].append(fr)
                    st["think_tokens_total"]+=int(tr.get("think_tokens_exact",0) or 0)
                    st["action_tokens_total"]+=int(tr.get("post_close_tokens_exact",0) or 0)+nt
                    if tr.get("natural_think_close"): st["natural_think_closes"]+=1
                    if tr.get("continuation_mode")=="natural_continuation": st["natural_continuations"]+=1
                    if tr.get("continuation_mode")=="forced_close":
                        st["forced_think_closes"]+=1; st["injected_stop_tokens_total"]+=len(stop_ids)
                elif phase=="finalizer":
                    st["model_turns"]+=1; st["attempt_turns"]+=1
                    st["generated_tokens"]+=nt; st["finish_reasons"].append(fr)
                    st["finalizer_output_tokens"]=nt; st["finalizer_finish_reason"]=fr
                st["done"]=True
                stop=f"engine_output_exceeded_max_tokens_{phase}"
                rec=complete_record(st,eval_challenges[tid],None,"failed",stop,err)
                rec["status"]="failed"; append_record(rec)
                save_event(tid,{"run_id":RUN_ID,"arm":arm,"task_id":tid,
                                "parsed_kind":"engine_output_cap_violation","phase":phase,
                                "observed_tokens":nt,"requested_cap":cap,"finish_reason":fr,
                                "error":err,"terminalized":True})
            return bad_tids

        try:
            for tid in tids:
                states[tid] = {
                    "task_id": tid, "arm": arm, "messages": [], "model_turns": 0, "attempt_turns": 0,
                    "tool_calls": 0, "tool_errors": 0, "tool_timeouts": 0, "worker_restarts": 0, "worker_restart_offset": 0,
                    "generated_tokens": 0, "max_prompt_tokens": 0, "finish_reasons": [], "done": False, "final_code": None,
                    "source_retries": 0, "step_retry_count": 0, "step_retries_total": 0, "sampling_retry_nonce": 0,
                    "sampling_seeds": [], "phase2_sampling_seeds": [], "retry_reasons": [], "budget_nudged": False, "budget_denials": 0,
                    "guard_rejections": 0, "salvaged_final_on_length": 0,
                    # v1.8 measurement-contract counters
                    "think_tokens_total": 0, "natural_think_closes": 0, "forced_think_closes": 0,
                    "injected_stop_tokens_total": 0, "action_tokens_total": 0,
                    "first_tool_turn": None, "first_tool_generated_tokens": None,
                    "tool_ok_count": 0, "tool_exception_count": 0, "tool_code_hashes": [], "turn_kinds": [],
                    "post_close_actions": [],
                    "repeated_tool_calls": 0, "defined_symbols": [], "stateful_reuse_events": 0, "stateful_reused_symbols": [],
                    "natural_continuations": 0, "phase2_length_turns": 0, "stop_without_think_close": 0,
                    "infrastructure_failure": False, "infrastructure_errors": [],
                    "generated_tokens_observed_lower_bound": False, "phase2_tokens_unknown": False,
                    "terminal_requested": False, "finalizer_started": False, "finalizer_trigger_reason": None,
                    "finalizer_prompt_tokens": None, "finalizer_output_tokens": None, "finalizer_finish_reason": None,
                    "terminal_parse_status": None, "terminal_render_method": None, "terminal_prompt_sha256": None,
                    "latest_feedback_sha256": None, "feedback_present_in_rendered_prompt": None,
                    "terminal_handoff_reasoning": "", "terminal_handoff_source_tokens": 0,
                    "terminal_handoff_kept_tokens": 0, "terminal_handoff_truncated": False,
                }
                start_fresh_attempt(tid, states[tid])

            while True:
                # v2.4B: reset per-wave phase-2 failure state. It used to be initialised only inside
                # `if p2_tids:`, so a previous wave's set could still be read later in the loop.
                _phase2_failed_tids=set(); _phase2_engine_error=None; _phase2_overcap=set()
                active = [t for t in tids if not states[t]["done"]]
                if not active: break

                if time.perf_counter() >= deadline:
                    for tid in active:
                        st = states[tid]; st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "failed", "global_deadline", "safety reserve reached")
                        rec["status"] = "failed"; append_record(rec)
                    break

                # v2.4 structural terminal state. Once entered, a task can never return to exploration.
                if TERMINAL_POLICY_ENABLED:
                    terminal_tids=[]
                    for tid in active:
                        st=states[tid]
                        _term_reason=_terminal_trigger_reason(st)
                        if _term_reason is not None:
                            st["terminal_requested"]=True
                            if not st.get("finalizer_trigger_reason"):
                                st["finalizer_trigger_reason"]=_term_reason
                            terminal_tids.append(tid); continue

                    if terminal_tids:
                        final_prompts=[]; final_params=[]; final_meta=[]; final_tids=[]
                        for tid in terminal_tids:
                            st=states[tid]
                            term_messages,rendered_term,term_method,_handoff_diag=_terminal_messages_with_reasoning_handoff(st)
                            pt=len(tokenizer.encode(rendered_term,add_special_tokens=False))
                            st["terminal_handoff_kept_tokens"]=int(_handoff_diag.get("kept_tokens") or 0)
                            st["terminal_handoff_source_tokens"]=max(int(st.get("terminal_handoff_source_tokens") or 0),int(_handoff_diag.get("source_tokens") or 0))
                            st["terminal_handoff_truncated"]=bool(st.get("terminal_handoff_truncated",False) or _handoff_diag.get("truncated",False))
                            st["max_prompt_tokens"]=max(st["max_prompt_tokens"],pt)
                            st["finalizer_started"]=True
                            st["finalizer_prompt_tokens"]=pt
                            st["terminal_render_method"]=term_method
                            st["terminal_prompt_sha256"]=hashlib.sha256(rendered_term.encode()).hexdigest()
                            _fd=_feedback_render_diag(st["messages"],rendered_term)
                            st["latest_feedback_sha256"]=_fd["latest_feedback_sha256"]
                            st["feedback_present_in_rendered_prompt"]=_fd["feedback_present_in_rendered_prompt"]
                            remaining_total=TOTAL_GENERATED_TOKEN_CAP-st["generated_tokens"]
                            safe_term=MAX_MODEL_LEN-pt-CONTEXT_SAFETY_TOKENS
                            cap_term=min(FINALIZER_MAX_TOKENS,remaining_total,safe_term)
                            if cap_term<32:
                                st["terminal_parse_status"]="finalizer_admission_failed"
                                st["done"]=True
                                rec=complete_record(st,eval_challenges[tid],None,"rejected","finalizer_admission_failed",
                                                    f"remaining_total={remaining_total}, safe_context={safe_term}, prompt_tokens={pt}")
                                append_record(rec)
                                save_event(tid,{"run_id":RUN_ID,"arm":arm,"task_id":tid,"parsed_kind":"terminal_finalizer",
                                                "finalizer_started":True,"finalizer_trigger_reason":st.get("finalizer_trigger_reason"),
                                                "terminal_parse_status":"finalizer_admission_failed","remaining_total_tokens":remaining_total,
                                                "context_headroom":safe_term,"prompt_tokens":pt,
                                                "feedback_present_in_rendered_prompt":st.get("feedback_present_in_rendered_prompt")})
                                continue
                            fseed=int(PIV_SAMPLING["seed"]+2_000_000+st["sampling_retry_nonce"])
                            final_prompts.append(rendered_term); final_tids.append(tid)
                            final_params.append(_mk_finalizer_params(int(cap_term),fseed))
                            final_meta.append({"cap":int(cap_term),"seed":fseed,"prompt_tokens":pt,"context_headroom":safe_term,
                                               "remaining_total_tokens":remaining_total})

                        if final_tids:
                            batch_wave_no+=1; global_wave_no+=1
                            _ft0=time.perf_counter()
                            try:
                                final_outs=arc_llm.generate(final_prompts,final_params,use_tqdm=False)
                                _final_caps=[m["cap"] for m in final_meta]
                                _assert_engine_output_count("finalizer",final_prompts,final_outs,_final_caps)
                                _final_overcap=_terminalize_overcap_outputs("finalizer",final_tids,final_outs,_final_caps)
                            except BaseException as _engine_exc:
                                _fsec=time.perf_counter()-_ft0
                                _terminalize_engine_exception("finalizer",final_tids,_engine_exc)
                                _terminalize_remaining("engine_unavailable_after_engine_exception",
                                                       f"{type(_engine_exc).__name__}: {_engine_exc}")
                                stop_all=True
                                break
                            _fsec=time.perf_counter()-_ft0
                            _ftoks=[]; _ffinish=[]
                            for tid,req,meta in zip(final_tids,final_outs,final_meta):
                                st=states[tid]
                                if not getattr(req,"outputs",None):
                                    text=""; nt=0; fr="no_output"
                                else:
                                    o=req.outputs[0]; text=o.text or ""; nt=len(getattr(o,"token_ids",[]) or []); fr=str(getattr(o,"finish_reason",None))
                                _ftoks.append(int(nt)); _ffinish.append(fr)
                                if tid in _final_overcap:
                                    continue
                                st["model_turns"]+=1; st["attempt_turns"]+=1
                                st["generated_tokens"]+=int(nt); st["finish_reasons"].append(fr)
                                st["finalizer_output_tokens"]=int(nt); st["finalizer_finish_reason"]=fr
                                parsed_term=parse_terminal_json_output(text)
                                st["terminal_parse_status"]=parsed_term["status"]
                                verifier=None; cand="terminal_parse_failed"; stop="terminal_finalizer_output"
                                if parsed_term["ok"]:
                                    code=parsed_term["solution_code"]; st["final_code"]=code
                                    verifier=fresh_verify_solution(code,eval_challenges[tid],VERIFIER_TIMEOUT_SEC)
                                    cand=verifier_candidate_status(verifier)
                                    stop="final_program"
                                st["done"]=True
                                rec=complete_record(st,eval_challenges[tid],verifier,cand,stop)
                                append_record(rec)
                                save_event(tid,{"run_id":RUN_ID,"arm":arm,"task_id":tid,"turn":st["model_turns"],
                                                "parsed_kind":"terminal_finalizer","finalizer_started":True,
                                                "finalizer_trigger_reason":st.get("finalizer_trigger_reason"),
                                                "terminal_parse_status":parsed_term["status"],"raw_text":text,
                                                "generated_tokens":int(nt),"finish_reason":fr,"request_seed":meta["seed"],
                                                "prompt_tokens":meta["prompt_tokens"],"remaining_total_tokens":meta["remaining_total_tokens"],
                                                "context_headroom":meta["context_headroom"],"cap":meta["cap"],
                                                "latest_feedback_sha256":st.get("latest_feedback_sha256"),
                                                "feedback_present_in_rendered_prompt":st.get("feedback_present_in_rendered_prompt"),
                                                "verifier_status":(verifier or {}).get("verifier_status")})
                            append_wave_metric({"run_id":RUN_ID,"arm":arm,"phase":"terminal_finalizer","batch_no":batch_no,
                                                "batch_wave_no":batch_wave_no,"global_wave_no":global_wave_no,
                                                "task_ids":list(final_tids),"active_count":len(final_tids),
                                                "caps":[m["cap"] for m in final_meta],"request_seeds":[m["seed"] for m in final_meta],
                                                "output_tokens":_ftoks,"finish_reasons":_ffinish,
                                                "overcap_task_ids":sorted(_final_overcap),
                                                "sum_output_tokens":int(sum(_ftoks)),"max_output_tokens":int(max(_ftoks) if _ftoks else 0),
                                                "wave_sec":float(_fsec),
                                                "aggregate_output_tokens_per_sec":float(sum(_ftoks)/_fsec) if _fsec>0 else None,
                                                "sec_per_output_token_longest":float(_fsec)/float(max(_ftoks)) if (_ftoks and max(_ftoks)>0) else None})
                        if stop_all:
                            break
                        continue

                render = []; render_tids = []; max_allowed = []; request_seeds = []; request_diags = {}
                for tid in active:
                    st = states[tid]
                    if st["model_turns"] >= MAX_MODEL_TURNS or st["generated_tokens"] >= TOTAL_GENERATED_TOKEN_CAP:
                        st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "rejected", "agent_budget_exhausted")
                        append_record(rec); continue

                    prompt, method = render_agent_chat(st["messages"])
                    pt = len(tokenizer.encode(prompt, add_special_tokens=False))
                    st["max_prompt_tokens"] = max(st["max_prompt_tokens"], pt)
                    safe = MAX_MODEL_LEN - pt - CONTEXT_SAFETY_TOKENS
                    remaining = TOTAL_GENERATED_TOKEN_CAP - st["generated_tokens"]
                    if TERMINAL_POLICY_ENABLED:
                        cap=exploration_cap_v24(generated_tokens=st["generated_tokens"],prompt_tokens=pt,forced_policy=forced_policy,phase2=False)
                    elif forced_policy:
                        # v1.9: keep ACTION_RESERVE_TOKENS of the total budget out of thinking so every turn (incl. the last)
                        # retains room for a phase-2 action; otherwise the final-answer turn dies with 0 action tokens.
                        cap = min(think_budget, remaining - int(ACTION_RESERVE_TOKENS), safe)
                    else:
                        cap = min(PER_TURN_MAX_TOKENS, remaining, safe)
                    if cap < 32:
                        if TERMINAL_POLICY_ENABLED:
                            st["terminal_requested"]=True
                            st["finalizer_trigger_reason"]="exploration_admission_reserve"
                            continue
                        st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "rejected", "context_or_token_budget_exhausted")
                        append_record(rec); continue

                    req_seed = int(PIV_SAMPLING["seed"] + st["sampling_retry_nonce"])
                    render.append(prompt); render_tids.append(tid); max_allowed.append(int(cap)); request_seeds.append(req_seed)
                    st["sampling_seeds"].append(req_seed)
                    _fd=_feedback_render_diag(st["messages"],prompt)
                    request_diags[tid]={"rendered_prompt_sha256":hashlib.sha256(prompt.encode()).hexdigest(),
                                        "latest_feedback_sha256":_fd["latest_feedback_sha256"],
                                        "feedback_present_in_rendered_prompt":_fd["feedback_present_in_rendered_prompt"],
                                        "remaining_total_tokens":remaining,
                                        "remaining_exploration_tokens":max(0,EXPLORATION_TOKEN_CAP-st["generated_tokens"]) if TERMINAL_POLICY_ENABLED else None,
                                        "context_headroom":safe,"request_seed":req_seed,
                                        "prompt_tokens":pt}

                if not render_tids: continue

                # v1.6: a wave runs for up to max(caps) tokens (+ the action phase for the forced-close arm). Refuse to
                # start one that would cross the deadline; affected tasks are recorded as failed (resumable).
                if wave_spt_samples:
                    _spt = max(float(DEFAULT_SEC_PER_OUTPUT_TOKEN), float(np.percentile(wave_spt_samples, 90)))
                else:
                    _spt = float(DEFAULT_SEC_PER_OUTPUT_TOKEN)
                _wave_tokens = float(max(max_allowed)) + (float(ACTION_MAX_TOKENS) if forced_policy else 0.0)
                _projected_wave_sec = _wave_tokens * _spt * float(WAVE_TIME_SAFETY_FACTOR) + float(WAVE_OVERHEAD_SEC)
                if time.perf_counter() + _projected_wave_sec >= deadline:
                    for tid in render_tids:
                        st = states[tid]; st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "failed", "global_deadline_projected",
                                              f"projected wave {_projected_wave_sec:.0f}s at {_spt:.3f}s/token would cross the safety deadline")
                        rec["status"] = "failed"; append_record(rec)
                    _terminalize_remaining("global_deadline_projected",
                                           f"projected wave {_projected_wave_sec:.0f}s at {_spt:.3f}s/token would cross the safety deadline")
                    stop_all = True
                    print(f"batch {batch_no}: projected wave {_projected_wave_sec/60:.1f} min would cross the deadline; stopping cleanly.")
                    break

                params = [_mk_params(c, s) for c, s in zip(max_allowed, request_seeds)]

                # ---------------- phase 1: thinking (natural arms: the whole turn) ----------------
                batch_wave_no += 1; global_wave_no += 1
                t0 = time.perf_counter()
                try:
                    outs = arc_llm.generate(render, params, use_tqdm=False)
                    _assert_engine_output_count("phase1", render, outs, max_allowed)
                    _phase1_overcap=_terminalize_overcap_outputs("phase1",render_tids,outs,max_allowed)
                except BaseException as _engine_exc:
                    wave_sec = time.perf_counter() - t0
                    _terminalize_engine_exception("phase1", render_tids, _engine_exc)
                    _terminalize_remaining("engine_unavailable_after_engine_exception",
                                           f"{type(_engine_exc).__name__}: {_engine_exc}")
                    stop_all = True
                    break
                wave_sec = time.perf_counter() - t0
                _wave_output_tokens = []; _wave_finish = []
                turn_results = {}
                for tid, req in zip(render_tids, outs):
                    if getattr(req, "outputs", None):
                        o = req.outputs[0]
                        t1 = o.text or ""; n1 = len(getattr(o, "token_ids", []) or []); f1 = str(getattr(o, "finish_reason", None))
                        _wave_output_tokens.append(n1); _wave_finish.append(f1)
                        if tid in _phase1_overcap:
                            continue
                        _ids1 = list(getattr(o, "token_ids", []) or [])
                        _think1, _post1, _nat1 = split_think_post_tokens(_ids1, t1)
                        _pids = list(getattr(req, "prompt_token_ids", None) or [])
                        if forced_policy and not _pids:
                            raise RuntimeError("vLLM RequestOutput.prompt_token_ids missing; token-exact continuation is mandatory for the forced_close arm")
                        turn_results[tid] = {
                            "no_output": False, "text": t1, "ntok": n1, "fr": f1,
                            "phase1_tokens": n1, "phase1_finish": f1,
                            "natural_think_close": bool(_nat1), "forced_think_close": False, "continuation_mode": None,
                            "think_tokens_exact": int(_think1), "post_close_tokens_exact": int(_post1),
                            "injected_stop_tokens": 0, "phase2_tokens": 0, "phase2_finish": None, "phase2_skipped_reason": None,
                            "prompt_token_ids": _pids, "output_token_ids": _ids1,
                        }
                    else:
                        _wave_output_tokens.append(0); _wave_finish.append("no_output")
                        turn_results[tid] = {"no_output": True}
                append_wave_metric({
                    "run_id": RUN_ID, "arm": arm, "phase": 1, "batch_no": batch_no, "batch_wave_no": batch_wave_no, "global_wave_no": global_wave_no,
                    "task_ids": list(render_tids), "active_count": len(render_tids), "caps": list(max_allowed),
                    "request_seeds": list(request_seeds), "output_tokens": _wave_output_tokens,
                    "finish_reasons": _wave_finish, "overcap_task_ids":sorted(_phase1_overcap),
                    "sum_output_tokens": int(sum(_wave_output_tokens)),
                    "max_output_tokens": int(max(_wave_output_tokens) if _wave_output_tokens else 0),
                    "wave_sec": float(wave_sec),
                    "aggregate_output_tokens_per_sec": float(sum(_wave_output_tokens) / wave_sec) if wave_sec > 0 else None,
                    "sec_per_output_token_longest": (float(wave_sec) / float(max(_wave_output_tokens))) if (_wave_output_tokens and max(_wave_output_tokens) > 0) else None,
                })
                if _wave_output_tokens and max(_wave_output_tokens) > 0 and wave_sec > 0:
                    wave_spt_samples.append(float(wave_sec) / float(max(_wave_output_tokens)))

                # ---------------- phase 2: forced close + action (forced_close arm only) ----------------
                if forced_policy:
                    p2_prompts = []; p2_tids = []; p2_params = []; p2_caps = []; p2_seeds = []
                    for tid in render_tids:
                        if states[tid].get("done"):
                            continue
                        tr = turn_results[tid]
                        if tr.get("no_output"): continue
                        if str(tr["phase1_finish"]).lower() == "stop": continue          # complete logical turn
                        # v1.9 (Qwen documented flow): length without </think> -> inject stop text; length WITH </think>
                        # (the action itself was cut at the think cap) -> continue the exact prefix, no injection.
                        tr["continuation_mode"] = "natural_continuation" if tr["natural_think_close"] else "forced_close"
                        st = states[tid]
                        _inject = list(stop_ids) if tr["continuation_mode"] == "forced_close" else []
                        remaining2 = TOTAL_GENERATED_TOKEN_CAP - st["generated_tokens"] - tr["phase1_tokens"]
                        pt1 = len(tr["prompt_token_ids"])
                        safe2 = MAX_MODEL_LEN - (pt1 + tr["phase1_tokens"] + len(_inject)) - CONTEXT_SAFETY_TOKENS
                        if TERMINAL_POLICY_ENABLED:
                            generated_after_p1=st["generated_tokens"]+tr["phase1_tokens"]
                            remaining_explore2=max(0,EXPLORATION_TOKEN_CAP-generated_after_p1)
                            safe_explore2=max(0,safe2-FINALIZER_CONTEXT_RESERVE_TOKENS)
                            cap2=min(int(ACTION_MAX_TOKENS),int(remaining_explore2),int(safe_explore2))
                        else:
                            cap2 = min(int(ACTION_MAX_TOKENS), int(remaining2), int(safe2))
                        if cap2 < 32:
                            tr["phase2_skipped_reason"] = "terminal_reserve_or_context" if TERMINAL_POLICY_ENABLED else "no_budget_or_context_for_action"
                            continue
                        # token-exact continuation of the same request (Qwen docs two-stage technique); no text fallback
                        p2_prompts.append({"prompt_token_ids": list(tr["prompt_token_ids"]) + list(tr["output_token_ids"]) + _inject})
                        p2_tids.append(tid); p2_caps.append(cap2)
                        # v1.9/v2.1: phase 2 is a fresh vLLM request with an explicitly logged phase-specific seed.
                        _seed2 = int(request_seeds[render_tids.index(tid)] + 1_000_000)
                        p2_seeds.append(_seed2); states[tid].setdefault("phase2_sampling_seeds",[]).append(_seed2)
                        p2_params.append(_mk_params(cap2, _seed2))
                    if p2_tids:
                        batch_wave_no += 1; global_wave_no += 1
                        t0 = time.perf_counter()
                        _phase2_failed_tids=set(); _phase2_engine_error=None
                        try:
                            outs2 = arc_llm.generate(p2_prompts, p2_params, use_tqdm=False)
                            _assert_engine_output_count("phase2", p2_prompts, outs2, p2_caps)
                            _phase2_overcap=_terminalize_overcap_outputs("phase2",p2_tids,outs2,p2_caps,turn_results=turn_results)
                        except BaseException as _engine_exc:
                            wave2_sec=time.perf_counter()-t0; _phase2_engine_error=f"{type(_engine_exc).__name__}: {_engine_exc}"; _phase2_failed_tids=set(p2_tids)
                            for _tid in p2_tids:
                                st=states[_tid]; tr=turn_results[_tid]
                                st["model_turns"]+=1; st["attempt_turns"]+=1
                                _n1=int(tr.get("phase1_tokens",0) or 0); _f1=str(tr.get("phase1_finish"))
                                st["generated_tokens"]+=_n1; st["finish_reasons"].append(_f1)
                                st["think_tokens_total"]+=int(tr.get("think_tokens_exact",0) or 0); st["action_tokens_total"]+=int(tr.get("post_close_tokens_exact",0) or 0)
                                if tr.get("natural_think_close"): st["natural_think_closes"]+=1
                                if tr.get("continuation_mode")=="natural_continuation": st["natural_continuations"]+=1
                                if tr.get("continuation_mode")=="forced_close": st["forced_think_closes"]+=1; st["injected_stop_tokens_total"]+=len(stop_ids)
                                st["infrastructure_failure"]=True
                                st.setdefault("infrastructure_errors",[]).append({"phase":"phase2","error":_phase2_engine_error})
                                st["phase2_tokens_unknown"]=True
                                st["generated_tokens_observed_lower_bound"]=True
                                _partial=parse_agent_turn(tr.get("text","") or ""); st["turn_kinds"].append(_partial.get("kind","text"))
                                _event={"run_id":RUN_ID,"arm":arm,"task_id":_tid,"turn":st["model_turns"],"attempt_turn":st["attempt_turns"],
                                        "parsed_kind":"engine_exception_phase2","engine_error":_phase2_engine_error,"generated_tokens":_n1,
                                        "phase1_tokens":_n1,"phase1_finish":_f1,"raw_text":tr.get("text","") or "",
                                        "natural_think_close":bool(tr.get("natural_think_close")),"continuation_mode":tr.get("continuation_mode"),
                                        "phase2_tokens":None,"phase2_tokens_observed":False,"phase2_tokens_unknown":True,
                                        "phase2_request_seed":(st.get("phase2_sampling_seeds") or [None])[-1],"phase2_failed":True}
                                # v2.3: use the SAME safe length-salvage contract as the normal path.
                                # Draft code inside unclosed <think> is never treated as final merely because phase 2 failed.
                                _salvaged=salvage_final_code_on_length(tr.get("text","") or "", _partial) if str(_f1).lower()=="length" else None
                                if _salvaged is not None:
                                    st["salvaged_final_on_length"]+=1
                                    st["final_code"]=_salvaged; _ver=fresh_verify_solution(_salvaged,eval_challenges[_tid],VERIFIER_TIMEOUT_SEC)
                                    _cand=verifier_candidate_status(_ver)
                                    st["done"]=True
                                    _rec=complete_record(st,eval_challenges[_tid],_ver,_cand,
                                                         "final_program_safe_phase1_salvage_after_phase2_exception",_phase2_engine_error)
                                else:
                                    st["done"]=True
                                    _rec=complete_record(st,eval_challenges[_tid],None,"failed","engine_generate_exception_phase2",_phase2_engine_error)
                                    _rec["status"]="failed"
                                append_record(_rec); save_event(_tid,_event)
                            append_wave_metric({"run_id":RUN_ID,"arm":arm,"phase":2,"batch_no":batch_no,"batch_wave_no":batch_wave_no,
                                                "global_wave_no":global_wave_no,"task_ids":list(p2_tids),"active_count":len(p2_tids),"caps":list(p2_caps),
                                                "request_seeds":list(p2_seeds),"output_tokens":[None]*len(p2_tids),"finish_reasons":["engine_exception"]*len(p2_tids),
                                                "sum_output_tokens":None,"max_output_tokens":None,"tokens_observed":False,"tokens_unknown":True,
                                                "wave_sec":float(wave2_sec),"engine_exception":_phase2_engine_error})
                            (ARM_DIR/"engine_error_phase2.json").write_text(json.dumps({"run_id":RUN_ID,"arm":arm,"phase":"phase2","task_ids":list(p2_tids),
                                                                                       "error":_phase2_engine_error,"traceback":traceback.format_exc(limit=20)},indent=2),encoding="utf-8")
                            print(f"[{arm}] ENGINE GENERATE EXCEPTION (phase2): {_phase2_engine_error}; preserved phase-1 evidence for {len(p2_tids)} continuation tasks")
                            stop_all=True; outs2=[]
                        wave2_sec = time.perf_counter()-t0 if not _phase2_engine_error else wave2_sec
                        _w2_tokens=[]; _w2_finish=[]
                        if not _phase2_engine_error:
                            for tid, req2 in zip(p2_tids, outs2):
                                tr=turn_results[tid]
                                if not getattr(req2,"outputs",None):
                                    _w2_tokens.append(0); _w2_finish.append("no_output")
                                    if tid not in _phase2_overcap: tr["phase2_skipped_reason"]="no_output"
                                    continue
                                o2=req2.outputs[0]; t2=o2.text or ""; n2=len(getattr(o2,"token_ids",[]) or []); f2=str(getattr(o2,"finish_reason",None))
                                _w2_tokens.append(n2); _w2_finish.append(f2)
                                if tid in _phase2_overcap:
                                    continue
                                _forced=(tr["continuation_mode"]=="forced_close")
                                tr.update({"text":tr["text"]+(stop_text if _forced else "")+t2,"ntok":tr["phase1_tokens"]+n2,"fr":f2,
                                           "forced_think_close":_forced,"injected_stop_tokens":(len(stop_ids) if _forced else 0),
                                           "phase2_tokens":n2,"phase2_finish":f2,"post_close_tokens_exact":(n2 if _forced else tr["post_close_tokens_exact"]+n2)})
                            append_wave_metric({"run_id":RUN_ID,"arm":arm,"phase":2,"batch_no":batch_no,"batch_wave_no":batch_wave_no,"global_wave_no":global_wave_no,
                                                "task_ids":list(p2_tids),"active_count":len(p2_tids),"caps":list(p2_caps),"request_seeds":list(p2_seeds),
                                                "output_tokens":_w2_tokens,"finish_reasons":_w2_finish,"overcap_task_ids":sorted(_phase2_overcap),
                                                "sum_output_tokens":int(sum(_w2_tokens)),
                                                "max_output_tokens":int(max(_w2_tokens) if _w2_tokens else 0),"injected_stop_tokens_per_request":len(stop_ids),
                                                "continuation_modes":[turn_results[t]["continuation_mode"] for t in p2_tids],"wave_sec":float(wave2_sec),
                                                "aggregate_output_tokens_per_sec":float(sum(_w2_tokens)/wave2_sec) if wave2_sec>0 else None,
                                                "sec_per_output_token_longest":(float(wave2_sec)/float(max(_w2_tokens))) if (_w2_tokens and max(_w2_tokens)>0) else None})
                        wave_sec += wave2_sec

                # ---------------- per-task turn handling (v1.7 logic + contract counters) ----------------
                for tid in render_tids:
                    if states[tid].get("done"):
                        continue
                    if forced_policy and "_phase2_failed_tids" in locals() and tid in _phase2_failed_tids:
                        continue
                    st = states[tid]; st["model_turns"] += 1; st["attempt_turns"] += 1
                    tr = turn_results[tid]

                    if tr.get("no_output"):
                        event = {"run_id": RUN_ID, "arm": arm, "task_id": tid, "turn": st["model_turns"], "attempt_turn": st["attempt_turns"],
                                 "parsed_kind": "no_output", "wave_sec": wave_sec}
                        st["turn_kinds"].append("no_output")
                        if register_step_retry(st, "no_output"):
                            event["step_retry"] = True; save_event(tid, event); continue
                        if start_fresh_attempt(tid, st, "no_output"):
                            event["source_retry"] = True; save_event(tid, event); continue
                        st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "failed", "no_output", "vLLM returned no output")
                        rec["status"] = "failed"; append_record(rec); save_event(tid, event); continue

                    text = tr["text"]; nt = tr["ntok"]; fr = tr["fr"]
                    st["generated_tokens"] += nt
                    st["finish_reasons"].append(fr)
                    parsed = parse_agent_turn(text)
                    if parsed.get("reasoning_content"):
                        _rh,_rdiag=_bounded_reasoning_handoff(parsed.get("reasoning_content"))
                        st["terminal_handoff_reasoning"]=_rh
                        st["terminal_handoff_source_tokens"]=int(_rdiag.get("source_tokens") or 0)
                        st["terminal_handoff_kept_tokens"]=int(_rdiag.get("kept_tokens") or 0)
                        st["terminal_handoff_truncated"]=bool(_rdiag.get("truncated",False))
                    # v1.9: model-generated thinking tokens only (exact id split); injected stop text is never counted here
                    think_tokens = int(tr["think_tokens_exact"])
                    st["think_tokens_total"] += think_tokens
                    st["action_tokens_total"] += int(tr["post_close_tokens_exact"])
                    if tr["natural_think_close"]: st["natural_think_closes"] += 1
                    if str(tr["phase1_finish"]).lower() == "stop" and not tr["natural_think_close"]: st["stop_without_think_close"] += 1
                    if tr["continuation_mode"] == "natural_continuation": st["natural_continuations"] += 1
                    if str(tr["phase2_finish"]).lower() == "length": st["phase2_length_turns"] += 1
                    if tr["forced_think_close"]:
                        st["forced_think_closes"] += 1; st["injected_stop_tokens_total"] += tr["injected_stop_tokens"]
                        st["post_close_actions"].append(parsed["kind"])
                    st["turn_kinds"].append(parsed["kind"])
                    event = {"run_id": RUN_ID, "arm": arm, "task_id": tid, "turn": st["model_turns"], "attempt_turn": st["attempt_turns"],
                             "source_retry_index": st["source_retries"],
                             # v2.4B: this used to log the running MAX, not this request's prompt size.
                             "prompt_tokens": request_diags.get(tid,{}).get("prompt_tokens"),
                             "max_prompt_tokens_so_far": st["max_prompt_tokens"],
                             "generated_tokens": nt, "finish_reason": fr, "raw_text": text, "parsed_kind": parsed["kind"],
                             "wave_sec": wave_sec, "request_seed": st["sampling_seeds"][-1],
                             "phase2_request_seed": (st.get("phase2_sampling_seeds") or [None])[-1] if tr.get("phase2_tokens",0) else None,
                             "phase1_tokens": tr["phase1_tokens"], "phase1_finish": tr["phase1_finish"],
                             "natural_think_close": tr["natural_think_close"], "forced_think_close": tr["forced_think_close"],
                             "injected_stop_tokens": tr["injected_stop_tokens"], "phase2_tokens": tr["phase2_tokens"],
                             "phase2_finish": tr["phase2_finish"], "phase2_skipped_reason": tr["phase2_skipped_reason"],
                             "think_tokens": think_tokens, "post_close_tokens": int(tr["post_close_tokens_exact"]),
                             "continuation_mode": tr["continuation_mode"], "action_cap_hit": (str(tr["phase2_finish"]).lower() == "length"),
                             "reasoning_mode": parsed.get("reasoning_mode"),
                             "rendered_prompt_sha256":request_diags.get(tid,{}).get("rendered_prompt_sha256"),
                             "latest_feedback_sha256":request_diags.get(tid,{}).get("latest_feedback_sha256"),
                             "feedback_present_in_rendered_prompt":request_diags.get(tid,{}).get("feedback_present_in_rendered_prompt"),
                             "remaining_total_tokens_before_request":request_diags.get(tid,{}).get("remaining_total_tokens"),
                             "remaining_exploration_tokens_before_request":request_diags.get(tid,{}).get("remaining_exploration_tokens"),
                             "context_headroom_before_request":request_diags.get(tid,{}).get("context_headroom"),
                             "post_close_action": (parsed["kind"] if tr["forced_think_close"] else None)}

                    _salvaged = salvage_final_code_on_length(text, parsed) if str(fr).lower() == "length" else None
                    if _salvaged is not None:
                        st["salvaged_final_on_length"] += 1; event["salvaged_final_on_length"] = True
                    if not generation_finish_reason_ok(fr) and _salvaged is None:
                        # v2.4F: do not count a retry that can never execute because this just-completed
                        # turn already crossed an immediate terminal threshold.
                        _post_turn_terminal=_terminal_trigger_reason(st)
                        if _post_turn_terminal is not None:
                            st["terminal_requested"]=True
                            if not st.get("finalizer_trigger_reason"):
                                st["finalizer_trigger_reason"]=_post_turn_terminal
                            event["terminal_requested_after_turn"]=True; save_event(tid,event); continue
                        if register_step_retry(st, f"invalid_finish_reason:{fr}"):
                            event["step_retry"] = True; save_event(tid, event); continue
                        if start_fresh_attempt(tid, st, f"invalid_finish_reason:{fr}"):
                            event["source_retry"] = True; save_event(tid, event); continue
                        if _request_terminal_after_exhaustion(st, f"invalid_finish_retries_exhausted:{fr}"):
                            event["terminal_requested_after_retry_exhaustion"]=True; save_event(tid,event); continue
                        st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "rejected", "invalid_finish_reason",
                                              f"finish_reason={fr}; retries exhausted or terminal policy disabled")
                        append_record(rec); save_event(tid, event); continue

                    if parsed["kind"] == "malformed_tool":
                        if register_step_retry(st, "malformed_tool_xml"):
                            event["step_retry"] = True; save_event(tid, event); continue
                        if start_fresh_attempt(tid, st, "malformed_tool_xml"):
                            event["source_retry"] = True; save_event(tid, event); continue
                        st["tool_errors"] += 1
                        if _request_terminal_after_exhaustion(st, "malformed_tool_retries_exhausted"):
                            event["terminal_requested_after_retry_exhaustion"]=True; save_event(tid,event); continue
                        st["done"] = True
                        rec = complete_record(st, eval_challenges[tid], None, "rejected", "malformed_tool_xml")
                        append_record(rec); save_event(tid, event); continue

                    if parsed["kind"] == "tool":
                        # This reasoning/tool turn is now valid conversation history; any scratch handoff from
                        # an abandoned earlier attempt is superseded by the accepted assistant message below.
                        st["terminal_handoff_reasoning"]=""
                        st["terminal_handoff_source_tokens"]=0
                        st["terminal_handoff_kept_tokens"]=0
                        st["terminal_handoff_truncated"]=False
                        calls = parsed.get("tool_calls", [])
                        # Source first retries malformed/unsupported calls at the SAME tool-loop step.
                        malformed = any(tc.get("name") != "python" or "code" not in tc.get("arguments", {}) for tc in calls)
                        if malformed:
                            if register_step_retry(st, "malformed_or_unsupported_structured_tool"):
                                event["step_retry"] = True; save_event(tid, event); continue
                            st["step_retry_count"] = 0
                            st["messages"].append(assistant_tool_message(parsed))
                            for tc in calls:
                                st["messages"].append({"role": "tool", "content":
                                    f"Tool call rejected: expected python(code=...). Got name={tc.get('name')!r}, keys={list(tc.get('arguments', {}))}."})
                            st["tool_errors"] += 1
                            event["tool_result"] = "unsupported_or_invalid_structured_tool_call"
                            save_event(tid, event); continue

                        st["step_retry_count"] = 0   # v1.6: reset only once a VALID structured tool turn is accepted
                        st["messages"].append(assistant_tool_message(parsed))
                        event_calls = []
                        denial = ("LOCAL P0 tool budget exhausted. No further Python execution is available. "
                                  "Return the self-contained final `solution(grid)` Python code block now.")
                        for call_idx, tc in enumerate(calls):
                            if st["tool_calls"] >= MAX_TOOL_CALLS:
                                st["messages"].append({"role": "tool", "content": denial})
                                st["budget_denials"] += 1
                                event_calls.append({"tool_code": tc.get("arguments", {}).get("code"),
                                                    "tool_ok": False, "tool_output_to_model": denial,
                                                    "budget_denied": True})
                                continue
                            code = tc["arguments"]["code"]
                            rr = workers[tid].execute(code)
                            if rr.get("infrastructure_error"):
                                st["infrastructure_failure"]=True
                                st.setdefault("infrastructure_errors",[]).append({"phase":"repl_execute","error":str(rr.get("text") or "REPL infrastructure error"),"restarted":bool(rr.get("restarted",False))})
                            if rr.get("guard_rejected"):
                                # v1.6: a locally rejected snippet never reached Python; do not charge the 6-call budget.
                                st["guard_rejections"] += 1
                            else:
                                st["tool_calls"] += 1
                                if st["first_tool_turn"] is None:
                                    st["first_tool_turn"] = st["model_turns"]; st["first_tool_generated_tokens"] = st["generated_tokens"]
                                if rr.get("ok"): st["tool_ok_count"] += 1
                                elif not rr.get("timeout") and not rr.get("infrastructure_error"): st["tool_exception_count"] += 1
                            _ch = hashlib.sha256(code.strip().encode()).hexdigest()[:16]
                            if _ch in st["tool_code_hashes"]: st["repeated_tool_calls"] += 1
                            else: st["tool_code_hashes"].append(_ch)
                            _reuse = code_state_effects(st, code) if not rr.get("guard_rejected") else {"reuse": [], "defined": []}
                            # worker.restart_count is cumulative within the current source-attempt worker.
                            current_worker_restarts = int(getattr(workers[tid], "restart_count", 0))
                            st["worker_restarts"] = st["worker_restart_offset"] + current_worker_restarts
                            if not rr.get("ok"): st["tool_errors"] += 1
                            if rr.get("timeout"): st["tool_timeouts"] += 1
                            raw_tool = str(rr.get("text", ""))[:MAX_TOOL_RAW_ARTIFACT_CHARS]
                            _terminal_suffix=""
                            if st["tool_calls"] >= MAX_TOOL_CALLS:
                                _terminal_suffix=("\n\n[LOCAL P0 BUDGET] This was the final permitted Python execution. "
                                                  "The scheduler will now enter terminal commit; no further Python execution is available.")
                                if TERMINAL_POLICY_ENABLED:
                                    st["terminal_requested"]=True
                                    st["finalizer_trigger_reason"]="tool_budget_exhausted"
                            if TERMINAL_POLICY_ENABLED:
                                model_tool,_feedback_fit=fit_tool_feedback_to_terminal_reserve(st["messages"],raw_tool,_terminal_suffix)
                            else:
                                model_tool=sanitize_tool_output(compact_tool_output(raw_tool))+_terminal_suffix
                                _feedback_fit={"fits":True,"truncated":False,"terminal_prompt_tokens":None,
                                               "max_terminal_prompt_tokens":None,"raw_feedback_tokens":None,"kept_feedback_tokens":None}
                            if not _feedback_fit.get("fits"):
                                st["infrastructure_failure"]=True
                                st.setdefault("infrastructure_errors",[]).append("terminal_context_reserve_breach_after_tool_feedback")
                                st["done"]=True
                                rec=complete_record(st,eval_challenges[tid],None,"failed","terminal_context_reserve_breach",
                                                    json.dumps(_feedback_fit,sort_keys=True))
                                rec["status"]="failed"; append_record(rec)
                                event_calls.append({"tool_code":code,"tool_code_sha16":_ch,"tool_ok":rr.get("ok"),
                                                    "stateful_reuse":_reuse["reuse"],"guard_rejected":bool(rr.get("guard_rejected")),
                                                    "timeout":bool(rr.get("timeout")),"tool_raw_output":raw_tool,
                                                    "tool_output_to_model":model_tool,"terminal_feedback_fit":_feedback_fit,
                                                    "current_worker_restarts":current_worker_restarts})
                                break
                            st["messages"].append({"role": "tool", "content": model_tool})
                            event_calls.append({"tool_code": code, "tool_code_sha16": _ch, "tool_ok": rr.get("ok"), "stateful_reuse": _reuse["reuse"],
                                                "guard_rejected": bool(rr.get("guard_rejected")), "timeout": bool(rr.get("timeout")),
                                                "tool_raw_output": raw_tool, "tool_output_to_model": model_tool,
                                                "terminal_feedback_fit":_feedback_fit,
                                                "current_worker_restarts": current_worker_restarts})
                        event["tool_calls_executed"] = event_calls
                        if st.get("done") and st.get("infrastructure_failure") and "terminal_context_reserve_breach_after_tool_feedback" in st.get("infrastructure_errors",[]):
                            event["tool_result"]="terminal_context_reserve_breach"
                            save_event(tid,event); continue
                        _denied_this_turn = sum(1 for x in event_calls if x.get("budget_denied"))
                        if _denied_this_turn:
                            event["tool_result"] = "partial_tool_budget_denial_finalization_nudge"
                            if st["budget_nudged"] and st["tool_calls"] >= MAX_TOOL_CALLS:
                                st["done"] = True
                                rec = complete_record(st, eval_challenges[tid], None, "rejected", "tool_budget_exhausted_after_nudge")
                                append_record(rec)
                            else:
                                st["budget_nudged"] = True
                        save_event(tid, event); continue

                    if parsed["kind"] == "final":
                        st["step_retry_count"] = 0
                        code = parsed["final_code"]; st["final_code"] = code
                        verifier = fresh_verify_solution(code, eval_challenges[tid], VERIFIER_TIMEOUT_SEC)
                        cand = verifier_candidate_status(verifier)
                        st["done"] = True; event["verifier_status"] = verifier.get("verifier_status")
                        rec = complete_record(st, eval_challenges[tid], verifier, cand, "final_program")
                        append_record(rec); save_event(tid, event); continue

                    # No tool and no valid final code: source CodeSolutionTurn rejects and AbstractTurn retries.
                    if start_fresh_attempt(tid, st, "no_tool_call_or_final_solution"):
                        event["source_retry"] = True; save_event(tid, event); continue
                    if _request_terminal_after_exhaustion(st, "no_action_retries_exhausted"):
                        event["terminal_requested_after_retry_exhaustion"]=True; save_event(tid,event); continue
                    st["done"] = True
                    rec = complete_record(st, eval_challenges[tid], None, "rejected", "no_tool_call_or_final_solution")
                    append_record(rec); save_event(tid, event)

                print(f"[{arm}] batch {batch_no}: active={sum(not states[t]['done'] for t in tids)} | "
                      f"wave={wave_sec:.1f}s | caps={min(max_allowed)}..{max(max_allowed)}")
                if stop_all:
                    for _tid in tids:
                        _st=states[_tid]
                        if not _st.get("done"):
                            _st["done"]=True
                            _rec=complete_record(_st,eval_challenges[_tid],None,"failed","engine_unavailable_after_phase2_exception",
                                                 _phase2_engine_error or "engine unavailable after phase2 exception")
                            _rec["status"]="failed"; append_record(_rec)
                    break
        finally:
            for tid, w in workers.items():
                try: w.close()
                except Exception: pass

    loop_elapsed_sec = time.perf_counter() - loop_t0
    # v2.4B: the adaptive controller runs each arm TWICE (pilot stage, then the remaining SMOKE tasks).
    # Opening this file with "w" threw away the pilot stage's loop time while Cell 9 still multiplied by
    # 48/8, making the Proxy-48 wall-clock projection optimistic. Accumulate across stages instead.
    _prev_runtime={}
    if AGENT_LOOP_RUNTIME_PATH.exists():
        try: _prev_runtime=json.loads(AGENT_LOOP_RUNTIME_PATH.read_text(encoding="utf-8")) or {}
        except Exception: _prev_runtime={}
    _prev_loop=float(_prev_runtime.get("loop_elapsed_sec",0.0) or 0.0)
    _prev_executed=int(_prev_runtime.get("tasks_executed",0) or 0)
    _prev_waves=int(_prev_runtime.get("global_waves",0) or 0)
    _stages=list(_prev_runtime.get("stages",[]) or [])
    _stages.append({"run_id":RUN_ID,"tasks_executed":len(pending),
                    "loop_elapsed_sec":float(loop_elapsed_sec),"global_waves":int(global_wave_no)})
    _cumulative_loop=float(_prev_loop+loop_elapsed_sec)
    _cumulative_tasks=int(_prev_executed+len(pending))
    with open(AGENT_LOOP_RUNTIME_PATH, "w") as _f:
        json.dump({
            "run_id": RUN_ID, "arm": arm,
            "pre_loop_elapsed_sec": float(pre_loop_elapsed_sec),
            "loop_elapsed_sec": _cumulative_loop,
            "loop_elapsed_sec_this_stage": float(loop_elapsed_sec),
            "tasks_executed": _cumulative_tasks,
            "stages": _stages,
            "agent_batch_size": AGENT_BATCH_SIZE,
            "active_tasks": len(ACTIVE_TASK_IDS),
            "global_waves": int(_prev_waves+global_wave_no),
            "global_waves_this_stage": int(global_wave_no),
        }, _f, indent=2)
    print(f"[{arm}] Agent loop runtime:", json.dumps({
        "pre_loop_elapsed_sec": round(pre_loop_elapsed_sec, 2),
        "loop_elapsed_sec_cumulative": round(_cumulative_loop, 2),
        "loop_elapsed_sec_this_stage": round(loop_elapsed_sec, 2),
        "tasks_executed_cumulative": _cumulative_tasks,
        "global_waves": global_wave_no,
    }))

QUICK_BUDGET_DEATH_REASONS={"invalid_finish_reason","agent_budget_exhausted","context_or_token_budget_exhausted","tool_budget_exhausted_after_nudge",
                            "finalizer_admission_failed"}
QUICK_TERMINAL_ARTIFACT_FAILURE_REASONS={"terminal_finalizer_output","no_tool_call_or_final_solution","malformed_tool_xml"}
QUICK_INFRA_REASONS={"no_output","global_deadline","global_deadline_projected"}

def arm_quick_summary(arm, print_sample=True):
    """Instant model-behaviour readout printed the moment an arm finishes (PROBE canary; early signal in SMOKE).
    Reads only this arm's records + traces. No scoring/gate here — cells 9-11 do the pre-registered analysis."""
    _raw=[r for r in read_existing_records() if r.get("arm")==arm and str(r.get("status")) in {"complete","failed"}]
    # v2.4B: a task re-run in a later adaptive stage appends a second terminal record. Score the LATEST
    # one per task, exactly like score_arm(), otherwise every rate below is diluted.
    _latest={}
    for r in _raw: _latest[str(r.get("task_id"))]=r
    recs=[_latest[t] for t in ACTIVE_TASK_IDS if t in _latest]
    n=len(recs)
    def _rate(f): return round(sum(1 for r in recs if f(r))/n,3) if n else 0.0
    turns=[]
    for tid in ACTIVE_TASK_IDS:
        p=TRACE_DIR/f"{tid}.jsonl"
        if not p.exists(): continue
        for line in p.read_text(encoding="utf-8").splitlines():
            try: e=json.loads(line)
            except Exception: continue
            if "generated_tokens" in e: turns.append(e)
    first=[e for e in turns if int(e.get("turn") or 0)==1]
    think=[int(e.get("think_tokens") or 0) for e in turns]
    kinds={}
    for e in turns: kinds[str(e.get("parsed_kind"))]=kinds.get(str(e.get("parsed_kind")),0)+1
    stops={}
    for r in recs: stops[str(r.get("stop_reason"))]=stops.get(str(r.get("stop_reason")),0)+1
    _wave=[]
    if WAVE_METRICS_PATH.exists():
        for line in WAVE_METRICS_PATH.read_text(encoding="utf-8").splitlines():
            try: _wave.append(json.loads(line))
            except Exception: pass
    _spt=[float(w["sec_per_output_token_longest"]) for w in _wave if w.get("sec_per_output_token_longest")]
    summ={
        "arm":arm,"reasoning_effort":ARM_CONFIG[arm]["reasoning_effort"],"think_policy":ARM_CONFIG[arm]["think_policy"],
        "tasks_terminal":n,"expected_tasks":len(ACTIVE_TASK_IDS),
        "tool_engagement_rate":_rate(lambda r:int(r.get("tool_calls") or 0)>0),
        "final_program_rate":_rate(lambda r:str(r.get("stop_reason"))=="final_program"),
        "all_train_exact_rate":_rate(lambda r:bool(r.get("all_train_exact"))),
        "budget_death_rate":_rate(lambda r:str(r.get("stop_reason")) in QUICK_BUDGET_DEATH_REASONS),
        "infra_failure_rate":_rate(lambda r:str(r.get("stop_reason")) in QUICK_INFRA_REASONS or str(r.get("status"))=="failed"),
        "turns_total":len(turns),
        "first_turn_finish":{k:sum(1 for e in first if str(e.get("finish_reason")).lower()==k) for k in ("stop","length")},
        "first_turn_natural_close_rate":(round(sum(1 for e in first if e.get("natural_think_close"))/len(first),3) if first else None),
        "first_turn_tool_call_rate":(round(sum(1 for e in first if e.get("parsed_kind")=="tool")/len(first),3) if first else None),
        "natural_close_rate_turns":(round(sum(1 for e in turns if e.get("natural_think_close"))/len(turns),3) if turns else None),
        "forced_close_turns":sum(1 for e in turns if e.get("forced_think_close")),
        "think_tokens_per_turn":{"p50":(int(np.percentile(think,50)) if think else None),"p90":(int(np.percentile(think,90)) if think else None),"max":(max(think) if think else None)},
        "turn_kinds":kinds,"stop_reasons":stops,
        # v2.4B: the terminal-commit funnel is the thing most likely to be silently broken.
        "finalizer_started_tasks":sum(1 for r in recs if bool(r.get("finalizer_started"))),
        "terminal_parse_status_counts":{s:sum(1 for r in recs if str(r.get("terminal_parse_status"))==s)
                                        for s in sorted({str(r.get("terminal_parse_status")) for r in recs if r.get("terminal_parse_status")})},
        "terminal_artifact_failure_rate":_rate(lambda r:str(r.get("stop_reason")) in QUICK_TERMINAL_ARTIFACT_FAILURE_REASONS),
        "final_guard_rejected_tasks":sum(1 for r in recs if str(r.get("verifier_status"))=="guard_rejected"),
        "executed_calls_total":int(sum(int(r.get("tool_calls") or 0) for r in recs)),
        "sec_per_output_token_p50":(round(float(np.percentile(_spt,50)),4) if _spt else None),
        "waves":len(_wave),"wave_sec_total":round(sum(float(w.get("wave_sec") or 0) for w in _wave),1),
    }
    summ["activation_pass"]=bool(summ["tool_engagement_rate"]>=VERDICT_THRESHOLDS["activation_tool_engagement_rate"])
    print("-"*116); print(f"[{arm}] QUICK READOUT (model behaviour, {n}/{len(ACTIVE_TASK_IDS)} tasks terminal)")
    print(json.dumps(summ,indent=1,default=str))
    if print_sample:
        # one concrete piece of evidence: the first tool turn seen (or, failing that, the first turn's tail)
        ev=next((e for e in turns if e.get("parsed_kind")=="tool"),None) or (first[0] if first else None)
        if ev is not None:
            raw=str(ev.get("raw_text") or ""); post=raw.split("</think>",1)[1] if "</think>" in raw else raw[-600:]
            print(f"[{arm}] sample task={ev.get('task_id')} turn={ev.get('turn')} kind={ev.get('parsed_kind')} finish={ev.get('finish_reason')} think_tokens={ev.get('think_tokens')}")
            print("    " + post.strip()[:700].replace("\n","\n    "))
    print("-"*116)
    with open(ARM_DIR/"quick_readout.json","w") as _f: json.dump(summ,_f,indent=2,default=str)
    return summ

def write_probe_verdict(quick):
    """PROBE_GPU canary verdict: which arms showed activation under the probe budget. This is the ONLY input SMOKE needs."""
    go=[a for a in ARMS if quick.get(a,{}).get("activation_pass")]
    verdict={"probe_run_id":RUN_ID,"profile":RUN_PROFILE,"tasks":len(ACTIVE_TASK_IDS),
             "probe_budget":{"per_turn_max_tokens":PER_TURN_MAX_TOKENS,"total_generated_token_cap":TOTAL_GENERATED_TOKEN_CAP,
                             "max_tool_calls":MAX_TOOL_CALLS,"max_model_turns":MAX_MODEL_TURNS},
             "activation_threshold":VERDICT_THRESHOLDS["activation_tool_engagement_rate"],
             "arms":{a:{k:quick[a].get(k) for k in ("tool_engagement_rate","first_turn_tool_call_rate","first_turn_natural_close_rate","final_program_rate","budget_death_rate","infra_failure_rate","think_tokens_per_turn","activation_pass")} for a in quick},
             "smoke_go_arms":go,"smoke_recommended":bool(go),
             "paste_into_cell1":{"PROBE_APPROVAL":{"probe_run_id":RUN_ID,"arms":go}},
             "note":"Probe budgets are smaller than P0; this verdict answers ONLY 'does the arm reach a tool call at all'. Quality is measured in SMOKE."}
    with open(WORK_ROOT/"probe_verdict.json","w") as _f: json.dump(verdict,_f,indent=2,default=str)
    print("="*116)
    if go: print(f"PROBE VERDICT: activation in {go} -> run SMOKE_GPU with PROBE_APPROVAL = {verdict['paste_into_cell1']['PROBE_APPROVAL']}")
    else:  print("PROBE VERDICT: NO arm reached a tool call under the probe budget -> do NOT spend a SMOKE run; change the design first.")
    print("="*116)
    return verdict

def run_arm(arm):
    if IS_PROBE:
        # PROBE is a re-runnable canary (interactive session): every run_arm() starts the arm from scratch.
        # (Only here — score_arm() also calls activate_arm() and must see the records.)
        shutil.rmtree(WORK_ROOT / "arms" / arm, ignore_errors=True)
    activate_arm(arm)
    print("=" * 116); print(f"ARM {arm}: {ARM_CONFIG[arm]}"); print("=" * 116)
    run_real_piv()

def run_fake_llm_scheduler_preflight_test():
    """Legacy-policy CPU regression: drive the REAL scheduler for every reasoning arm with the terminal package OFF.
    It preserves pre-v2.4 retry/REPL/two-phase coverage. Shipped terminal-policy coverage is provided separately by
    run_terminal_policy_preflight_selftest() (full trigger matrix) and run_terminal_commit_scheduler_selftest()
    (real scheduler, terminal policy ON, structured terminal transition)."""
    _g=globals()
    _saved={k:_g.get(k) for k in ["arc_llm","SamplingParams","ACTIVE_TASK_IDS","ACTIVE_TASK_SET","deadline","CURRENT_ARM",
                                  "PROMPT_RECORD_MAP","prompt_records","prompt_df",
                                  "RAW_GENERATIONS_PATH","TRACE_DIR","WAVE_METRICS_PATH","AGENT_LOOP_RUNTIME_PATH","ARM_DIR"]}
    fake_task={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]},{"input":[[5,0],[0,5]],"output":[[5,0],[0,5]]}],
               "test":[{"input":[[7,7],[8,9]]}]}
    fake_ids=["fakeA_happy","fakeB_budget_death","fakeC_wrong_tool_name","fakeD_guard_rejected","fakeE_length_salvage",
              "fakeF_natural_close_cut_midaction","fakeG_repeated_tool_code"]
    CUT1="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\n"; CUT2="puzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    TOOL_KEYS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    TOOL_LEN="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nprint(len(puzzle[\"train\"]))\n</parameter>\n</function>\n</tool_call>"
    TOOL_ROWS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nrows=len(puzzle[\"train\"])\nprint(rows)\n</parameter>\n</function>\n</tool_call>"
    TOOL_USE_ROWS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nprint(rows*2)\n</parameter>\n</function>\n</tool_call>"   # loads 'rows' defined by an earlier call
    TOOL_OS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nimport os\nprint(os.getcwd())\n</parameter>\n</function>\n</tool_call>"
    WRONG="inspect\n</think>\n<tool_call>\n<function=python2>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    FINAL="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
    def _after_think(s):   # the action part only (what a forced-close phase 2 would emit)
        return s.split("</think>",1)[1].lstrip("\n")
    def script(arm, tid, n_tool, cap, phase2):
        forced=(ARM_CONFIG[arm]["think_policy"]=="forced_close")
        if phase2=="natural_continuation":
            return (CUT2,30,"stop")
        if forced and not phase2:
            # phase 1 of the forced arm: the model never closes </think> on its own
            if tid=="fakeE_length_salvage": return (FINAL,500,"length")      # natural close inside phase 1 -> salvage path
            if tid=="fakeF_natural_close_cut_midaction": return (CUT1,40,"length") if n_tool==0 else (FINAL,400,"stop")
            return ("still reasoning about the rule ... row 0 ... row 1 ...",cap,"length")
        pick=None
        if tid=="fakeA_happy":   pick=(TOOL_KEYS,600,"stop") if n_tool==0 else ((TOOL_LEN,600,"stop") if n_tool==1 else (FINAL,400,"stop"))
        elif tid=="fakeB_budget_death":
            if forced: pick=("I think the rule is a rotation but I am not sure yet.",60,"stop")   # forced close -> prose (no action)
            else: pick=(TOOL_KEYS,600,"stop") if n_tool==0 else ("still reasoning about the rule ...",cap,"length")
        elif tid=="fakeC_wrong_tool_name": pick=(WRONG,300,"stop")
        elif tid=="fakeD_guard_rejected": pick=(TOOL_OS,300,"stop") if n_tool==0 else (FINAL,400,"stop")
        elif tid=="fakeE_length_salvage": pick=(FINAL,500,"length")
        elif tid=="fakeF_natural_close_cut_midaction": pick=(TOOL_KEYS,600,"stop") if n_tool==0 else (FINAL,400,"stop")
        elif tid=="fakeG_repeated_tool_code":   # same snippet twice (repeat) then a call that reuses the symbol it defined (stateful reuse)
            _seqG=[TOOL_ROWS,TOOL_ROWS,TOOL_USE_ROWS]; pick=(_seqG[n_tool],600,"stop") if n_tool<3 else (FINAL,400,"stop")
        else: raise RuntimeError(tid)
        if phase2:   # phase 2 emits only the post-</think> action
            return (_after_think(pick[0]) if "</think>" in pick[0] else pick[0], pick[1], pick[2])
        return pick
    class _Out:
        # v1.9: token ids are REAL tokenizer ids of the fake text (padded by repeating the last id) so decode round-trips
        def __init__(s,text,n,fr):
            s.text=text; s.finish_reason=fr
            ids=list(tokenizer.encode(text,add_special_tokens=False)) or [0]
            s.token_ids=(ids+[ids[-1]]*max(0,int(n)-len(ids)))
    class _Req:
        def __init__(s,o,pids): s.outputs=[o]; s.prompt_token_ids=pids
    class _FakeSP:
        def __init__(s,**kw): s.__dict__.update(kw)
    prefix_keys={}
    class _FakeLLM:
        def generate(s,prompts,params,use_tqdm=False):
            outs=[]
            for p,sp in zip(prompts,params):
                if isinstance(p,dict) and "prompt_token_ids" in p:
                    ids=list(p["prompt_token_ids"]); text=tokenizer.decode(ids,skip_special_tokens=False)
                elif isinstance(p,dict): text=p["prompt"]; ids=tokenizer.encode(text,add_special_tokens=False)
                else: text=p; ids=tokenizer.encode(text,add_special_tokens=False)
                tid=next(t for t in fake_ids if f"fake lineage: {t}]" in text)
                _tail=text.rstrip()
                if _tail.endswith("</think>") and (STOP_THINKING_SENTENCE in text[-800:]): phase2="forced_close"
                elif _tail.endswith("<parameter=code>"): phase2="natural_continuation"
                else: phase2=False
                t,n,fr=script(CURRENT_ARM,tid,text.count("<tool_response>"),int(sp.max_tokens),phase2)
                outs.append(_Req(_Out(t,min(int(n),int(sp.max_tokens)),fr),ids))
            return outs
    results={}
    added_task=[]
    try:
        for tid in fake_ids:
            eval_challenges[tid]=copy.deepcopy(fake_task); added_task.append(tid)
        _g["arc_llm"]=_FakeLLM(); _g["SamplingParams"]=_FakeSP
        _g["ACTIVE_TASK_IDS"]=list(fake_ids); _g["ACTIVE_TASK_SET"]=set(fake_ids)
        for arm in ARMS:
            tmp=Path(tempfile.mkdtemp(prefix=f"piv_fake_sched_{arm}_"))
            _g["CURRENT_ARM"]=arm
            # arm-specific prompt records for the fake lineages (tagged so the fake LLM can route by rendered prefix)
            prm={}
            prefix_keys.clear()
            for tid in fake_ids:
                _user=build_agent_user_prompt(fake_task)+f"\n\n[PIV PREFLIGHT fake lineage: {tid}]"
                _msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":_user}]
                _rendered,_m=render_agent_chat(_msgs)
                prm[tid]={"task_id":tid,"user_prompt":_user,"initial_messages":_msgs,"prompt":_rendered,
                          "prompt_sha256":hashlib.sha256(_rendered.encode()).hexdigest(),
                          "prompt_tokens":len(tokenizer.encode(_rendered,add_special_tokens=False)),
                          "n_train":2,"n_test":1,"render_method":_m}
                prefix_keys[tid]=_rendered[:_rendered.rfind("<|im_start|>assistant")]
            _g["PROMPT_RECORD_MAP"]=prm
            _g["ARM_DIR"]=tmp; _g["RAW_GENERATIONS_PATH"]=tmp/"raw.jsonl"; _g["TRACE_DIR"]=tmp/"traces"; (tmp/"traces").mkdir()
            _g["WAVE_METRICS_PATH"]=tmp/"wave.jsonl"; _g["AGENT_LOOP_RUNTIME_PATH"]=tmp/"rt.json"
            _g["deadline"]=time.perf_counter()+3600.0
            run_real_piv()
            recs={json.loads(l)["task_id"]:json.loads(l) for l in open(tmp/"raw.jsonl",encoding="utf-8")}
            waves=[json.loads(l) for l in open(tmp/"wave.jsonl",encoding="utf-8")]
            A,B,C,D,E,F,Gr=(recs[t] for t in fake_ids)
            forced=(ARM_CONFIG[arm]["think_policy"]=="forced_close")
            assert A["candidate_status"]=="verified" and A["tool_calls"]==2 and A["all_train_exact"] and A["test_outputs"]==[[[7,7],[8,9]]], A
            if forced: assert A["generated_tokens"]<=TOTAL_GENERATED_TOKEN_CAP and A["forced_think_closes"]==3, ("forced arm must fit 3 forced turns + final under the cap", A["generated_tokens"])
            # natural arms exhaust the 12-turn cap on cheap malformed turns; the forced arm burns 3,072 think tokens per turn and dies on budget first
            assert C["tool_calls"]==0 and C["step_retries_total"]>=2 and C["tool_errors"]>=1 and C["stop_reason"] in {"agent_budget_exhausted","invalid_finish_reason","context_or_token_budget_exhausted"}, C
            _cev=[json.loads(l) for l in open(tmp/"traces"/"fakeC_wrong_tool_name.jsonl",encoding="utf-8")]
            _run=0; _max_run=0
            for ev in _cev:
                if ev.get("step_retry"): _run+=1; _max_run=max(_max_run,_run)
                else: _run=0
            assert _max_run<=MAX_RETRIES, ("consecutive step retries exceeded MAX_RETRIES",_max_run)
            assert D["candidate_status"]=="verified" and D["tool_calls"]==0 and D["guard_rejections"]==1, D
            assert E["candidate_status"]=="verified" and E["salvaged_final_on_length"]==1, E
            assert F["candidate_status"]=="verified" and F["tool_calls"]==1, F
            assert Gr["candidate_status"]=="verified" and Gr["tool_calls"]==3 and Gr["repeated_tool_calls"]==1, Gr
            assert Gr["stateful_reuse_events"]>=1 and "rows" in Gr["stateful_reused_symbols"], ("stateful-reuse metric not exercised", Gr)
            assert all(w.get("sec_per_output_token_longest") is not None for w in waves if w["phase"]==1), waves[:1]
            if forced:
                assert A["forced_think_closes"]==3 and A["natural_think_closes"]==0, A
                assert A["injected_stop_tokens_total"]==3*len(tokenizer.encode(STOP_THINKING_TEXT,add_special_tokens=False)), A
                assert A["post_close_actions"]==["tool","tool","final"], A
                assert any(w["phase"]==2 for w in waves), "no phase-2 wave recorded"
                assert B["tool_calls"]==0 and B["forced_think_closes"]>=1 and all(x=="text" for x in B["post_close_actions"]), B
                assert B["stop_reason"] in {"no_tool_call_or_final_solution","agent_budget_exhausted"}, B
                assert E["natural_think_closes"]==1 and E["forced_think_closes"]==0, E
                _aev=[json.loads(l) for l in open(tmp/"traces"/"fakeA_happy.jsonl",encoding="utf-8")]
                assert all(ev["phase2_tokens"]>0 and ev["forced_think_close"] for ev in _aev), _aev[0]
                # replayed history must carry the stop sentence but never a literal </think>
                assert all(ev["reasoning_mode"]=="think_close" for ev in _aev), _aev[0]
                assert all(ev["think_tokens"]==ev["phase1_tokens"] for ev in _aev), "forced turn think_tokens must equal model phase-1 tokens (no injected text)"
                # natural </think> + cut action -> continuation WITHOUT injection
                assert F["natural_continuations"]==1 and F["forced_think_closes"]==0 and F["injected_stop_tokens_total"]==0, F
                _fev=[json.loads(l) for l in open(tmp/"traces"/"fakeF_natural_close_cut_midaction.jsonl",encoding="utf-8")]
                assert _fev[0]["continuation_mode"]=="natural_continuation" and _fev[0]["phase2_tokens"]>0, _fev[0]
            else:
                assert B["stop_reason"]=="invalid_finish_reason" and B["generated_tokens"]==TOTAL_GENERATED_TOKEN_CAP and B["tool_calls"]==1, B
                assert A["natural_think_closes"]==3 and A["forced_think_closes"]==0 and A["injected_stop_tokens_total"]==0, A
                assert F["candidate_status"]=="verified" and F["natural_continuations"]==0, F
            results[arm]={"ok":True,"happy":A["candidate_status"],"budget_death":B["stop_reason"],
                          "wrong_tool_max_consecutive_step_retries":_max_run,"wrong_tool_step_retries_total":C["step_retries_total"],
                          "guard_not_charged":{"tool_calls":D["tool_calls"],"guard_rejections":D["guard_rejections"]},
                          "length_salvage":E["candidate_status"],"waves":len(waves),
                          "phase2_waves":sum(1 for w in waves if w["phase"]==2),
                          "A_forced_closes":A["forced_think_closes"],"A_natural_closes":A["natural_think_closes"],
                          "A_injected_stop_tokens":A["injected_stop_tokens_total"],"B_post_close_actions":B["post_close_actions"],
                          "F_natural_continuations":F["natural_continuations"],"G_repeated_tool_calls":Gr["repeated_tool_calls"],
                          "G_stateful_reuse_events":Gr.get("stateful_reuse_events",0),"G_reused_symbols":Gr.get("stateful_reused_symbols",[])}
            # keep the evidence: real-scheduler traces/records/waves of this arm's fake run go into the bundle
            _keep=WORK_ROOT/"fake_llm_selftest"/arm
            shutil.rmtree(_keep,ignore_errors=True); shutil.copytree(tmp,_keep)
        # deadline pre-check (last arm): a deadline closer than one projected wave must fail cleanly and stop.
        (tmp/"raw.jsonl").unlink()
        _g["deadline"]=time.perf_counter()+120.0
        run_real_piv()
        recs2=[json.loads(l) for l in open(tmp/"raw.jsonl",encoding="utf-8")]
        assert len(recs2)==len(fake_ids) and all(r["status"]=="failed" and r["stop_reason"]=="global_deadline_projected" for r in recs2), recs2[:1]
        results["deadline_precheck"]="all_failed_global_deadline_projected"
    finally:
        for tid in added_task: eval_challenges.pop(tid,None)
        for k,v in _saved.items(): _g[k]=v
    with open(WORK_ROOT/"fake_llm_scheduler_selftest.json","w") as f: json.dump(results,f,indent=2)
    print("Fake-LLM REAL-scheduler PREFLIGHT self-test PASS (all arms):",json.dumps(results,indent=1))


def run_terminal_commit_scheduler_selftest():
    """v2.4 CPU fake-LLM test of the REAL scheduler terminal transition, including feedback transport."""
    _g=globals()
    _keys=["arc_llm","SamplingParams","StructuredOutputsParams","ACTIVE_TASK_IDS","ACTIVE_TASK_SET","deadline","CURRENT_ARM",
           "PROMPT_RECORD_MAP","prompt_records","prompt_df","RAW_GENERATIONS_PATH","TRACE_DIR","WAVE_METRICS_PATH",
           "AGENT_LOOP_RUNTIME_PATH","ARM_DIR","TERMINAL_POLICY_ENABLED"]
    _saved={k:_g.get(k) for k in _keys}
    tid="fakeV24_terminal"
    fake_task={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],"test":[{"input":[[5,6],[7,8]]}]}
    tool_text="inspect\n</think>\n<tool_call><function=python><parameter=code>len(puzzle['train'])</parameter></function></tool_call>"
    # v2.4B: the fake LLM must emit exactly what a real vLLM call with skip_special_tokens=False emits,
    # i.e. with the trailing chat-control token. The v2.4A version returned bare JSON, which is why this
    # self-test passed while every real terminal commit failed to parse.
    final_json=json.dumps({"solution_code":"def solution(grid):\n    return [row[:] for row in grid]"})+"<|im_end|>"
    class _Out:
        def __init__(self,text,n,fr):
            self.text=text; self.finish_reason=fr
            ids=list(tokenizer.encode(text,add_special_tokens=False)) or [0]
            self.token_ids=ids+[ids[-1]]*max(0,int(n)-len(ids))
    class _Req:
        def __init__(self,o,pids): self.outputs=[o]; self.prompt_token_ids=pids
    class _FakeSP:
        def __init__(self,**kw): self.__dict__.update(kw)
    class _FakeStructured:
        def __init__(self,**kw): self.__dict__.update(kw)
    class _FakeLLM:
        def generate(self,prompts,params,use_tqdm=False):
            out=[]
            for p,sp in zip(prompts,params):
                text=(tokenizer.decode(list(p["prompt_token_ids"]),skip_special_tokens=False)
                      if isinstance(p,dict) and "prompt_token_ids" in p
                      else (p.get("prompt","") if isinstance(p,dict) else p))
                pids=(list(p["prompt_token_ids"]) if isinstance(p,dict) and "prompt_token_ids" in p
                      else tokenizer.encode(text,add_special_tokens=False))
                if "[TERMINAL COMMIT]" in text:
                    out.append(_Req(_Out(final_json,100,"stop"),pids))
                else:
                    seen=text.count("<tool_response>")
                    n=500 if seen==0 else EXPLORATION_TOKEN_CAP-500
                    out.append(_Req(_Out(tool_text,n,"stop"),pids))
            return out
    tmp=Path(tempfile.mkdtemp(prefix="piv_v24_terminal_sched_"))
    try:
        eval_challenges[tid]=copy.deepcopy(fake_task)
        _g["arc_llm"]=_FakeLLM(); _g["SamplingParams"]=_FakeSP; _g["StructuredOutputsParams"]=_FakeStructured
        _g["TERMINAL_POLICY_ENABLED"]=True
        _g["ACTIVE_TASK_IDS"]=[tid]; _g["ACTIVE_TASK_SET"]={tid}; _g["CURRENT_ARM"]="ACT-LOW"
        user=build_agent_user_prompt(fake_task)+f"\n\n[PIV PREFLIGHT fake lineage: {tid}]"
        msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}]
        rendered,method=render_agent_chat(msgs,reasoning_effort="low")
        rec={"task_id":tid,"user_prompt":user,"initial_messages":msgs,"prompt":rendered,
             "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
             "prompt_tokens":len(tokenizer.encode(rendered,add_special_tokens=False)),
             "n_train":1,"n_test":1,"render_method":method}
        _g["PROMPT_RECORD_MAP"]={tid:rec}; _g["prompt_records"]=[rec]
        _g["prompt_df"]=pd.DataFrame([{k:v for k,v in rec.items() if k not in {"prompt","initial_messages","user_prompt"}}])
        _g["ARM_DIR"]=tmp; _g["RAW_GENERATIONS_PATH"]=tmp/"raw.jsonl"; _g["TRACE_DIR"]=tmp/"traces"; _g["TRACE_DIR"].mkdir()
        _g["WAVE_METRICS_PATH"]=tmp/"waves.jsonl"; _g["AGENT_LOOP_RUNTIME_PATH"]=tmp/"runtime.json"
        _g["deadline"]=time.perf_counter()+10000
        run_real_piv()
        rows=[json.loads(x) for x in (tmp/"raw.jsonl").read_text().splitlines() if x.strip()]
        assert len(rows)==1,rows
        r=rows[0]
        assert r["finalizer_started"] and r["terminal_parse_status"]=="ok",r
        assert "<|im_end|>" in final_json, "self-test must exercise the control-token path"
        assert r["candidate_status"]=="verified" and r["all_train_exact"] and r["test_execution_ok"],r
        assert int(r["generated_tokens"])<=TOTAL_GENERATED_TOKEN_CAP,r
        assert int(r["tool_calls"])==2,r
        assert r["feedback_present_in_rendered_prompt"] is True,r
        waves=[json.loads(x) for x in (tmp/"waves.jsonl").read_text().splitlines() if x.strip()]
        assert any(w.get("phase")=="terminal_finalizer" for w in waves),waves
        payload={"ok":True,"generated_tokens":r["generated_tokens"],"tool_calls":r["tool_calls"],
                 "terminal_parse_status":r["terminal_parse_status"],"all_train_exact":r["all_train_exact"],
                 "feedback_present_in_rendered_prompt":r["feedback_present_in_rendered_prompt"],
                 "terminal_wave_present":True}
        (WORK_ROOT/"terminal_scheduler_selftest.json").write_text(json.dumps(payload,indent=2),encoding="utf-8")
        print("v2.4 REAL-scheduler terminal-commit self-test PASS:",payload)
    finally:
        eval_challenges.pop(tid,None)
        for k,v in _saved.items(): _g[k]=v
        shutil.rmtree(tmp,ignore_errors=True)

def run_phase2_exception_preservation_selftest():
    """v2.2 regression: phase-2 engine failure must preserve phase-1 compute evidence and
    must not erase a different task's already-complete natural final program in the same batch."""
    _g=globals()
    _saved={k:_g.get(k) for k in ["arc_llm","SamplingParams","ACTIVE_TASK_IDS","ACTIVE_TASK_SET","deadline","CURRENT_ARM",
                                  "PROMPT_RECORD_MAP","prompt_records","prompt_df",
                                  "RAW_GENERATIONS_PATH","TRACE_DIR","WAVE_METRICS_PATH","AGENT_LOOP_RUNTIME_PATH","ARM_DIR"]}
    fake_task={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
    fake_final="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
    fake_reason="still reasoning about the rule and objects"
    ids=["fakeP2_final","fakeP2_cont"]
    class _Out:
        def __init__(self,text,n,fr):
            self.text=text; self.finish_reason=fr
            toks=list(tokenizer.encode(text,add_special_tokens=False)) or [0]
            self.token_ids=toks+[toks[-1]]*max(0,int(n)-len(toks))
    class _Req:
        def __init__(self,o,pids): self.outputs=[o]; self.prompt_token_ids=pids
    class _FakeSP:
        def __init__(self,**kw): self.__dict__.update(kw)
    class _BoomPhase2LLM:
        def generate(self,prompts,params,use_tqdm=False):
            # Phase 2 is token-prompt continuation and should fail synchronously.
            if prompts and all(isinstance(x,dict) and "prompt_token_ids" in x for x in prompts):
                raise RuntimeError("synthetic phase2 engine failure")
            outs=[]
            for p,sp in zip(prompts,params):
                text=(p.get("prompt","") if isinstance(p,dict) else p)
                pids=tokenizer.encode(text,add_special_tokens=False)
                tid=next(t for t in ids if f"fake lineage: {t}]" in text)
                if tid=="fakeP2_final": outs.append(_Req(_Out(fake_final,120,"stop"),pids))
                else: outs.append(_Req(_Out(fake_reason,min(1234,int(sp.max_tokens)),"length"),pids))
            return outs
    tmp=Path(tempfile.mkdtemp(prefix="piv_v22_p2fail_"))
    try:
        for tid in ids: eval_challenges[tid]=copy.deepcopy(fake_task)
        _g["arc_llm"]=_BoomPhase2LLM(); _g["SamplingParams"]=_FakeSP
        _g["ACTIVE_TASK_IDS"]=list(ids); _g["ACTIVE_TASK_SET"]=set(ids); _g["CURRENT_ARM"]="ACT-BUDGET3072"
        prm={}
        for tid in ids:
            _user=build_agent_user_prompt(fake_task)+f"\n\n[PIV PREFLIGHT fake lineage: {tid}]"
            _msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":_user}]
            _rendered,_m=render_agent_chat(_msgs,reasoning_effort="xhigh")
            prm[tid]={"task_id":tid,"user_prompt":_user,"initial_messages":_msgs,"prompt":_rendered,
                      "prompt_sha256":hashlib.sha256(_rendered.encode()).hexdigest(),
                      "prompt_tokens":len(tokenizer.encode(_rendered,add_special_tokens=False)),"n_train":1,"n_test":1,"render_method":_m}
        _g["PROMPT_RECORD_MAP"]=prm; _g["prompt_records"]=list(prm.values())
        _g["prompt_df"]=pd.DataFrame([{k:v for k,v in x.items() if k not in {"prompt","initial_messages","user_prompt"}} for x in prm.values()])
        _g["ARM_DIR"]=tmp; _g["RAW_GENERATIONS_PATH"]=tmp/"raw.jsonl"; _g["TRACE_DIR"]=tmp/"traces"; _g["TRACE_DIR"].mkdir(parents=True,exist_ok=True)
        _g["WAVE_METRICS_PATH"]=tmp/"waves.jsonl"; _g["AGENT_LOOP_RUNTIME_PATH"]=tmp/"runtime.json"; _g["deadline"]=time.perf_counter()+3600
        run_real_piv()
        recs=[json.loads(x) for x in (tmp/"raw.jsonl").read_text().splitlines() if x.strip()]
        by={r["task_id"]:r for r in recs}
        assert set(by)==set(ids), recs
        assert by["fakeP2_final"]["all_train_exact"] and by["fakeP2_final"]["test_execution_ok"] and by["fakeP2_final"]["candidate_status"]=="verified", by["fakeP2_final"]
        assert by["fakeP2_cont"]["status"]=="failed" and by["fakeP2_cont"]["stop_reason"]=="engine_generate_exception_phase2", by["fakeP2_cont"]
        assert int(by["fakeP2_cont"]["generated_tokens"])>0, by["fakeP2_cont"]
        result={"ok":True,"final_task_preserved":True,"phase1_tokens_preserved":int(by["fakeP2_cont"]["generated_tokens"]),
                "continuation_stop_reason":by["fakeP2_cont"]["stop_reason"]}
        with open(WORK_ROOT/"phase2_exception_preservation_selftest.json","w") as f: json.dump(result,f,indent=2)
        print("Phase-2 exception preservation self-test PASS:",result)
    finally:
        for tid in ids: eval_challenges.pop(tid,None)
        for k,v in _saved.items(): _g[k]=v
        shutil.rmtree(tmp,ignore_errors=True)

def run_forced_close_replay_selftest():
    """The stop-thinking sentence must survive into history WITHOUT a literal </think>, and the template must
    re-wrap it exactly as the model saw it during the forced turn."""
    stop_text=STOP_THINKING_TEXT
    merged="partial reasoning about rows"+stop_text+"<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    p=parse_agent_turn(merged)
    assert p["kind"]=="tool" and "</think>" not in p["reasoning_content"] and p["reasoning_content"].endswith(STOP_THINKING_SENTENCE), p
    msgs=[{"role":"system","content":"S"},{"role":"user","content":"U"},assistant_tool_message(p),{"role":"tool","content":"2"}]
    rendered,_=render_agent_chat(msgs)
    seg=rendered[rendered.index("<|im_start|>assistant"):]
    assert seg.count("</think>")==1 and ("partial reasoning about rows\n\n"+STOP_THINKING_SENTENCE+"\n</think>\n\n<tool_call>") in seg, seg[:400]
    with open(WORK_ROOT/"forced_close_replay_selftest.json","w") as f: json.dump({"ok":True,"single_think_close_in_replay":True},f,indent=2)
    print("Forced-close replay self-test PASS")

def run_mock_preflight():
    retry_test=run_retry_policy_preflight_test()
    fake_scheduler_test=run_fake_scheduler_preflight_test()
    multicall_budget_test=run_multicall_budget_accounting_selftest()
    with open(WORK_ROOT/"retry_policy_selftest.json","w") as f:
        json.dump({"retry_policy":retry_test,"fake_scheduler":fake_scheduler_test,
                   "multicall_budget":multicall_budget_test},f,indent=2)
    print("Retry/scheduler PREFLIGHT self-tests PASS:",retry_test,fake_scheduler_test,multicall_budget_test)
    run_scripted_preflight_agent_loop_test()
    run_terminal_policy_preflight_selftest()
    run_forced_close_replay_selftest()
    run_fake_llm_scheduler_preflight_test()
    run_terminal_commit_scheduler_selftest()
    run_phase2_exception_preservation_selftest()
    # Exercise downstream checkpoint/scoring paths for EVERY arm without pretending to measure Qwen accuracy.
    for arm in ARMS:
        activate_arm(arm)
        for tid in ACTIVE_TASK_IDS:
            state={"task_id":tid,"arm":arm,"model_turns":0,"tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,
                   "generated_tokens":0,"max_prompt_tokens":PROMPT_RECORD_MAP[tid]["prompt_tokens"],
                   "finish_reasons":["mock_preflight"],"final_code":None,"sampling_seeds":[],"phase2_sampling_seeds":[],
                   "source_retries":0,"step_retries_total":0,"retry_reasons":[],"budget_denials":0,
                   "guard_rejections":0,"salvaged_final_on_length":0,"repeated_tool_calls":0,"stop_without_think_close":0}
            rec=complete_record(state,eval_challenges[tid],None,"mock_rejected","mock_preflight")
            append_record(rec)
        print(f"[{arm}] PREFLIGHT mock records written:",len(ACTIVE_TASK_IDS))

def write_engine_bootstrap_failure_records(arm):
    """Terminalize every active task when the real engine never became usable, so reporting/bundling still completes."""
    activate_arm(arm)
    for tid in ACTIVE_TASK_IDS:
        state={"task_id":tid,"arm":arm,"model_turns":0,"tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,
               "generated_tokens":0,"max_prompt_tokens":PROMPT_RECORD_MAP[tid]["prompt_tokens"],"finish_reasons":[],"final_code":None,
               "sampling_seeds":[],"phase2_sampling_seeds":[],"source_retries":0,"step_retries_total":0,"retry_reasons":[],
               "budget_denials":0,"guard_rejections":0,"salvaged_final_on_length":0,"repeated_tool_calls":0,"stop_without_think_close":0}
        rec=complete_record(state,eval_challenges[tid],None,"failed","engine_bootstrap_failure",ENGINE_BOOTSTRAP_ERROR or ENGINE_BOOTSTRAP_STAGE)
        rec["status"]="failed"; append_record(rec)
    (ARM_DIR/"engine_bootstrap_failure_ref.json").write_text(json.dumps(ENGINE_BOOTSTRAP_STATUS,indent=2,default=str),encoding="utf-8")

# v1.9: setup cost (model load, probes, prompt rendering) measured ONCE; every arm's projection uses the same value.
ACT_FIXED_SETUP_ELAPSED_SEC = time.perf_counter() - notebook_start
ARM_QUICK = {}
if not USE_MODEL:
    run_mock_preflight()
elif not ENGINE_BOOTSTRAP_OK:
    print("Real model bootstrap failed; skipping solver generation while preserving terminal infrastructure evidence.")
    # v2.4B: without these, Cell 11 reported an engine-bootstrap failure as "adaptive pilot gate failed".
    ADAPTIVE_SMOKE_COMPLETED=False
    ADAPTIVE_STOP_REASON="engine_bootstrap_failure"
    for _arm in ARMS:
        write_engine_bootstrap_failure_records(_arm)
        ARM_QUICK[_arm]=arm_quick_summary(_arm)
else:
    if RUN_PROFILE == "ADAPTIVE_SMOKE":
        # -------------------------------------------------------------------------
        # v2.4A one-session controller.
        # Safety checks are embedded; there is no upload/review pause before GPU work.
        # -------------------------------------------------------------------------
        print("="*116)
        print("v2.4A ONE-SESSION ADAPTIVE WORKFLOW")
        print("embedded safety checks -> positive control -> PILOT2 -> automatic SMOKE8 continuation")
        print("="*116)

        # Compact regressions that directly protect the new mechanism.
        run_terminal_policy_preflight_selftest()
        run_terminal_commit_scheduler_selftest()
        run_forced_close_replay_selftest()

        _adaptive_full_ids = list(choose_smoke8_from_proxy())
        _adaptive_pilot_ids = list(TERMINAL_PILOT_TASK_IDS)
        if not set(_adaptive_pilot_ids).issubset(set(_adaptive_full_ids)):
            # v2.4B: Cell 4 already enforces this BEFORE the 27B allocation; kept as a defensive re-check.
            raise RuntimeError("Adaptive pilot tasks must be members of frozen SMOKE8.")

        # Stage A: 2-task diagnostic across BOTH real arms.
        ACTIVE_TASK_IDS = list(_adaptive_pilot_ids)
        ACTIVE_TASK_SET = set(ACTIVE_TASK_IDS)
        print("ADAPTIVE STAGE A — diagnostic tasks:", ACTIVE_TASK_IDS)
        PILOT_QUICK = {}
        for _arm in ARMS:
            run_arm(_arm)
            PILOT_QUICK[_arm] = arm_quick_summary(_arm)

        def _adaptive_interface_gate():
            pc_ok = bool((globals().get("TERMINAL_POSITIVE_CONTROL") or {}).get("ok"))
            per_arm = {}
            overall = pc_ok
            for _arm in ARMS:
                activate_arm(_arm)
                _recs = [
                    r for r in read_existing_records()
                    if r.get("arm") == _arm
                    and r.get("task_id") in set(_adaptive_pilot_ids)
                    and str(r.get("status")) in {"complete","failed"}
                ]
                _by = {str(r.get("task_id")): r for r in _recs}
                coverage = (set(_by) == set(_adaptive_pilot_ids))
                no_infra = coverage and all(
                    (not bool(r.get("infrastructure_failure")))
                    and str(r.get("status")) != "failed"
                    for r in _by.values()
                )
                # This is an INTERFACE gate, not a solve-quality gate.
                # v2.4B: the old rule was `all(finalizer_started)`. That was wrong in BOTH directions:
                #   * a pilot task that produced a good program during exploration never starts the
                #     finalizer, so a perfectly healthy run FAILED the gate;
                #   * a task whose terminal JSON failed to parse still has finalizer_started=True,
                #     status="complete" and no infrastructure_failure, so a 100% broken terminal
                #     interface PASSED the gate and burned the rest of SMOKE8. That is precisely the
                #     v2.4A <|im_end|> defect the gate was supposed to catch.
                # The correct interface question: did every pilot task end with a usable program
                # artifact, by whichever route it took?
                _finalizer_tasks=[r for r in _by.values() if bool(r.get("finalizer_started"))]
                _finalizer_parse_ok=[r for r in _finalizer_tasks if str(r.get("terminal_parse_status"))=="ok"]
                _exploration_final=[r for r in _by.values()
                                    if (not bool(r.get("finalizer_started"))) and bool(r.get("final_code"))]
                terminal_reached = coverage and bool(_finalizer_tasks)
                artifact_ok = coverage and (len(_finalizer_parse_ok)+len(_exploration_final))==len(_by)
                per_arm[_arm] = {
                    "coverage_complete": bool(coverage),
                    "no_infrastructure_failure": bool(no_infra),
                    "pilot_tasks_entering_finalizer": len(_finalizer_tasks),
                    "finalizer_artifacts_parsed_ok": len(_finalizer_parse_ok),
                    "exploration_final_programs": len(_exploration_final),
                    "terminal_parse_statuses": sorted({str(r.get("terminal_parse_status")) for r in _finalizer_tasks}),
                    "usable_artifact_for_every_pilot_task": bool(artifact_ok),
                    "terminal_finalizer_reached_any_pilot_task": bool(terminal_reached),
                    "final_programs": int(sum(1 for r in _by.values() if str(r.get("stop_reason")) == "final_program")),
                    "train_exact": int(sum(1 for r in _by.values() if bool(r.get("all_train_exact")))),
                }
                overall = overall and coverage and no_infra and artifact_ok
            result = {
                "run_id": RUN_ID,
                "profile": RUN_PROFILE,
                "positive_control_ok": bool(pc_ok),
                "pilot_task_ids": list(_adaptive_pilot_ids),
                "per_arm": per_arm,
                "interface_gate_pass": bool(overall),
                "quality_not_required_for_gate": True,
                "gate_rule": ("v2.4B: positive control ok AND, for every pilot task, complete coverage, "
                              "no infrastructure failure, and a usable program artifact from either the "
                              "terminal commit (terminal_parse_status=='ok') or exploration (final_code). "
                              "Solve correctness is explicitly NOT required."),
            }
            (WORK_ROOT/"adaptive_interface_gate.json").write_text(
                json.dumps(result,indent=2,default=str),encoding="utf-8"
            )
            return result

        ADAPTIVE_INTERFACE_GATE = _adaptive_interface_gate()
        print("ADAPTIVE INTERFACE GATE:", json.dumps(ADAPTIVE_INTERFACE_GATE, indent=2))

        if ADAPTIVE_INTERFACE_GATE["interface_gate_pass"]:
            # Stage B: same session, same model, same generation contract.
            # Pilot records are contract-compatible because TASK_SET was SMOKE8 from the start.
            ACTIVE_TASK_IDS = list(_adaptive_full_ids)
            ACTIVE_TASK_SET = set(ACTIVE_TASK_IDS)
            print("="*116)
            print("ADAPTIVE STAGE B — interface clean; continuing automatically to full frozen SMOKE8")
            print("SMOKE8:", ACTIVE_TASK_IDS)
            print("="*116)
            for _arm in ARMS:
                run_arm(_arm)  # checkpoint logic skips the two already-completed pilot tasks
                ARM_QUICK[_arm] = arm_quick_summary(_arm)
            ADAPTIVE_SMOKE_COMPLETED = True
            ADAPTIVE_STOP_REASON = None
        else:
            # Keep downstream scoring scoped to what was actually executed.
            ACTIVE_TASK_IDS = list(_adaptive_pilot_ids)
            ACTIVE_TASK_SET = set(ACTIVE_TASK_IDS)
            ARM_QUICK = dict(PILOT_QUICK)
            ADAPTIVE_SMOKE_COMPLETED = False
            ADAPTIVE_STOP_REASON = "pilot_interface_gate_failed"
            print("Adaptive gate failed: remaining SMOKE6 intentionally skipped. Evidence will still be bundled.")
    else:
        # Legacy/manual profile retained only for debugging.
        for _arm in ((globals().get("PROBE_ARMS") or ARMS) if IS_PROBE else ARMS):
            run_arm(_arm)
            ARM_QUICK[_arm] = arm_quick_summary(_arm)
        if IS_PROBE:
            PROBE_VERDICT = write_probe_verdict(ARM_QUICK)

print("Cell 8 complete. Arms:", ARMS, "| work root:", WORK_ROOT, "| fixed setup sec:", round(ACT_FIXED_SETUP_ELAPSED_SEC, 1))


In [ ]:
# =====================================================================================
# CELL 8B — v2.5 CANDIDATE-FIRST VERIFIER-FEEDBACK REPAIR (ONE ROUND, POST-BASELINE)
# Runs after the frozen v2.4G scheduler has produced candidate #1 records and before scoring.
# =====================================================================================

# v2.5A interface guard: the repair wave builds finalizer requests at module scope. Assert the factories exist
# in EVERY profile (PREFLIGHT included) so a nesting regression cannot survive until after the scheduler has run.
for _fn in ("_mk_finalizer_params","_mk_params"):
    assert callable(globals().get(_fn)), f"v2.5A: {_fn} must be defined at module level in the scheduler cell"
assert _mk_finalizer_params.__qualname__==_mk_finalizer_params.__name__, _mk_finalizer_params.__qualname__

REPAIR_ARTIFACT_ROWS=[]
V25_REPAIR_RUN_SUMMARY={"profile":RUN_PROFILE,"enabled":bool(REPAIR_POLICY_ENABLED),"arms":{},"skipped":False}


def _v25_latest_records_for_active_arm(arm):
    activate_arm(arm)
    recs,audit=audit_checkpoint_records(read_existing_records())
    latest={}
    for r in recs:
        if r.get("task_id") in ACTIVE_TASK_SET and r.get("status") in {"complete","failed"}:
            latest[str(r["task_id"])]=r
    return latest,audit

def _v25_update_record_from_selected(base, selected_idx, code, verifier, repair_meta, generated_add=0, repair_finish=None):
    r=copy.deepcopy(base)
    r.pop("_checkpoint_order",None)
    r["record_id"]=str(uuid.uuid4())
    r["record_stage"]="v2.5_candidate_first_repair"
    r["repair_rounds_allowed"]=1
    r.update(copy.deepcopy(repair_meta))
    r["selected_candidate_index"]=int(selected_idx)
    r["final_code"]=code
    r["final_code_sha256"]=hashlib.sha256((code or "").encode()).hexdigest() if code else None
    try:
        _tree=py_ast.parse(code) if code else None
        r["final_code_syntax_valid"]=bool(_tree and any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in _tree.body))
    except Exception:
        r["final_code_syntax_valid"]=False
    v=verifier or {}
    r["candidate_status"]=verifier_candidate_status(v) if code else r.get("candidate_status")
    r["train_exact_count"]=_v25_train_exact_count(v) if code else r.get("train_exact_count")
    r["verifier_invoked"]=bool(code)
    r["verifier_executed"]=bool(v.get("verifier_started")) if code else r.get("verifier_executed")
    r["all_train_exact"]=bool(v.get("all_train_exact")) if code else r.get("all_train_exact")
    r["test_execution_ok"]=bool(v.get("test_execution_ok")) if code else r.get("test_execution_ok")
    r["any_test_output_ok"]=bool(v.get("any_test_output_ok")) if code else r.get("any_test_output_ok")
    r["test_outputs"]=list(v.get("test_outputs") or []) if code else r.get("test_outputs",[])
    r["test_errors"]=list(v.get("test_errors") or []) if code else r.get("test_errors",[])
    r["verifier_status"]=v.get("verifier_status") if code else r.get("verifier_status")
    r["verifier_error"]=(v.get("train_error") or v.get("test_error") or v.get("error")) if code else r.get("verifier_error")
    _vinfra=bool(v.get("infrastructure_failure"))
    _verrs=list(v.get("infrastructure_errors") or [])
    r["verifier_infrastructure_failure"]=bool(r.get("verifier_infrastructure_failure") or _vinfra)
    _merged_verrs=[]
    for _x in list(r.get("verifier_infrastructure_errors") or [])+_verrs:
        _k=json.dumps(_x,sort_keys=True,default=str)
        if all(json.dumps(_y,sort_keys=True,default=str)!=_k for _y in _merged_verrs): _merged_verrs.append(_x)
    r["verifier_infrastructure_errors"]=_merged_verrs
    if _vinfra:
        r["infrastructure_failure"]=True
        r["infrastructure_errors"]=list(r.get("infrastructure_errors") or [])+[{"phase":"repair_verifier","error":x} for x in _verrs]
    if generated_add:
        r["generated_tokens"]=int(r.get("generated_tokens",0))+int(generated_add)
        r["model_turns"]=int(r.get("model_turns",0))+1
        r["finish_reasons"]=list(r.get("finish_reasons") or [])+[str(repair_finish)]
    # Keep frozen v2.4G gate taxonomy; repair provenance is in record_stage/repair_* fields.
    r["stop_reason"]="final_program"
    r["generation_contract_sha256"]=current_contract_sha(str(r["task_id"]))
    return r

def _v25_append_infra_invalid(base, meta, error, generated_add=0, finish=None):
    r=copy.deepcopy(base); r.pop("_checkpoint_order",None); r["record_id"]=str(uuid.uuid4())
    r["record_stage"]="v2.5_candidate_first_repair"; r.update(copy.deepcopy(meta))
    r["infrastructure_failure"]=True
    errs=list(r.get("infrastructure_errors") or []); errs.append({"phase":"repair","error":str(error)}); r["infrastructure_errors"]=errs
    if generated_add:
        r["generated_tokens"]=int(r.get("generated_tokens",0))+int(generated_add)
        r["model_turns"]=int(r.get("model_turns",0))+1
        r["finish_reasons"]=list(r.get("finish_reasons") or [])+[str(finish)]
    # Preserve known success stop taxonomy; infrastructure flags independently block promotion.
    r["stop_reason"]="final_program"
    r["generation_contract_sha256"]=current_contract_sha(str(r["task_id"]))
    append_record(r)
    return r

def _v25_phase2_768_diagnostic(arm):
    activate_arm(arm)
    rows=[]
    for tid in ACTIVE_TASK_IDS:
        p=TRACE_DIR/f"{tid}.jsonl"
        if not p.exists(): continue
        for line in p.read_text(encoding="utf-8").splitlines():
            if not line.strip(): continue
            try: e=json.loads(line)
            except Exception: continue
            if int(e.get("phase2_tokens") or 0)==768 and str(e.get("phase2_finish"))=="length" and bool(e.get("action_cap_hit")):
                rows.append({"task_id":tid,"turn":e.get("turn"),"phase1_tokens":e.get("phase1_tokens"),"phase2_tokens":768,
                             "parsed_kind":e.get("parsed_kind"),"remaining_exploration_tokens_before_request":e.get("remaining_exploration_tokens_before_request")})
    return rows

def run_v25_repair_round_for_arm(arm):
    _repair_t0=time.perf_counter()
    d=activate_arm(arm)
    latest,audit=_v25_latest_records_for_active_arm(arm)
    feedback_path=d/"repair_feedback.jsonl"
    summary_rows=[]; jobs=[]
    phase2_768=_v25_phase2_768_diagnostic(arm)
    (d/"deferred_phase2_768_diagnostic.json").write_text(json.dumps({"arm":arm,"action_reserve_tokens":ACTION_RESERVE_TOKENS,
        "action_max_tokens":ACTION_MAX_TOKENS,"behavior_changed_in_v25":False,"hits":phase2_768},indent=2),encoding="utf-8")

    for tid in ACTIVE_TASK_IDS:
        base=latest.get(tid)
        if not base:
            summary_rows.append({"arm":arm,"task_id":tid,"repair_eligible":False,"repair_status":"missing_candidate1_record"}); continue
        code1=base.get("final_code")
        if base.get("status")!="complete" or not code1 or bool(base.get("infrastructure_failure")):
            summary_rows.append({"arm":arm,"task_id":tid,"repair_eligible":False,"repair_status":"candidate1_not_eligible"}); continue
        # Re-run the fresh verifier to obtain pair-level predictions/errors for the feedback. This is train-authority evidence;
        # test execution still occurs only if the train gate is exact, per fresh_verify_solution().
        v1=fresh_verify_solution(code1,eval_challenges[tid],VERIFIER_TIMEOUT_SEC)
        c1=_v25_train_exact_count(v1)
        if bool(v1.get("infrastructure_failure")):
            meta={"repair_eligible":False,"repair_attempted":False,"repair_status":"candidate1_reverify_infrastructure_failure",
                  "candidate1_train_exact_count_reverified":c1,"candidate1_all_train_exact_reverified":bool(v1.get("all_train_exact"))}
            _v25_append_infra_invalid(base,meta,"candidate1 fresh re-verification infrastructure failure")
            summary_rows.append({"arm":arm,"task_id":tid,**meta}); continue
        if bool(v1.get("all_train_exact")):
            meta={"repair_eligible":False,"repair_attempted":False,"repair_status":"repair_not_needed_candidate1_train_exact",
                  "candidate1_train_exact_count_reverified":c1,"candidate1_all_train_exact_reverified":True,
                  "repair_success":False,"repair_delta_train_exact_pairs":0}
            r=_v25_update_record_from_selected(base,1,code1,v1,meta)
            append_record(r); summary_rows.append({"arm":arm,"task_id":tid,**meta}); continue

        fb=_v25_feedback_text(v1,eval_challenges[tid])
        rp=_v25_build_repair_messages(eval_challenges[tid],code1,fb["text"])
        pt=len(tokenizer.encode(rp["rendered"],add_special_tokens=False))
        adm=_v25_repair_admission(pt,int(base.get("generated_tokens",0)))
        meta={"repair_eligible":True,"repair_attempted":False,"repair_status":"prepared",
              "candidate1_train_exact_count_reverified":c1,"candidate1_all_train_exact_reverified":False,
              "repair_feedback_tokens":fb["tokens"],"repair_feedback_truncated":fb["truncated"],
              "repair_identity_pairs":fb["identity_pairs"],"repair_feedback_sha256":rp["feedback_sha256"],
              "repair_feedback_raw_sha256":rp["feedback_raw_sha256"],"repair_feedback_present_raw":rp["feedback_present_raw"],
              "repair_feedback_present_stripped":rp["feedback_present_stripped"],"repair_feedback_marker_present":rp["feedback_marker_present"],
              "repair_prompt_tokens":pt,"repair_cap":adm["cap"],"repair_context_headroom":adm["context_headroom"],
              "repair_remaining_total_tokens":adm["remaining_total_tokens"],"repair_render_method":rp["render_method"]}
        with open(feedback_path,"a",encoding="utf-8") as f:
            f.write(json.dumps({"run_id":RUN_ID,"arm":arm,"task_id":tid,"candidate1_code_sha256":base.get("final_code_sha256"),
                                "feedback":fb,"transport":{k:rp[k] for k in ["feedback_sha256","feedback_raw_sha256","feedback_present_raw","feedback_present_stripped","feedback_marker_present"]},
                                "prompt_tokens":pt,"admission":adm},ensure_ascii=False,default=str)+"\n")
        if not (rp["feedback_present_stripped"] and rp["feedback_marker_present"]):
            meta["repair_status"]="feedback_transport_failure"
            _v25_append_infra_invalid(base,meta,"canonical repair feedback missing from rendered prompt")
            summary_rows.append({"arm":arm,"task_id":tid,**meta}); continue
        if int(adm["cap"])<32:
            meta["repair_status"]="repair_admission_failed"
            # budget/context admission failure is mechanism coverage failure, not a backend infrastructure failure.
            r=_v25_update_record_from_selected(base,1,code1,v1,meta)
            append_record(r); summary_rows.append({"arm":arm,"task_id":tid,**meta}); continue
        seed=int(PIV_SAMPLING["seed"]+REPAIR_SEED_OFFSET)
        jobs.append({"tid":tid,"base":base,"v1":v1,"code1":code1,"feedback":fb,"rp":rp,"meta":meta,"seed":seed,"cap":int(adm["cap"])})

    if jobs:
        prompts=[j["rp"]["rendered"] for j in jobs]
        params=[_mk_finalizer_params(j["cap"],j["seed"]) for j in jobs]
        t0=time.perf_counter()
        try:
            outs=arc_llm.generate(prompts,params,use_tqdm=False)
            if len(outs)!=len(jobs):
                raise RuntimeError(f"repair output/cardinality mismatch: {len(outs)} outputs / {len(jobs)} jobs")
        except BaseException as exc:
            sec=time.perf_counter()-t0
            for j in jobs:
                m=copy.deepcopy(j["meta"]); m.update({"repair_attempted":True,"repair_status":"repair_engine_exception",
                                                       "repair_seed":j["seed"],"repair_wave_sec":sec})
                _v25_append_infra_invalid(j["base"],m,f"{type(exc).__name__}: {exc}")
                summary_rows.append({"arm":arm,"task_id":j["tid"],**m})
            outs=[]
        if outs:
            sec=time.perf_counter()-t0
            out_tokens=[]; finishes=[]; overcap=[]
            for j,req in zip(jobs,outs):
                _has_output=bool(getattr(req,"outputs",None))
                if not _has_output:
                    text=""; nt=0; fr="no_output"
                else:
                    o=req.outputs[0]; text=o.text or ""; nt=len(getattr(o,"token_ids",[]) or []); fr=str(getattr(o,"finish_reason",None))
                out_tokens.append(nt); finishes.append(fr)
                m=copy.deepcopy(j["meta"]); m.update({"repair_attempted":True,"repair_seed":j["seed"],"repair_output_tokens":int(nt),
                                                       "repair_finish_reason":fr,"repair_wave_sec":sec})
                if not _has_output:
                    m["repair_status"]="repair_no_output"
                    _v25_append_infra_invalid(j["base"],m,"repair request returned no RequestOutput.output")
                    summary_rows.append({"arm":arm,"task_id":j["tid"],**m}); continue
                if _v25_repair_output_cap_violation(nt,j["cap"]):
                    overcap.append(j["tid"]); m["repair_status"]="repair_output_exceeded_max_tokens"
                    _v25_append_infra_invalid(j["base"],m,f"observed {nt} > requested repair cap {j['cap']}",generated_add=nt,finish=fr)
                    summary_rows.append({"arm":arm,"task_id":j["tid"],**m}); continue
                parsed=parse_terminal_json_output(text)
                m["repair_parse_status"]=parsed.get("status")
                if not parsed.get("ok"):
                    m.update({"repair_status":"repair_output_parse_failed","repair_success":False,"repair_delta_train_exact_pairs":0,
                              "selected_candidate_index":1})
                    r=_v25_update_record_from_selected(j["base"],1,j["code1"],j["v1"],m,generated_add=nt,repair_finish=fr)
                    append_record(r); summary_rows.append({"arm":arm,"task_id":j["tid"],**m})
                    save_event(j["tid"],{"run_id":RUN_ID,"arm":arm,"task_id":j["tid"],"parsed_kind":"verifier_feedback_repair",
                        "repair_parse_status":parsed.get("status"),"generated_tokens":nt,"finish_reason":fr,"cap":j["cap"],
                        "feedback_sha256":j["rp"]["feedback_sha256"],"feedback_present_in_rendered_prompt":j["rp"]["feedback_present_stripped"]})
                    continue
                code2=parsed["solution_code"]
                v2=fresh_verify_solution(code2,eval_challenges[j["tid"]],VERIFIER_TIMEOUT_SEC)
                c1=_v25_train_exact_count(j["v1"]); c2=_v25_train_exact_count(v2)
                if bool(v2.get("infrastructure_failure")):
                    m.update({"repair_status":"candidate2_verifier_infrastructure_failure","candidate2_train_exact_count":c2,
                              "candidate2_all_train_exact":bool(v2.get("all_train_exact")),"candidate2_verifier_status":v2.get("verifier_status"),
                              "candidate2_code_sha256":hashlib.sha256(code2.encode()).hexdigest(),"repair_delta_train_exact_pairs":int(c2-c1),
                              "selected_candidate_index":1})
                    _v25_append_infra_invalid(j["base"],m,"candidate #2 fresh verifier infrastructure failure",generated_add=nt,finish=fr)
                    summary_rows.append({"arm":arm,"task_id":j["tid"],**m}); continue
                sel_idx,sel_code,sel_v=_v25_choose_candidate(j["code1"],j["v1"],code2,v2)
                m.update({"repair_status":"repair_verified","repair_success":bool(v2.get("all_train_exact")),
                          "candidate2_train_exact_count":c2,"candidate2_all_train_exact":bool(v2.get("all_train_exact")),
                          "candidate2_verifier_status":v2.get("verifier_status"),"candidate2_code_sha256":hashlib.sha256(code2.encode()).hexdigest(),
                          "repair_delta_train_exact_pairs":int(c2-c1),"selected_candidate_index":int(sel_idx)})
                r=_v25_update_record_from_selected(j["base"],sel_idx,sel_code,sel_v,m,generated_add=nt,repair_finish=fr)
                append_record(r); summary_rows.append({"arm":arm,"task_id":j["tid"],**m})
                save_event(j["tid"],{"run_id":RUN_ID,"arm":arm,"task_id":j["tid"],"parsed_kind":"verifier_feedback_repair",
                    "repair_parse_status":parsed.get("status"),"generated_tokens":nt,"finish_reason":fr,"cap":j["cap"],
                    "feedback_sha256":j["rp"]["feedback_sha256"],"feedback_present_in_rendered_prompt":j["rp"]["feedback_present_stripped"],
                    "candidate1_train_exact_count":c1,"candidate2_train_exact_count":c2,"candidate2_all_train_exact":bool(v2.get("all_train_exact")),
                    "selected_candidate_index":sel_idx,"verifier_status":v2.get("verifier_status")})
            append_wave_metric({"run_id":RUN_ID,"arm":arm,"phase":"verifier_feedback_repair","task_ids":[j["tid"] for j in jobs],
                                "active_count":len(jobs),"caps":[j["cap"] for j in jobs],"request_seeds":[j["seed"] for j in jobs],
                                "output_tokens":out_tokens,"finish_reasons":finishes,"overcap_task_ids":overcap,
                                "sum_output_tokens":int(sum(out_tokens)),"max_output_tokens":int(max(out_tokens) if out_tokens else 0),
                                "wave_sec":float(sec),"aggregate_output_tokens_per_sec":float(sum(out_tokens)/sec) if sec>0 else None})

    sdf=pd.DataFrame(summary_rows)
    sdf.to_csv(d/"repair_summary.csv",index=False)
    (d/"repair_summary.json").write_text(json.dumps(summary_rows,indent=2,default=str),encoding="utf-8")
    _repair_elapsed=float(time.perf_counter()-_repair_t0)
    # Include repair wall-time in the per-arm projection while keeping tasks_executed at the unique-task count.
    try:
        _rt=json.loads(AGENT_LOOP_RUNTIME_PATH.read_text(encoding="utf-8")) if AGENT_LOOP_RUNTIME_PATH.exists() else {}
    except Exception:
        _rt={}
    _rt["loop_elapsed_sec"]=float(_rt.get("loop_elapsed_sec",0.0) or 0.0)+_repair_elapsed
    _rt["repair_elapsed_sec"]=float(_rt.get("repair_elapsed_sec",0.0) or 0.0)+_repair_elapsed
    _rt["repair_rounds"]=1
    _rt["repair_tasks_eligible"]=int(sum(bool(x.get("repair_eligible")) for x in summary_rows))
    _rt["repair_tasks_attempted"]=int(sum(bool(x.get("repair_attempted")) for x in summary_rows))
    _stages=list(_rt.get("stages",[]) or [])
    _stages.append({"run_id":RUN_ID,"stage":"verifier_feedback_repair","tasks_executed":0,"repair_tasks_attempted":_rt["repair_tasks_attempted"],"loop_elapsed_sec":_repair_elapsed,"global_waves":1 if jobs else 0})
    _rt["stages"]=_stages
    AGENT_LOOP_RUNTIME_PATH.write_text(json.dumps(_rt,indent=2,default=str),encoding="utf-8")
    out={"arm":arm,"rows":len(summary_rows),"eligible":int(sum(bool(x.get("repair_eligible")) for x in summary_rows)),
         "attempted":int(sum(bool(x.get("repair_attempted")) for x in summary_rows)),
         "candidate2_all_train_exact":int(sum(bool(x.get("candidate2_all_train_exact")) for x in summary_rows)),
         "feedback_transport_failures":int(sum(x.get("repair_status")=="feedback_transport_failure" for x in summary_rows)),
         "admission_failures":int(sum(x.get("repair_status")=="repair_admission_failed" for x in summary_rows)),
         "phase2_768_length_hits_deferred":len(phase2_768),"repair_elapsed_sec":_repair_elapsed}
    return out

if RUN_PROFILE=="PREFLIGHT" or not USE_MODEL:
    V25_REPAIR_RUN_SUMMARY.update({"skipped":True,"reason":"PREFLIGHT/fake-model: helper self-tests only; no solver-quality repair inference"})
    (WORK_ROOT/"v25_repair_run_summary.json").write_text(json.dumps(V25_REPAIR_RUN_SUMMARY,indent=2,default=str),encoding="utf-8")
    print("v2.5 repair round skipped in PREFLIGHT; helper/failure-injection self-tests already passed.")
else:
    for _arm in ARMS:
        V25_REPAIR_RUN_SUMMARY["arms"][_arm]=run_v25_repair_round_for_arm(_arm)
    (WORK_ROOT/"v25_repair_run_summary.json").write_text(json.dumps(V25_REPAIR_RUN_SUMMARY,indent=2,default=str),encoding="utf-8")
    print("v2.5 REPAIR ROUND SUMMARY:",json.dumps(V25_REPAIR_RUN_SUMMARY,indent=2))


In [ ]:
# =====================================================================================
# CELL 9 — GATE + MEASUREMENT-CONTRACT FUNCTIONS (defined before any arm is scored)
# =====================================================================================
CORE_BUDGET_DEATH_REASONS={"invalid_finish_reason","agent_budget_exhausted","context_or_token_budget_exhausted",
                           # v2.4B: no token/context left to even issue the terminal request is a budget death.
                           "finalizer_admission_failed"}
# LOCAL stricter accounting: failing to finalize after the six-call P0 tool budget is also a budget death.
ALL_BUDGET_DEATH_REASONS=CORE_BUDGET_DEATH_REASONS | {"tool_budget_exhausted_after_nudge"}
INFRA_STOP_REASONS={"no_output","global_deadline","global_deadline_projected","engine_bootstrap_failure","engine_generate_exception_phase1","engine_generate_exception_phase2","engine_generate_exception_finalizer","engine_unavailable_after_phase2_exception",
                    "engine_unavailable_after_engine_exception","repl_start_failure",
    "terminal_context_reserve_breach",
    "engine_output_exceeded_max_tokens_phase1","engine_output_exceeded_max_tokens_phase2","engine_output_exceeded_max_tokens_finalizer"
}
# v2.4B: the model produced a terminal turn but no usable artifact. These were previously UNCLASSIFIED,
# so a systematic terminal-parse failure reported "0 budget deaths, 0 infrastructure failures" and the
# compute gate passed while final_program_rate was 0.
TERMINAL_ARTIFACT_FAILURE_REASONS={"terminal_finalizer_output","no_tool_call_or_final_solution","malformed_tool_xml"}
SUCCESS_STOP_REASONS={"final_program","final_program_safe_phase1_salvage_after_phase2_exception"}
KNOWN_STOP_REASONS=(ALL_BUDGET_DEATH_REASONS | INFRA_STOP_REASONS
                    | TERMINAL_ARTIFACT_FAILURE_REASONS | SUCCESS_STOP_REASONS | {"mock_preflight"})

def _json_list(value):
    if isinstance(value,list): return value
    if value is None: return []
    if isinstance(value,str):
        try:
            x=json.loads(value); return x if isinstance(x,list) else [x]
        except Exception: return [value]
    try:
        if pd.isna(value): return []
    except Exception: pass
    return [value]

def _num(v,default=0.0):
    try:
        if v is None or (isinstance(v,float) and np.isnan(v)): return default
        return float(v)
    except Exception: return default

def _summ(vals):
    vals=[float(x) for x in vals if x is not None and not (isinstance(x,float) and np.isnan(x))]
    if not vals: return {"count":0,"min":None,"median":None,"mean":None,"p90":None,"max":None}
    return {"count":len(vals),"min":float(np.min(vals)),"median":float(np.median(vals)),"mean":float(np.mean(vals)),
            "p90":float(np.percentile(vals,90)),"max":float(np.max(vals))}

def compute_smoke_gate(records, *, wave_rows=None, runtime_info=None, smoke_task_count=None):
    smoke_task_count=int(smoke_task_count or len(records) or 1)
    recs=list(records)
    stops=[str(r.get("stop_reason","")) for r in recs]
    budget_dead=[s in ALL_BUDGET_DEATH_REASONS for s in stops]
    core_budget_dead=[s in CORE_BUDGET_DEATH_REASONS for s in stops]
    final_program=[s=="final_program" for s in stops]
    death_tokens=[r.get("generated_tokens") for r,d in zip(recs,budget_dead) if d]
    _terminal_ids=[str(r.get("task_id")) for r in recs if r.get("task_id") is not None]
    terminal_coverage_complete=bool(len(recs)==smoke_task_count and len(set(_terminal_ids))==smoke_task_count
                                    and all(str(r.get("status","")) in {"complete","failed"} for r in recs))
    first_finish=[]; infra_bad=[]; tool_engaged=[]; train_exact=[]
    for r in recs:
        fr=_json_list(r.get("finish_reasons",[]))
        first_finish.append(str(fr[0]).lower() if fr else "missing")
        tool_engaged.append(int(_num(r.get("tool_calls",0))>0))
        train_exact.append(bool(r.get("all_train_exact")))
        infra_bad.append(bool(r.get("infrastructure_failure")) or str(r.get("stop_reason","")) in INFRA_STOP_REASONS or str(r.get("status",""))=="failed")
    first_turn_length_rate=float(np.mean([x=="length" for x in first_finish])) if first_finish else 1.0
    budget_death_rate=float(np.mean(budget_dead)) if recs else 1.0
    core_budget_death_rate=float(np.mean(core_budget_dead)) if recs else 1.0
    final_program_rate=float(np.mean(final_program)) if recs else 0.0
    tool_engagement_rate=float(np.mean(tool_engaged)) if recs else 0.0
    train_exact_rate=float(np.mean(train_exact)) if recs else 0.0
    infra_failure_rate=float(np.mean(infra_bad)) if recs else 1.0
    waves=list(wave_rows or [])
    wave_secs=[float(w.get("wave_sec",0.0) or 0.0) for w in waves]
    wave_sec_total=float(sum(wave_secs))
    def _wave_phase(w):
        """v2.4B CRITICAL: wave rows carry phase 1, 2 OR the string "terminal_finalizer".
        int(w.get("phase",1)) raised ValueError on the terminal wave, so compute_smoke_gate() — and
        therefore all of Cell 10 — crashed on every real TERMINAL_PILOT / ADAPTIVE_SMOKE run."""
        p=w.get("phase",1)
        if isinstance(p,bool): return None
        if isinstance(p,(int,float)): return int(p)
        s=str(p).strip().lower()
        if s in {"1","phase1"}: return 1
        if s in {"2","phase2"}: return 2
        if s in {"3","terminal_finalizer","finalizer"}: return 3
        if s in {"4","verifier_feedback_repair","repair"}: return 4
        return None
    def _spts(phase=None):
        return [float(w["wave_sec"])/float(w["max_output_tokens"]) for w in waves
                if w.get("max_output_tokens") and float(w.get("wave_sec",0) or 0)>0 and (phase is None or _wave_phase(w)==phase)]
    spts=_spts(1); spts2=_spts(2); spts3=_spts(3); spts4=_spts(4); spts_all=_spts(None)
    measured_spt=float(np.median(spts)) if spts else None
    wave_summary={"count":len(wave_secs),"phase1_count":sum(1 for w in waves if _wave_phase(w)==1),
                  "phase2_count":sum(1 for w in waves if _wave_phase(w)==2),
                  "terminal_finalizer_count":sum(1 for w in waves if _wave_phase(w)==3),
                  "terminal_finalizer_sec_per_output_token_median":(float(np.median(spts3)) if spts3 else None),
                  "repair_count":sum(1 for w in waves if _wave_phase(w)==4),
                  "repair_sec_per_output_token_median":(float(np.median(spts4)) if spts4 else None),
                  "unclassified_phase_count":sum(1 for w in waves if _wave_phase(w) is None),
                  "measured_sec_per_output_token_median":measured_spt,
                  "measured_sec_per_output_token_max":(float(np.max(spts)) if spts else None),
                  "phase2_sec_per_output_token_median":(float(np.median(spts2)) if spts2 else None),
                  "combined_sec_per_output_token_median":(float(np.median(spts_all)) if spts_all else None),
                  "projected_max_wave_minutes":((TOTAL_GENERATED_TOKEN_CAP*measured_spt)/60.0 if measured_spt else None),
                  "sum_sec":wave_sec_total,"mean_sec":float(np.mean(wave_secs)) if wave_secs else None,
                  "median_sec":float(np.median(wave_secs)) if wave_secs else None,
                  "p90_sec":float(np.percentile(wave_secs,90)) if wave_secs else None,
                  "max_sec":float(np.max(wave_secs)) if wave_secs else None}
    scale=float(len(PROXY48_TASK_IDS))/float(smoke_task_count)
    projected_generation_hours=(wave_sec_total*scale)/3600.0 if waves else None
    projected_total_hours=None; projected_total_hours_20pct_margin=None
    if runtime_info:
        pre_loop_sec=float(runtime_info.get("pre_loop_elapsed_sec",0.0) or 0.0)
        smoke_loop_sec=float(runtime_info.get("loop_elapsed_sec",0.0) or 0.0)
        # v2.4B: scale the loop time by the tasks it ACTUALLY covered. With the accumulated runtime file
        # this equals smoke_task_count, but a partial/resumed run no longer silently under-projects.
        _executed=int(runtime_info.get("tasks_executed") or smoke_task_count or 1)
        loop_scale=float(len(PROXY48_TASK_IDS))/float(max(1,_executed))
        projected_total_hours=(pre_loop_sec + smoke_loop_sec*loop_scale)/3600.0
        projected_total_hours_20pct_margin=(pre_loop_sec + smoke_loop_sec*loop_scale*1.20)/3600.0
    # v2.4B measurement-integrity diagnostics.
    artifact_fail_rate=float(np.mean([s in TERMINAL_ARTIFACT_FAILURE_REASONS for s in stops])) if recs else 0.0
    unclassified_stops=sorted({s for s in stops if s and s not in KNOWN_STOP_REASONS})
    terminal_parse_statuses=[str(r.get("terminal_parse_status")) for r in recs if r.get("terminal_parse_status")]
    terminal_parse_failures=sum(1 for s in terminal_parse_statuses if s!="ok")
    final_guard_rejected=sum(1 for r in recs if str(r.get("verifier_status"))=="guard_rejected")
    measurement_integrity_pass=bool(not unclassified_stops)
    budget_pass=bool(budget_death_rate<=0.50)
    transport_pass=bool(infra_failure_rate==0.0)
    wallclock_pass=bool(projected_total_hours_20pct_margin is not None and projected_total_hours_20pct_margin<=10.0)
    coverage_pass=bool(terminal_coverage_complete)
    gate={
        "first_turn_finish_reasons":first_finish,
        "first_turn_length_rate_diagnostic_only":first_turn_length_rate,
        "core_budget_death_reasons":sorted(CORE_BUDGET_DEATH_REASONS),"all_budget_death_reasons":sorted(ALL_BUDGET_DEATH_REASONS),
        "core_budget_death_rate":core_budget_death_rate,"budget_death_rate":budget_death_rate,
        "final_program_rate":final_program_rate,"tokens_at_budget_death":_summ(death_tokens),
        "tool_engagement_rate":tool_engagement_rate,"all_train_exact_rate":train_exact_rate,
        "infrastructure_failure_rate":infra_failure_rate,"wave_runtime":wave_summary,
        "projected_proxy48_generation_hours":projected_generation_hours,
        "projected_proxy48_total_hours":projected_total_hours,
        "projected_proxy48_total_hours_20pct_margin":projected_total_hours_20pct_margin,
        "terminal_records":len(recs),"expected_terminal_records":smoke_task_count,
        "terminal_coverage_complete":terminal_coverage_complete,
        "budget_viability_pass":budget_pass,"transport_viability_pass":transport_pass,
        "wallclock_viability_pass":wallclock_pass,"coverage_pass":coverage_pass,
        # v2.4B diagnostics
        "terminal_artifact_failure_rate":artifact_fail_rate,
        "terminal_parse_status_counts":{s:terminal_parse_statuses.count(s) for s in sorted(set(terminal_parse_statuses))},
        "terminal_parse_failure_tasks":int(terminal_parse_failures),
        "final_guard_rejected_tasks":int(final_guard_rejected),
        "unclassified_stop_reasons":unclassified_stops,
        "measurement_integrity_pass":measurement_integrity_pass,
    }
    gate["compute_gate_pass"]=bool(budget_pass and transport_pass and wallclock_pass and coverage_pass and measurement_integrity_pass)
    gate["proxy48_advisory_pass"]=gate["compute_gate_pass"]   # kept for continuity; the v1.8 verdict adds activation/productivity/quality tiers
    gate["rule"]=("Compute gate: budget_death_rate <= 0.50, no infrastructure failures, projected_proxy48_total_hours_20pct_margin <= 10.0, "
                  "complete unique terminal coverage. Promotion requires verified train quality + usable test prediction; tool engagement/productivity remain diagnostics.")
    return gate

def compute_measurement_report(arm, records, events_by_task, wave_rows, gate, scores_df, category_df):
    """Pre-registered MEASUREMENT_CONTRACT, evaluated from terminal records + per-turn trace events."""
    recs=list(records); n=max(1,len(recs))
    evs=[e for t in events_by_task.values() for e in t]
    gen_turns=[e for e in evs if e.get("parsed_kind") not in (None,"no_output")]
    tool_turns=[e for e in evs if e.get("parsed_kind")=="tool"]
    exec_calls=[c for e in tool_turns for c in (e.get("tool_calls_executed") or []) if not c.get("budget_denied")]
    exec_real=[c for c in exec_calls if not c.get("guard_rejected")]
    forced_turns=[e for e in gen_turns if e.get("forced_think_close")]
    natural_turns=[e for e in gen_turns if e.get("natural_think_close")]
    def rate(xs): return float(np.mean(xs)) if len(xs) else None
    # transitions between consecutive generated turns within a task
    trans={}
    for t,tev in events_by_task.items():
        kinds=[e.get("parsed_kind") for e in tev if e.get("parsed_kind") not in (None,"no_output")]
        for a,b in zip(kinds,kinds[1:]): trans[f"{a}->{b}"]=trans.get(f"{a}->{b}",0)+1
    retry_tokens=sum(int(e.get("generated_tokens",0) or 0) for e in evs if e.get("step_retry") or e.get("source_retry"))
    codes=[r.get("final_code_sha256") for r in recs if r.get("final_code_sha256")]
    grids=[hashlib.sha256(json.dumps(g,separators=(",",":")).encode()).hexdigest() for r in recs for g in (r.get("test_outputs") or []) if g]
    tool_hashes=[c.get("tool_code_sha16") for c in exec_real if c.get("tool_code_sha16")]
    rep={
     "arm":arm,"arm_config":ARM_CONFIG[arm],"tasks":len(recs),
     "L1_thinking_behavior":{
        "generated_turns":len(gen_turns),
        "natural_think_close_rate_turns":rate([bool(e.get("natural_think_close")) for e in gen_turns]),
        "forced_think_close_rate_turns":rate([bool(e.get("forced_think_close")) for e in gen_turns]),
        "think_tokens_per_turn":_summ([e.get("think_tokens") for e in gen_turns]),
        "think_tokens_per_task":_summ([r.get("think_tokens_total") for r in recs]),
        "phase1_tokens_per_turn":_summ([e.get("phase1_tokens") for e in gen_turns]),
        "first_turn_finish_reasons":gate["first_turn_finish_reasons"],
        "tasks_with_any_natural_close":sum(1 for r in recs if _num(r.get("natural_think_closes"))>0),
        "tasks_with_any_forced_close":sum(1 for r in recs if _num(r.get("forced_think_closes"))>0)},
     "L2_tool_activation":{
        "tool_engagement_rate_tasks":gate["tool_engagement_rate"],
        "tool_turn_rate_turns":rate([e.get("parsed_kind")=="tool" for e in gen_turns]),
        "executed_calls_total":len(exec_real),"executed_calls_per_task":_summ([r.get("tool_calls") for r in recs]),
        "first_tool_turn":_summ([r.get("first_tool_turn") for r in recs if r.get("first_tool_turn") is not None]),
        "first_tool_generated_tokens":_summ([r.get("first_tool_generated_tokens") for r in recs if r.get("first_tool_generated_tokens") is not None]),
        "tasks_never_calling_tool":sum(1 for r in recs if _num(r.get("tool_calls"))==0)},
     "L3_tool_usefulness":{
        "tool_ok_total":int(sum(_num(r.get("tool_ok_count")) for r in recs)),
        "tool_exception_total":int(sum(_num(r.get("tool_exception_count")) for r in recs)),
        "guard_rejections_total":int(sum(_num(r.get("guard_rejections")) for r in recs)),
        "tool_timeouts_total":int(sum(_num(r.get("tool_timeouts")) for r in recs)),
        "budget_denials_total":int(sum(_num(r.get("budget_denials")) for r in recs)),
        "puzzle_keys_only_calls":sum(1 for c in exec_real if (c.get("tool_code") or "").strip() in ("puzzle.keys()","print(puzzle.keys())")),
        "duplicate_tool_code_rate":(1.0-len(set(tool_hashes))/len(tool_hashes)) if tool_hashes else None},
     "L4_stateful_benefit":{"worker_restarts_total":int(sum(_num(r.get("worker_restarts")) for r in recs)),
        "tasks_with_2plus_calls":sum(1 for r in recs if _num(r.get("tool_calls"))>=2),
        "stateful_reuse_events_total":int(sum(_num(r.get("stateful_reuse_events")) for r in recs)),
        "tasks_with_stateful_reuse":sum(1 for r in recs if _num(r.get("stateful_reuse_events"))>0),
        "repeated_tool_calls_total":int(sum(_num(r.get("repeated_tool_calls")) for r in recs))},
     "L5_reason_action_transitions":{"transition_counts":trans,"turns_per_task":_summ([r.get("model_turns") for r in recs]),
        "turn_kind_totals":{k:sum(1 for e in gen_turns if e.get("parsed_kind")==k) for k in ("tool","final","text","malformed_tool")}},
     "L6_program_production":{"final_program_rate":gate["final_program_rate"],
        "final_programs":sum(1 for r in recs if r.get("final_code")),
        "syntax_valid_program_rate":rate([bool(r.get("final_code_syntax_valid")) for r in recs]),
        "salvaged_final_on_length_total":int(sum(_num(r.get("salvaged_final_on_length")) for r in recs)),
        "final_code_chars":_summ([len(r.get("final_code") or "") for r in recs if r.get("final_code")])},
     "L7_program_correctness":{"verifier_execution_rate":rate([bool(r.get("verifier_executed")) for r in recs]),
        "all_train_exact_rate":gate["all_train_exact_rate"],
        "all_train_exact_tasks":[r["task_id"] for r in recs if r.get("all_train_exact")],
        "train_exact_pairs":_summ([r.get("train_exact_count") for r in recs if r.get("train_exact_count") is not None]),
        "verifier_status_counts":{s:sum(1 for r in recs if str(r.get("verifier_status"))==s) for s in sorted({str(r.get("verifier_status")) for r in recs})}},
     "L8_test_execution":{"test_execution_ok_tasks":sum(1 for r in recs if r.get("test_execution_ok")),
        "any_test_output_ok_tasks":sum(1 for r in recs if r.get("any_test_output_ok")),
        "valid_grid_rate_outputs":float(scores_df["piv_valid"].mean()) if len(scores_df) else 0.0},
     "L9_arc_quality":{"competition_output_exact_rate":float(scores_df["piv_exact"].mean()) if len(scores_df) else 0.0,
        "all_queries_exact_task_rate":float(scores_df.groupby("task_id")["piv_exact"].min().mean()) if len(scores_df) else 0.0,
        "task_normalized_exact":float(scores_df.groupby("task_id")["piv_exact"].mean().mean()) if len(scores_df) else 0.0,
        "solved_task_ids_strict":sorted([str(t) for t,v in scores_df.groupby("task_id")["piv_exact"].min().items() if int(v)==1]) if len(scores_df) else [],
        "exact_output_pairs":int(scores_df["piv_exact"].sum()) if len(scores_df) else 0,
        "total_output_pairs":int(len(scores_df))},
     "L10_budget_behavior":{"generated_tokens_per_task":_summ([r.get("generated_tokens") for r in recs]),
        "generated_tokens_per_turn":_summ([e.get("generated_tokens") for e in gen_turns]),
        "injected_stop_tokens_total":int(sum(_num(r.get("injected_stop_tokens_total")) for r in recs)),
        "budget_death_rate":gate["budget_death_rate"],"tokens_at_budget_death":gate["tokens_at_budget_death"],
        "context_headroom_deaths":sum(1 for r in recs if r.get("stop_reason")=="context_or_token_budget_exhausted"),
        "stop_reason_counts":{s:sum(1 for r in recs if str(r.get("stop_reason"))==s) for s in sorted({str(r.get("stop_reason")) for r in recs})}},
     "L11_wall_clock":{"wave_runtime":gate["wave_runtime"],
        "projected_proxy48_total_hours":gate["projected_proxy48_total_hours"],
        "projected_proxy48_total_hours_20pct_margin":gate["projected_proxy48_total_hours_20pct_margin"]},
     "L12_batch_scheduler":{"waves":len(wave_rows),
        "active_count_per_wave":[int(w.get("active_count",0)) for w in wave_rows],
        "idle_tail_fraction_per_wave":[(1.0-float(np.mean(w.get("output_tokens") or [0]))/float(w.get("max_output_tokens") or 1)) if w.get("max_output_tokens") else None for w in wave_rows]},
     "L13_forced_close_effects":{"forced_turns":len(forced_turns),
        "natural_continuation_turns":sum(1 for e in gen_turns if e.get("continuation_mode")=="natural_continuation"),
        "action_cap_hit_turns":sum(1 for e in gen_turns if e.get("action_cap_hit")),
        "action_cap_hit_rate_of_phase2_turns":rate([bool(e.get("action_cap_hit")) for e in gen_turns if e.get("phase2_tokens")]),
        "tasks_with_action_cap_hit":sum(1 for r in recs if _num(r.get("phase2_length_turns"))>0),
        "stop_without_think_close_turns":int(sum(_num(r.get("stop_without_think_close")) for r in recs)),
        "post_close_action_counts":{k:sum(1 for e in forced_turns if e.get("post_close_action")==k) for k in ("tool","final","text","malformed_tool")},
        "natural_turn_action_counts":{k:sum(1 for e in natural_turns if e.get("parsed_kind")==k) for k in ("tool","final","text","malformed_tool")},
        "phase2_skipped_reasons":{s:sum(1 for e in gen_turns if e.get("phase2_skipped_reason")==s) for s in sorted({str(e.get("phase2_skipped_reason")) for e in gen_turns if e.get("phase2_skipped_reason")})},
        "phase2_tokens":_summ([e.get("phase2_tokens") for e in forced_turns])},
     "L14_retry_effects":{"step_retries_total":int(sum(_num(r.get("step_retries_total")) for r in recs)),
        "source_retries_total":int(sum(_num(r.get("source_retries")) for r in recs)),
        "tokens_spent_in_retried_turns":int(retry_tokens),
        "retry_reason_counts":{}},
     "L15_parser_interface":{"malformed_tool_turns":sum(1 for e in gen_turns if e.get("parsed_kind")=="malformed_tool"),
        "text_turns_no_action":sum(1 for e in gen_turns if e.get("parsed_kind")=="text"),
        "salvaged_final_on_length":int(sum(_num(r.get("salvaged_final_on_length")) for r in recs))},
     "L16_sandbox_safety":{"guard_rejections":int(sum(_num(r.get("guard_rejections")) for r in recs)),
        "tool_timeouts":int(sum(_num(r.get("tool_timeouts")) for r in recs)),
        "worker_restarts":int(sum(_num(r.get("worker_restarts")) for r in recs))},
     "L17_diversity":{"unique_final_code_hashes":len(set(codes)),"final_programs":len(codes),
        "unique_predicted_grids":len(set(grids)),"predicted_grids":len(grids),
        "unique_tool_code_hashes_per_task":_summ([r.get("unique_tool_code_hashes") for r in recs])},
     "L18_eda_slices":category_df.to_dict(orient="records") if category_df is not None else [],
     "L22_terminal_commit":{
        "terminal_artifact_failure_rate":gate.get("terminal_artifact_failure_rate"),
        "terminal_parse_failure_tasks":gate.get("terminal_parse_failure_tasks"),
        "final_guard_rejected_tasks":gate.get("final_guard_rejected_tasks"),
        "finalizer_started_tasks":sum(1 for r in recs if bool(r.get("finalizer_started"))),
        "finalizer_trigger_reasons":{s:sum(1 for r in recs if str(r.get("finalizer_trigger_reason"))==s) for s in sorted({str(r.get("finalizer_trigger_reason")) for r in recs if r.get("finalizer_trigger_reason")})},
        "finalizer_output_tokens":_summ([r.get("finalizer_output_tokens") for r in recs if r.get("finalizer_output_tokens") is not None]),
        "terminal_parse_status_counts":{s:sum(1 for r in recs if str(r.get("terminal_parse_status"))==s) for s in sorted({str(r.get("terminal_parse_status")) for r in recs if r.get("terminal_parse_status")})},
        "feedback_present_in_rendered_prompt":{str(v):sum(1 for r in recs if r.get("feedback_present_in_rendered_prompt") is v) for v in (True,False,None)},
     },
     "L20_provenance":{"generation_contract_version":GENERATION_CONTRACT_VERSION,"prompt_contract_version":PROMPT_CONTRACT_VERSION,
        "chat_template_sha256":template_sha,"engine_config_sha256":ENGINE_CONFIG_SHA256,
        "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,"seed_base":PIV_SAMPLING["seed"],
        "verifier_contract_version":VERIFIER_CONTRACT_VERSION,"verifier_timeout_contract":VERIFIER_TIMEOUT_CONTRACT},
    }
    for r in recs:
        for x in _json_list(r.get("retry_reasons",[])):
            rep["L14_retry_effects"]["retry_reason_counts"][str(x)]=rep["L14_retry_effects"]["retry_reason_counts"].get(str(x),0)+1
    return rep

def arm_verdict(gate, report):
    """v2.1 quality verdict. Tool engagement/productivity are diagnostics only. Promotion needs verified train quality + usable test prediction."""
    act=bool(gate["tool_engagement_rate"]>=VERDICT_THRESHOLDS["activation_tool_engagement_rate"])
    prod=bool(gate["final_program_rate"]>=VERDICT_THRESHOLDS["productivity_final_program_rate"] and gate["infrastructure_failure_rate"]==0.0)
    qual=bool(gate["all_train_exact_rate"]>0.0)
    usable=bool((report.get("L8_test_execution") or {}).get("any_test_output_ok_tasks",0)>0)
    comp=bool(gate["compute_gate_pass"])
    no_infra=bool(gate["infrastructure_failure_rate"]==0.0)
    promote=bool(qual and usable and comp and no_infra)
    return {"activation_pass":act,"productivity_pass":prod,"quality_signal":qual,"usable_test_signal":usable,
            "compute_gate_pass":comp,"promotion_candidate":promote,
            "tier_reached":("usable_quality" if (qual and usable) else "quality_train_only" if qual else "productivity" if prod else "activation" if act else "none")}

# v2.4B regression: a terminal_finalizer wave row must not crash the gate, and an unknown stop reason
# must be surfaced instead of silently counting as "healthy".
_v24b_waves=[{"wave_sec":10.0,"max_output_tokens":500,"phase":1},
             {"wave_sec":4.0,"max_output_tokens":300,"phase":"terminal_finalizer"},
             {"wave_sec":2.0,"max_output_tokens":100,"phase":2}]
_v24b_recs=[{"task_id":"t0","status":"complete","stop_reason":"terminal_finalizer_output",
             "finish_reasons":["stop","stop"],"generated_tokens":9000,"tool_calls":3,"all_train_exact":False,
             "terminal_parse_status":"json_parse_error:JSONDecodeError"}]
_v24b_gate=compute_smoke_gate(_v24b_recs,wave_rows=_v24b_waves,
                              runtime_info={"pre_loop_elapsed_sec":100,"loop_elapsed_sec":200,"tasks_executed":1},
                              smoke_task_count=1)
assert _v24b_gate["wave_runtime"]["terminal_finalizer_count"]==1 and _v24b_gate["wave_runtime"]["phase1_count"]==1
assert abs(_v24b_gate["wave_runtime"]["measured_sec_per_output_token_median"]-0.02)<1e-12
assert _v24b_gate["terminal_artifact_failure_rate"]==1.0 and _v24b_gate["terminal_parse_failure_tasks"]==1
assert _v24b_gate["measurement_integrity_pass"] and not _v24b_gate["unclassified_stop_reasons"]
_v24b_unknown=compute_smoke_gate([dict(_v24b_recs[0],stop_reason="brand_new_reason_v9")],wave_rows=_v24b_waves,
                                 runtime_info=None,smoke_task_count=1)
assert _v24b_unknown["unclassified_stop_reasons"]==["brand_new_reason_v9"]
assert not _v24b_unknown["measurement_integrity_pass"] and not _v24b_unknown["compute_gate_pass"]
# v2.4C: verifier infrastructure failure must invalidate the compute/infrastructure gate even with a valid final artifact.
_v24c_vi_rec=dict(_v24b_recs[0],stop_reason="final_program",terminal_parse_status="ok",
                  infrastructure_failure=True,verifier_infrastructure_failure=True,
                  verifier_status="train_launch_error")
_v24c_vi_gate=compute_smoke_gate([_v24c_vi_rec],wave_rows=_v24b_waves,
                                 runtime_info={"pre_loop_elapsed_sec":100,"loop_elapsed_sec":200,"tasks_executed":1},
                                 smoke_task_count=1)
assert _v24c_vi_gate["infrastructure_failure_rate"]==1.0 and not _v24c_vi_gate["compute_gate_pass"]

# Regression self-tests for the gate policy (Scenario G, missing terminal record, duplicate id) — unchanged from v1.7.
_scenario_g=[{"task_id":f"g{i}","status":"complete","stop_reason":"invalid_finish_reason","finish_reasons":["stop","length"],
              "generated_tokens":12288,"tool_calls":1,"all_train_exact":False} for i in range(8)]
_scenario_g_gate=compute_smoke_gate(_scenario_g,wave_rows=[],runtime_info=None,smoke_task_count=8)
assert _scenario_g_gate["first_turn_length_rate_diagnostic_only"]==0.0 and _scenario_g_gate["budget_death_rate"]==1.0 and not _scenario_g_gate["budget_viability_pass"]
_scenario_cov=[{"task_id":f"c{i}","status":"complete","stop_reason":"final_program","finish_reasons":["stop","stop"],
                "generated_tokens":3000,"tool_calls":2,"all_train_exact":True} for i in range(7)]
_scenario_cov_gate=compute_smoke_gate(_scenario_cov,wave_rows=[{"wave_sec":100.0,"max_output_tokens":500,"phase":1}],
                                      runtime_info={"pre_loop_elapsed_sec":300,"loop_elapsed_sec":600},smoke_task_count=8)
assert _scenario_cov_gate["budget_viability_pass"] and _scenario_cov_gate["wallclock_viability_pass"]
assert not _scenario_cov_gate["coverage_pass"] and not _scenario_cov_gate["compute_gate_pass"]
assert abs(_scenario_cov_gate["wave_runtime"]["measured_sec_per_output_token_median"]-0.2)<1e-9
_scenario_dup=[dict(x) for x in _scenario_cov]+[dict(_scenario_cov[0])]
_scenario_dup_gate=compute_smoke_gate(_scenario_dup,wave_rows=[{"wave_sec":100.0,"max_output_tokens":500,"phase":1}],
                                      runtime_info={"pre_loop_elapsed_sec":300,"loop_elapsed_sec":600},smoke_task_count=8)
assert not _scenario_dup_gate["terminal_coverage_complete"] and not _scenario_dup_gate["compute_gate_pass"]
# v2.1 quality-smoke verdict regressions. Activation is diagnostic; usable test evidence is required for promotion.
_empty_test_report={"L8_test_execution":{"any_test_output_ok_tasks":0}}
_usable_test_report={"L8_test_execution":{"any_test_output_ok_tasks":1}}
_v=arm_verdict({"tool_engagement_rate":1.0,"final_program_rate":0.5,"infrastructure_failure_rate":0.0,"all_train_exact_rate":0.0,"compute_gate_pass":True},_usable_test_report)
assert not _v["promotion_candidate"] and _v["tier_reached"]=="productivity"
_v_train_only=arm_verdict({"tool_engagement_rate":1.0,"final_program_rate":0.25,"infrastructure_failure_rate":0.0,"all_train_exact_rate":0.125,"compute_gate_pass":True},_empty_test_report)
assert _v_train_only["quality_signal"] and not _v_train_only["promotion_candidate"] and _v_train_only["tier_reached"]=="quality_train_only"
_v_direct=arm_verdict({"tool_engagement_rate":0.0,"final_program_rate":0.25,"infrastructure_failure_rate":0.0,"all_train_exact_rate":0.125,"compute_gate_pass":True},_usable_test_report)
assert (not _v_direct["activation_pass"]) and _v_direct["promotion_candidate"] and _v_direct["tier_reached"]=="usable_quality"
_v0=arm_verdict({"tool_engagement_rate":0.0,"final_program_rate":0.0,"infrastructure_failure_rate":0.0,"all_train_exact_rate":0.0,"compute_gate_pass":True},_empty_test_report)
assert not _v0["promotion_candidate"] and _v0["tier_reached"]=="none"
_v24d_compute_block=arm_verdict({"tool_engagement_rate":1.0,"final_program_rate":1.0,"infrastructure_failure_rate":0.0,"all_train_exact_rate":1.0,"compute_gate_pass":False},_usable_test_report)
assert _v24d_compute_block["quality_signal"] and _v24d_compute_block["usable_test_signal"] and not _v24d_compute_block["promotion_candidate"]

# v2.3 safe-recovery regressions.
# 1) A draft solution() inside unclosed reasoning is not a final candidate.
_v23_draft_text="""<think>
still reasoning
```python
def solution(grid):
    return grid
```
"""
_v23_draft_parsed=parse_agent_turn(_v23_draft_text)
assert salvage_final_code_on_length(_v23_draft_text,_v23_draft_parsed) is None
# 2) A closed action-region solution can be salvaged by the normal length contract.
_v23_closed_text="""analysis complete
</think>
```python
def solution(grid):
    return grid
```
"""
_v23_closed_parsed=parse_agent_turn(_v23_closed_text)
assert salvage_final_code_on_length(_v23_closed_text,_v23_closed_parsed) is not None
# 3) Recovered quality evidence under an engine failure remains infrastructure-invalid and cannot promote.
_v23_infra_rec={
    "task_id":"infra0","status":"complete",
    "stop_reason":"final_program_safe_phase1_salvage_after_phase2_exception",
    "finish_reasons":["length"],"generated_tokens":3472,"tool_calls":0,"all_train_exact":True,
    "infrastructure_failure":True,"phase2_tokens_unknown":True,
}
_v23_infra_gate=compute_smoke_gate([_v23_infra_rec],wave_rows=[{"wave_sec":1.0,"max_output_tokens":3472,"phase":1},
                                                               {"wave_sec":0.5,"max_output_tokens":None,"phase":2,"tokens_unknown":True}],
                                   runtime_info={"pre_loop_elapsed_sec":1.0,"loop_elapsed_sec":1.5},smoke_task_count=1)
assert _v23_infra_gate["infrastructure_failure_rate"]==1.0
assert not _v23_infra_gate["compute_gate_pass"]
_v23_infra_verdict=arm_verdict(_v23_infra_gate,{"L8_test_execution":{"any_test_output_ok_tasks":1}})
assert _v23_infra_verdict["quality_signal"] and not _v23_infra_verdict["promotion_candidate"]

with open(WORK_ROOT/"smoke_gate_policy_selftest.json","w") as _f:
    json.dump({"scenario_G":_scenario_g_gate,"scenario_missing_terminal_record":_scenario_cov_gate,
               "scenario_duplicate_terminal_id":_scenario_dup_gate,"verdict_tiers":{"productivity_not_promoted":_v,
               "train_exact_without_test_not_promoted":_v_train_only,"direct_final_without_tool_can_promote":_v_direct,"none_example":_v0},
               "v2.3_safe_recovery":{"draft_inside_unclosed_think_rejected":True,
               "closed_action_salvage_allowed":True,"infra_recovered_quality_not_promoted":_v23_infra_verdict,
               "infra_gate":_v23_infra_gate}},_f,indent=2)
print("Gate/verdict policy self-tests PASS")



In [ ]:
# =====================================================================================
# CELL 10 — PER-ARM: CONTRACT-FILTERED COVERAGE + SCORING + REGISTRY + EDA + DIAGNOSTICS + GATE + MEASUREMENT REPORT
# =====================================================================================
def weighted_mean(x,w):
    x=np.asarray(x,dtype=float); w=np.asarray(w,dtype=float); return float(np.sum(x*w)/np.sum(w)) if np.sum(w)>0 else float("nan")

def final_code_syntax_valid(code):
    if not code:
        return False
    try:
        tree=py_ast.parse(code)
    except Exception:
        return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

CATEGORY_MASKS={
 "ALL":lambda d:np.ones(len(d),dtype=bool),"tier1":lambda d:(d.proxy_tier==1).to_numpy(),"tier2":lambda d:(d.proxy_tier==2).to_numpy(),
 "tier3":lambda d:(d.proxy_tier==3).to_numpy(),"easy_static":lambda d:d.proxy_easy_static.astype(bool).to_numpy(),
 "rare":lambda d:d.proxy_rare.astype(bool).to_numpy(),"multi_query":lambda d:(d.proxy_n_queries>1).to_numpy(),
 "dynamic":lambda d:d.proxy_dynamic.astype(bool).to_numpy(),"high_ratio":lambda d:d.proxy_high_ratio.astype(bool).to_numpy(),
 "extreme_ratio":lambda d:d.proxy_extreme_ratio.astype(bool).to_numpy(),
}
weight_map=dict(zip(eval_weights["task_id"],eval_weights["weight"]))
quartile_map=dict(zip(eval_weights["task_id"],eval_weights["test_relevance_quartile"]))

def score_arm(arm):
    d=activate_arm(arm)   # re-renders arm prompts (contract SHA depends on them) and points paths at the arm dir
    raw_records=read_existing_records(); compatible_records,checkpoint_audit=audit_checkpoint_records(raw_records)
    with open(d/"checkpoint_audit.json","w") as f: json.dump(checkpoint_audit,f,indent=2)
    if not compatible_records: raise RuntimeError(f"[{arm}] No compatible PIV records")
    latest_map={}
    for r in compatible_records:
        if r.get("status") in {"complete","failed"}: latest_map[r["task_id"]]=r
    # Instrumentation-only annotations for the real quality/pilot run. They do not affect candidate admission.
    for _tid,_r in latest_map.items():
        _r["final_code_syntax_valid"] = final_code_syntax_valid(_r.get("final_code"))
        _r["verifier_executed"] = bool(_r.get("verifier_executed", False))
    latest=pd.DataFrame([latest_map[t] for t in ACTIVE_TASK_IDS if t in latest_map])
    complete_set={t for t,r in latest_map.items() if r.get("status")=="complete"}
    failed_set={t for t,r in latest_map.items() if r.get("status")=="failed"}
    missing=sorted(ACTIVE_TASK_SET-set(latest_map))
    coverage={"complete_tasks":len(complete_set),"failed_tasks":len(failed_set),"expected_tasks":len(ACTIVE_TASK_IDS),
              "missing_tasks":missing,"run_complete":len(missing)==0 and len(failed_set)==0}
    rows=[]; registry=[]
    for tid in ACTIVE_TASK_IDS:
        r=latest_map.get(tid,{})
        outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
        for qi,target in enumerate(eval_solutions[tid]):
            pred=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
            exact=int(pred is not None and grids_equal(pred,target))
            rows.append({"arm":arm,"task_id":tid,"test_index":qi,"piv_valid":int(pred is not None),"piv_exact":exact,
                         "train_exact":int(bool(r.get("all_train_exact"))),"candidate_status":r.get("candidate_status"),
                         "weight":weight_map[tid],"quartile":quartile_map[tid]})
            if pred is not None:
                registry.append({"arm":arm,"task_id":tid,"test_index":qi,"grid":pred,
                    "canonical_sha256":hashlib.sha256(json.dumps(pred,separators=(",",":"),sort_keys=False).encode()).hexdigest(),
                    "source":f"qwen3.8_27b_piv_{arm}","run_id":r.get("run_id"),"record_id":r.get("record_id"),"seed":r.get("seed"),
                    "program_sha256":r.get("final_code_sha256"),"train_exact":True,"piv_source_commit":PIV_SOURCE_COMMIT,
                    "generation_contract_sha256":r.get("generation_contract_sha256")})
    scores=pd.DataFrame(rows)
    _registry_cols=["arm","task_id","test_index","grid","canonical_sha256","source","run_id","record_id","seed",
                    "program_sha256","train_exact","piv_source_commit","generation_contract_sha256"]
    registry_df=pd.DataFrame(registry,columns=_registry_cols)
    scores.to_csv(d/"task_output_scores.csv",index=False); registry_df.to_csv(d/"candidate_registry.csv",index=False)
    raw_acc=float(scores.piv_exact.mean()); weighted_acc=weighted_mean(scores.piv_exact,scores.weight)
    valid_rate=float(scores.piv_valid.mean())
    # v2.1 scoring contract: Kaggle averages exactness over task test outputs, not strict all-query task success.
    _score_contract_demo=pd.DataFrame({"task_id":["a","a","b"],"piv_exact":[1,0,1]})
    assert abs(float(_score_contract_demo.piv_exact.mean())-(2/3))<1e-12
    assert abs(float(_score_contract_demo.groupby("task_id").piv_exact.min().mean())-0.5)<1e-12
    train_perfect_rate=float(np.mean([bool(latest_map.get(tid,{}).get("all_train_exact")) for tid in ACTIVE_TASK_IDS]))
    mode_summary=pd.DataFrame([{"branch":f"piv_{arm}","output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,
                                "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_perfect_rate,
                                "tasks":len(ACTIVE_TASK_IDS),"test_outputs":len(scores)}])
    mode_summary.to_csv(d/"mode_summary.csv",index=False)
    task_scores=scores.groupby("task_id")["piv_exact"].min().rename("piv_all_queries").reset_index()
    task_scores.to_csv(d/"strict_task_level_scores.csv",index=False)

    # C1 v2.1 matched-quality task table: one row per frozen task, directly comparable across arms.
    _score_by_task=scores.groupby("task_id").agg(
        valid_test_outputs=("piv_valid","sum"),
        exact_test_outputs=("piv_exact","sum"),
        n_scored_outputs=("piv_exact","size"),
        all_queries_exact=("piv_exact","min"),
    ).reset_index().set_index("task_id")
    tq=[]
    for tid in ACTIVE_TASK_IDS:
        r=latest_map.get(tid,{})
        sr=_score_by_task.loc[tid] if tid in _score_by_task.index else None
        tq.append({
            "arm":arm,"task_id":tid,"status":r.get("status"),"stop_reason":r.get("stop_reason"),
            "tool_engaged":int(_num(r.get("tool_calls"))>0),"tool_calls":int(_num(r.get("tool_calls"))),
            "tool_ok_count":int(_num(r.get("tool_ok_count"))),"tool_errors":int(_num(r.get("tool_errors"))),
            "tool_timeouts":int(_num(r.get("tool_timeouts"))),"guard_rejections":int(_num(r.get("guard_rejections"))),
            "final_program":int(bool(r.get("final_code"))),"syntax_valid_program":int(bool(r.get("final_code_syntax_valid"))),
            "verifier_executed":int(bool(r.get("verifier_executed"))),
            "train_exact_pairs":r.get("train_exact_count"),"n_train":r.get("n_train"),
            "all_train_exact":int(bool(r.get("all_train_exact"))),
            "test_execution_ok":int(bool(r.get("test_execution_ok"))),
            "valid_test_outputs":(int(sr["valid_test_outputs"]) if sr is not None else 0),
            "n_test_outputs":(int(sr["n_scored_outputs"]) if sr is not None else int(r.get("n_test") or 0)),
            "all_queries_exact":(int(sr["all_queries_exact"]) if sr is not None else 0),
            "finalizer_started":int(bool(r.get("finalizer_started"))),
            "finalizer_trigger_reason":r.get("finalizer_trigger_reason"),
            "terminal_parse_status":r.get("terminal_parse_status"),
            "model_turns":r.get("model_turns"),"first_tool_turn":r.get("first_tool_turn"),
            "generated_tokens":r.get("generated_tokens"),"think_tokens_total":r.get("think_tokens_total"),
            "source_retries":r.get("source_retries"),"step_retries_total":r.get("step_retries_total"),
            "worker_restarts":r.get("worker_restarts"),"verifier_status":r.get("verifier_status"),
            "repair_eligible":bool(r.get("repair_eligible",False)),
            "repair_attempted":bool(r.get("repair_attempted",False)),
            "repair_status":r.get("repair_status"),
            "candidate1_train_exact_count_reverified":r.get("candidate1_train_exact_count_reverified"),
            "candidate2_train_exact_count":r.get("candidate2_train_exact_count"),
            "candidate2_all_train_exact":bool(r.get("candidate2_all_train_exact",False)),
            "repair_delta_train_exact_pairs":r.get("repair_delta_train_exact_pairs"),
            "repair_feedback_tokens":r.get("repair_feedback_tokens"),
            "repair_feedback_present_stripped":r.get("repair_feedback_present_stripped"),
            "selected_candidate_index":r.get("selected_candidate_index",1),
        })
    task_quality=pd.DataFrame(tq)
    task_quality.to_csv(d/"task_quality.csv",index=False)
    # EDA slices
    score_task=scores.groupby("task_id").agg(piv_exact=("piv_exact","mean"),valid=("piv_valid","mean"),train_exact=("train_exact","max")).reset_index()
    # v2.4B: profile_task_table is frozen at SMOKE8 in Cell 4. If the adaptive gate stopped the run at
    # the 2 pilot tasks, the unfiltered merge reported category slices over 8 tasks while only 2 were scored.
    cat=profile_task_table[profile_task_table["task_id"].isin(ACTIVE_TASK_IDS)].merge(score_task,on="task_id",how="left")
    crow=[]
    for name,fn in CATEGORY_MASKS.items():
        sub=cat.loc[fn(cat)]
        crow.append({"arm":arm,"category":name,"tasks":len(sub),"mean_output_exact":float(sub.piv_exact.mean()) if len(sub) else np.nan,
                     "mean_valid":float(sub.valid.mean()) if len(sub) else np.nan,"train_exact_task_rate":float(sub.train_exact.mean()) if len(sub) else np.nan})
    category_scores=pd.DataFrame(crow); category_scores.to_csv(d/"category_scores.csv",index=False)
    # diagnostics
    diag_cols=["task_id","status","candidate_status","stop_reason","model_turns","tool_calls","tool_errors","tool_timeouts","worker_restarts",
               "generated_tokens","max_rendered_prompt_tokens","think_tokens_total","natural_think_closes","forced_think_closes",
               "injected_stop_tokens_total","first_tool_turn","tool_ok_count","tool_exception_count","guard_rejections",
               "all_train_exact","test_execution_ok","any_test_output_ok","source_retries","step_retries_total","budget_denials","verifier_status",
               "finalizer_started","finalizer_trigger_reason","terminal_parse_status","finalizer_output_tokens"]
    failure_df=pd.DataFrame([{c:r.get(c) for c in diag_cols} for _,r in latest.iterrows()]) if len(latest) else pd.DataFrame(columns=diag_cols)
    failure_df.to_csv(d/"agent_diagnostics.csv",index=False)
    # events, waves, runtime -> gate + measurement report
    events_by_task={}
    for tid in ACTIVE_TASK_IDS:
        p=TRACE_DIR/f"{tid}.jsonl"
        if p.exists():
            events_by_task[tid]=[json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
            events_by_task[tid]=[e for e in events_by_task[tid] if e.get("run_id")==latest_map.get(tid,{}).get("run_id")]
    wave_rows=[]
    if WAVE_METRICS_PATH.exists():
        for ln in WAVE_METRICS_PATH.read_text(encoding="utf-8").splitlines():
            if ln.strip():
                try:
                    w=json.loads(ln)
                    if w.get("run_id")==RUN_ID: wave_rows.append(w)
                except Exception: pass
    runtime_info=None
    if AGENT_LOOP_RUNTIME_PATH.exists():
        try: runtime_info=json.loads(AGENT_LOOP_RUNTIME_PATH.read_text(encoding="utf-8"))
        except Exception: runtime_info=None
    _records=[latest_map[t] for t in ACTIVE_TASK_IDS if t in latest_map]
    gate=compute_smoke_gate(_records,wave_rows=wave_rows,runtime_info=runtime_info,smoke_task_count=len(ACTIVE_TASK_IDS))
    _pf=proxy_source_feat.set_index("task_id")
    gate["task_composition"]={"task_ids":list(ACTIVE_TASK_IDS),
        "tier_counts":{f"tier{k}":int(sum(int(_pf.loc[t,"proxy_tier"])==k for t in ACTIVE_TASK_IDS)) for k in (1,2,3)},
        "multi_query":int(sum(int(_pf.loc[t,"proxy_n_queries"])>1 for t in ACTIVE_TASK_IDS)),
        "proxy48_tier_counts":{f"tier{k}":int(sum(int(_pf.loc[t,"proxy_tier"])==k for t in PROXY48_TASK_IDS)) for k in (1,2,3)}}
    report=compute_measurement_report(arm,_records,events_by_task,wave_rows,gate,scores,category_scores)
    verdict=arm_verdict(gate,report)
    with open(d/"smoke_gate.json","w") as f: json.dump(gate,f,indent=2)
    with open(d/"measurement_report.json","w") as f: json.dump(report,f,indent=2,default=str)
    with open(d/"verdict.json","w") as f: json.dump(verdict,f,indent=2)
    print(f"\n[{arm}] coverage={coverage} | exact={raw_acc:.4f} | train_perfect={train_perfect_rate:.4f} | tool_engagement={gate['tool_engagement_rate']:.2f} | final_program={gate['final_program_rate']:.2f} | budget_death={gate['budget_death_rate']:.2f} | tier={verdict['tier_reached']}")
    return {"arm":arm,"dir":d,"latest_map":latest_map,"latest":latest,"coverage":coverage,"checkpoint_audit":checkpoint_audit,
            "scores":scores,"registry":registry_df,"mode_summary":mode_summary,"task_scores":task_scores,"category_scores":category_scores,
            "diagnostics":failure_df,"gate":gate,"report":report,"verdict":verdict,"raw_acc":raw_acc,"weighted_acc":weighted_acc,
            "valid_rate":valid_rate,"train_perfect_rate":train_perfect_rate,
            "syntax_valid_program_rate":float(task_quality["syntax_valid_program"].mean()) if len(task_quality) else 0.0,
            "verifier_execution_rate":float(task_quality["verifier_executed"].mean()) if len(task_quality) else 0.0,
            "competition_output_exact_rate":raw_acc,
            "all_queries_exact_task_rate":float(task_quality["all_queries_exact"].mean()) if len(task_quality) else 0.0,
            "task_quality":task_quality,"runtime_info":runtime_info,"waves":len(wave_rows)}

ARM_RESULTS={arm:score_arm(arm) for arm in ARMS}



In [ ]:
# =====================================================================================
# CELL 11 — MATCHED C1 QUALITY COMPARISON + QUALITY-FIRST RECOMMENDATION
# =====================================================================================
def _arm_row(arm, res):
    g=res["gate"]; r=res["report"]; v=res["verdict"]
    return {
        "arm":arm,"reasoning_effort":ARM_CONFIG[arm]["reasoning_effort"],"think_policy":ARM_CONFIG[arm]["think_policy"],
        "think_budget":ARM_CONFIG[arm]["think_budget"],"tasks":g["terminal_records"],
        "tool_engagement_rate":g["tool_engagement_rate"],
        "final_program_rate":g["final_program_rate"],
        "syntax_valid_program_rate":res["syntax_valid_program_rate"],
        "verifier_execution_rate":res["verifier_execution_rate"],
        "all_train_exact_rate":g["all_train_exact_rate"],
        "valid_test_output_rate":res["valid_rate"],
        "competition_output_exact_rate":res["competition_output_exact_rate"],
        "all_queries_exact_task_rate":res["all_queries_exact_task_rate"],
        "output_weighted_exact":res["raw_acc"],
        "budget_death_rate":g["budget_death_rate"],
        "infrastructure_failure_rate":g["infrastructure_failure_rate"],
        "first_turn_length_rate":g["first_turn_length_rate_diagnostic_only"],
        "natural_close_rate_turns":r["L1_thinking_behavior"]["natural_think_close_rate_turns"],
        "forced_close_rate_turns":r["L1_thinking_behavior"]["forced_think_close_rate_turns"],
        "think_tokens_per_turn_median":r["L1_thinking_behavior"]["think_tokens_per_turn"]["median"],
        "executed_calls_per_task_mean":r["L2_tool_activation"]["executed_calls_per_task"]["mean"],
        "generated_tokens_per_task_mean":r["L10_budget_behavior"]["generated_tokens_per_task"]["mean"],
        "sec_per_output_token":g["wave_runtime"]["measured_sec_per_output_token_median"],
        "phase2_sec_per_output_token":g["wave_runtime"].get("phase2_sec_per_output_token_median"),
        "action_cap_hit_rate":r["L13_forced_close_effects"].get("action_cap_hit_rate_of_phase2_turns"),
        "natural_continuation_turns":r["L13_forced_close_effects"].get("natural_continuation_turns"),
        "stateful_reuse_events_total":r["L4_stateful_benefit"].get("stateful_reuse_events_total"),
        "repeated_tool_calls_total":r["L4_stateful_benefit"].get("repeated_tool_calls_total"),
        "projected_proxy48_hours_20pct":g["projected_proxy48_total_hours_20pct_margin"],
        "compute_gate_pass":g["compute_gate_pass"],
        "activation_pass":v["activation_pass"],"productivity_pass":v["productivity_pass"],
        "quality_signal":v["quality_signal"],"promotion_candidate":v["promotion_candidate"],"tier_reached":v["tier_reached"],
    }

comparison_rows=[dict(FROZEN_XHIGH_REFERENCE,**{"arm":"ACT-XHIGH (frozen v1.7 SMOKE reference, not rerun)"})]
comparison_rows+=[_arm_row(a,ARM_RESULTS[a]) for a in ARMS]
comparison_df=pd.DataFrame(comparison_rows)
comparison_df.to_csv(WORK_ROOT/"arm_comparison.csv",index=False)
with open(WORK_ROOT/"arm_comparison.json","w") as f:
    json.dump(comparison_rows,f,indent=2,default=str)

# Matched per-task LOW vs BUDGET3072 table.
_quality_frames=[]
for a in ARMS:
    q=ARM_RESULTS[a]["task_quality"].copy()
    q.columns=[("task_id" if c=="task_id" else f"{a}__{c}") for c in q.columns]
    _quality_frames.append(q)
matched_task_quality=_quality_frames[0]
for q in _quality_frames[1:]:
    matched_task_quality=matched_task_quality.merge(q,on="task_id",how="outer",validate="one_to_one")
matched_task_quality=matched_task_quality.sort_values("task_id").reset_index(drop=True)
matched_task_quality.to_csv(WORK_ROOT/"matched_task_quality.csv",index=False)

# Deltas vs frozen XHIGH are historical context only. Solver-quality decisions use matched current arms.
deltas={}
for a in ARMS:
    row=_arm_row(a,ARM_RESULTS[a]); deltas[a]={}
    for k in ("tool_engagement_rate","final_program_rate","all_train_exact_rate","output_weighted_exact",
              "budget_death_rate","first_turn_length_rate","generated_tokens_per_task_mean"):
        try: deltas[a][f"delta_{k}_vs_xhigh"]=float(row[k])-float(FROZEN_XHIGH_REFERENCE[k])
        except Exception: deltas[a][f"delta_{k}_vs_xhigh"]=None
with open(WORK_ROOT/"arm_deltas_vs_frozen_xhigh.json","w") as f:
    json.dump(deltas,f,indent=2)

# Quality-first ranking. LOW never gates BUDGET3072, and BUDGET3072 never gates LOW.
# Ordering: competition output-level exact -> all-train-exact -> usable/valid execution -> compute/stability diagnostics.
def _rank_key(a):
    r=_arm_row(a,ARM_RESULTS[a])
    return (
        r["competition_output_exact_rate"],
        r["all_train_exact_rate"],
        r["valid_test_output_rate"],
        r["verifier_execution_rate"],
        r["syntax_valid_program_rate"],
        r["final_program_rate"],
        int(r["compute_gate_pass"]),
        -r["infrastructure_failure_rate"],
        -r["budget_death_rate"],
        -(r["generated_tokens_per_task_mean"] or 0),
    )

ranked=sorted(ARMS,key=_rank_key,reverse=True)
if RUN_PROFILE == "PREFLIGHT":
    retained_train_exact_arms=[]
    retained_quality_arms=[]
    promotion_eligible_arms=[]  # v2.4E: required by RECOMMENDATION below in PREFLIGHT too
    reproduction_required_arms=[]
    proxy_ready_arms=[]
    infrastructure_invalid_arms=[]
    comparison_valid=False
    _adaptive_done=False
    _adaptive_gate=None
    _matched_scope_complete=False
    _recommendation_scope_valid=False
    decision=("PREFLIGHT ONLY: mock/fake-model records validate the harness. "
              "No model-behaviour or solver-quality recommendation is permitted.")
else:
    retained_train_exact_arms=[a for a in ranked if ARM_RESULTS[a]["verdict"]["quality_signal"]]
    retained_quality_arms=[a for a in ranked if ARM_RESULTS[a]["verdict"]["quality_signal"] and ARM_RESULTS[a]["verdict"]["usable_test_signal"]]
    promotion_eligible_arms=[a for a in ranked if ARM_RESULTS[a]["verdict"]["promotion_candidate"]]
    infrastructure_invalid_arms=[a for a in ranked if (not ARM_RESULTS[a]["gate"].get("terminal_coverage_complete")) or ARM_RESULTS[a]["gate"].get("infrastructure_failure_rate",1.0)>0.0]
    comparison_valid=(len(infrastructure_invalid_arms)==0)

    _adaptive_done = bool(globals().get("ADAPTIVE_SMOKE_COMPLETED", RUN_PROFILE != "ADAPTIVE_SMOKE"))
    _adaptive_gate = globals().get("ADAPTIVE_INTERFACE_GATE")
    _matched_scope_complete = bool(RUN_PROFILE != "ADAPTIVE_SMOKE" or _adaptive_done)
    _recommendation_scope_valid = bool(comparison_valid and _matched_scope_complete)

    # Result-driven policy:
    # only a COMPLETED infrastructure-valid SMOKE8 can authorize Proxy48.
    # v2.5: a first non-zero repaired quality signal must be reproduced once before Proxy48.
    reproduction_required_arms = (
        list(promotion_eligible_arms)
        if (RUN_PROFILE=="ADAPTIVE_SMOKE" and _adaptive_done)
        else []
    )
    proxy_ready_arms = []

    if RUN_PROFILE=="ADAPTIVE_SMOKE" and not _adaptive_done:
        # v2.4B: an engine-bootstrap failure is not a pilot gate failure; say which one happened.
        _adaptive_stop=str(globals().get("ADAPTIVE_STOP_REASON") or "pilot_interface_gate_failed")
        decision=(f"ADAPTIVE RUN DID NOT REACH FULL SMOKE8 (stop_reason={_adaptive_stop}); the remaining tasks were "
                  "intentionally skipped in the same GPU session. Fix only the observed failure; do not start a "
                  "reserve-tuning or review loop.")
    elif infrastructure_invalid_arms:
        decision=("SMOKE EXECUTION/INFRASTRUCTURE INVALID FOR: "+", ".join(infrastructure_invalid_arms)+". "
                  "Do not interpret those arms as no-quality results.")
    elif RUN_PROFILE=="TERMINAL_PILOT":
        if any(ARM_RESULTS[a]["gate"]["final_program_rate"]>0 for a in ranked):
            decision=("TERMINAL_PILOT PRODUCED EXECUTABLE FINAL PROGRAMS. "
                      "This is a 2-task interface diagnostic only; it does not authorize Proxy48.")
        else:
            decision=("TERMINAL_PILOT COMPLETED BUT FINAL PROGRAM RATE REMAINS ZERO. "
                      "Inspect the terminal interface before scaling.")
    elif reproduction_required_arms:
        decision=("v2.5 PRODUCED FRESH-VERIFIER TRAIN-EXACT + USABLE TEST OUTPUT. "
                  "Reproduce the exact working configuration once before Proxy48; do not add another mechanism yet.")
    elif any(ARM_RESULTS[a]["gate"]["final_program_rate"]>0 for a in ranked):
        decision=("v2.5 COMPLETED WITH EXECUTABLE PROGRAMS BUT REPAIR DID NOT YIELD USABLE VERIFIED QUALITY. "
                  "Inspect semantic failure taxonomy / candidate-search strategy next; do not reopen terminal reserve tuning or add reviewer/pass@K yet.")
    else:
        decision=("FULL SMOKE8 COMPLETED BUT FINAL PROGRAM RATE REMAINS ZERO. "
                  "The terminal-policy package did not solve the current artifact bottleneck; rethink the interface rather than patching parameters serially.")

RECOMMENDATION={
    "ranked_arms":ranked,
    "retained_train_exact_arms":retained_train_exact_arms,
    "retained_quality_arms":retained_quality_arms,
    "promotion_eligible_arms":promotion_eligible_arms,
    "reproduction_required_arms":reproduction_required_arms,
    "proxy_ready_arms":proxy_ready_arms,
    "recommended_arm":(proxy_ready_arms[0] if proxy_ready_arms else None),
    "provisional_best_valid_arm":(proxy_ready_arms[0] if proxy_ready_arms else None),
    "infrastructure_invalid_arms":infrastructure_invalid_arms,
    "matched_comparison_valid":_recommendation_scope_valid,
    "infrastructure_comparison_valid":comparison_valid,
    "full_smoke_scope_complete":_matched_scope_complete,
    "decision":decision,
    "decision_hierarchy":["integrity","executable_final_program","all_train_exact","usable_test_prediction",
                          "competition_output_exact_rate","compute_efficiency","stability_failure_modes"],
    "independent_arm_rule":"ACT-BUDGET3072 is never gated on ACT-LOW succeeding, and vice versa.",
    "probe_approval":PROBE_APPROVAL,
    "thresholds":VERDICT_THRESHOLDS,
    "frozen_reference":"ACT-XHIGH = v1.7 SMOKE (tool_engagement 0.00, budget_death 1.00)",
    "frozen_reference_bundle_sha256":FROZEN_XHIGH_REFERENCE.get("bundle_sha256"),
    "quality_smoke_run_id":RUN_ID,
    "pilot_scope":("adaptive PILOT2 -> automatic SMOKE8 in one session" if RUN_PROFILE=="ADAPTIVE_SMOKE" else "legacy/manual profile"),
    "engine_intervention":{"enforce_eager":[True,False],"disable_custom_all_reduce":True,"enable_prefix_caching":False},
    "recommendation_sha256_input":None,
}
_rec_payload=json.dumps(RECOMMENDATION,sort_keys=True,separators=(",",":"),default=str).encode()
RECOMMENDATION["recommendation_sha256"]=hashlib.sha256(_rec_payload).hexdigest()
with open(WORK_ROOT/"recommendation.json","w") as f:
    json.dump(RECOMMENDATION,f,indent=2)

print("C1 MATCHED ARM COMPARISON (frozen XHIGH reference first):")
display_cols=["arm","tool_engagement_rate","final_program_rate","syntax_valid_program_rate","verifier_execution_rate",
              "all_train_exact_rate","valid_test_output_rate","competition_output_exact_rate","all_queries_exact_task_rate","output_weighted_exact",
              "budget_death_rate","generated_tokens_per_task_mean","sec_per_output_token","compute_gate_pass","tier_reached"]
display(comparison_df[[c for c in display_cols if c in comparison_df.columns]])
print("\nMATCHED TASK QUALITY:")
display(matched_task_quality)
print("\nRECOMMENDATION:",json.dumps(RECOMMENDATION,indent=2))


In [ ]:
# =====================================================================================
# CELL 12 — PER-ARM SOURCE-STYLE REPLAY PAYLOADS (None when the train gate rejects)
# =====================================================================================
PRED_PIV={}
for arm in ARMS:
    lm=ARM_RESULTS[arm]["latest_map"]; pred={}
    for tid in ACTIVE_TASK_IDS:
        r=lm.get(tid,{})
        outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
        pred[tid]=[]
        for qi in range(len(eval_challenges[tid]["test"])):
            candidate=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
            pred[tid].append({"attempt_1":candidate,"attempt_2":None})
    with open(ARM_RESULTS[arm]["dir"]/"predictions_piv_replay.json","w") as f: json.dump(pred,f)
    PRED_PIV[arm]=pred
print("Wrote per-arm predictions_piv_replay.json:",list(PRED_PIV))



In [ ]:
# =====================================================================================
# CELL 13 — REPLAY SELF-CHECK PER ARM
# =====================================================================================
for arm in ARMS:
    pass1=[]
    for tid in ACTIVE_TASK_IDS:
        for qi,target in enumerate(eval_solutions[tid]):
            a=PRED_PIV[arm][tid][qi]["attempt_1"]; pass1.append(int(a is not None and grids_equal(a,target)))
    replay=float(np.mean(pass1))
    assert abs(replay-ARM_RESULTS[arm]["raw_acc"])<1e-12,(arm,replay,ARM_RESULTS[arm]["raw_acc"])
    print(f"[{arm}] Replay self-check PASS:",replay)



In [ ]:
# =====================================================================================
# CELL 14 — METADATA + REPORT (PROVEN BY PREFLIGHT vs MEASURED ON GPU)
# =====================================================================================
total_elapsed_sec=time.perf_counter()-notebook_start
with open(WORK_ROOT/"measurement_contract.json","w") as f: json.dump(MEASUREMENT_CONTRACT,f,indent=2)
source_verification={
 "source_commit":PIV_SOURCE_COMMIT,"external_sources":PIV_EXTERNAL_SOURCES,"verified_source_files":PIV_SOURCE_FILES,
 "source_faithful":{"persistent_repl":True,"puzzle_preloaded":True,"tool_errors_to_model":True,"source_style_retries":True,
                    "self_contained_solution":True,"fresh_train_verifier":True,"all_train_exact_gate":True,
                    "separate_test_execution_state":True,"test_outputs_from_execution":True,"single_lineage_per_arm":True},
 "c1_causal_intervention":{
   "donor_notebook_sha256":DONOR_NOTEBOOK_SHA256,
   "activation_probe_run_id":ACTIVATION_PROBE_RUN_ID,
   "throughput_smoke_run_id":THROUGHPUT_SMOKE_RUN_ID,
   "throughput_evidence_sha256":THROUGHPUT_EVIDENCE_SHA256,
   "only_engine_diff":{"enforce_eager":[True,False]},
   "disable_custom_all_reduce":True,
   "enable_prefix_caching":False,
   "disable_log_stats_copied_from_probe":False,
   "quality_arms":QUALITY_SMOKE_ARMS,
 },
 "intentional_local_adaptations":{
   "openai_responses_server":"replaced by direct vLLM + Qwen official tool chat template",
   "docker_ipybox":"replaced by persistent local subprocess + hard kill/restart + Python audit-hook host isolation",
   "source_timeout_120s":"P0 starts at 10s tool / 20s verifier per portability note",
   "source_public_model":"Qwen3.8-27B-FP8; LOW vs BUDGET3072 reasoning policy is compared under one C1 engine session",
   "grid_gate":"strengthened to rectangular integer 0-9 and 1..30 dimensions",
   "budget":"6 executed tool calls (guard-rejected snippets not charged), 12 total model turns, 12,288 total generated tokens/task; in TERMINAL_PILOT exploration is capped at 8,192 and 4,096 + one turn slot are protected for terminal commit",
   "finish_reason_gate":"length/incomplete exploration generations retry the same tool-loop step only while the protected terminal reserve remains untouched",
   "terminal_commit_v24":"one-way terminal state; no tools; enable_thinking=False; structured JSON solution_code; fresh verifier remains correctness authority",
   "per_turn_ceiling":"natural arms: remaining total budget; forced_close arm: THINK budget then ACTION_MAX_TOKENS after the documented Qwen stop-thinking sentence",
   "forced_close":"Qwen documented two-stage thinking-budget technique, token-exact continuation of the same request; injected tokens accounted separately",
   "partial_test_outputs":"train-perfect programs retain successful test outputs even if another test input raises",
   "batch_caps":"per-request max_tokens; no minimum-cap cross-task starvation",
   "approved_library_probe":AVAILABLE_P0_LIBRARIES,
   "stratified_smoke8":"1 tier3 / 2 tier2 / 5 tier1",
   "prefix_caching":("enabled" if ENABLE_PREFIX_CACHING else "disabled (frozen C1 terminal-pilot setting)"),
 },
 "not_in_this_revision":["reviewer","second lineage","PTQ deterministic repair","NVARC seeding","judge","evolution","state compaction ablation","seed diversification","native-vLLM-thinking-budget switch"],
  "known_residual_risk":"A native synchronous vLLM/CUDA generate hard-hang cannot be forcibly timed out from this in-process notebook; raised exceptions are terminalized in v2.1.",
}
with open(WORK_ROOT/"source_level_verification.json","w") as f: json.dump(source_verification,f,indent=2)
proven_by_preflight=["Qwen tool-XML parser (incl. fence unwrap, prose-mention tolerance, rehearsal-in-think guard)",
    "persistent REPL semantics, protected puzzle, SystemExit survival, dead-worker restart",
    "fresh exact verifier: train gate before test execution, per-test partial outputs, sentinel protocol",
    "two-level retry accounting (max 2 consecutive step retries), guard-rejection not charged, dynamic multi-call budget",
    "forced-close construction: byte-exact Qwen early-stopping text; stop sentence replays in history with exactly one </think>",
    "natural </think> cut at the think cap -> token-exact continuation WITHOUT injection; exact </think>-id think/post token split",
    "repeated identical tool snippets counted (no KeyError); AST stateful-reuse metric; ranked winner selection",
    "real scheduler under a fake LLM for every arm incl. two-phase forced-close path and projected-deadline stop",
    "gate policy: Scenario-G, missing/duplicate terminal record, usable-test-required promotion, tool activation diagnostic only",
    "fresh verifier: test timeout preserves train-exact evidence; guard rejection does not claim subprocess execution",
    "competition scoring: output-level exact rate distinguished from strict all-query task diagnostic",
    "v2.4 terminal reserve/admission invariants, no-tools non-thinking renderer, structured terminal parser, and real-scheduler fake-LLM terminal transition",
    "v2.4C: terminal parser tolerates skip_special_tokens=False control tokens; indented tool payloads dedented; "
    "gate survives the terminal_finalizer wave row; every terminal stop reason is classified; "
    "engine output-count mismatch and batch aborts leave no task without a record"]
V24G_FIXES=[
 "per-request max_tokens overrun is quarantined to the offending task instead of aborting the entire vLLM wave/arm",
 "output cardinality mismatch remains wave-fatal because request/output mapping cannot be trusted",
 "phase1/phase2/finalizer cap-violation records preserve observed token counts and are classified infrastructure-invalid",
 "wave metrics expose overcap_task_ids so backend contract drift is visible without corrupting healthy neighbour tasks",
]
V24F_FIXES=[
 "PREFLIGHT terminal-policy selftest explicitly exercises the shipped terminal helper under temporary terminal-policy ON state",
 "production terminal-trigger predicates share one helper with a PREFLIGHT trigger matrix: exploration cap, final turn slot, tool budget, retry exhaustion, admission reserve",
 "engine cardinality guard also rejects any RequestOutput whose token_ids exceed its requested max_tokens cap",
 "invalid-finish retry accounting no longer counts a retry when the just-finished turn already crossed a terminal threshold",
 "TERMINAL_PILOT recommendation text no longer claims FULL SMOKE8",
 "ACT-BUDGET3072 remains intentionally frozen at 3072; reduced exploration-turn count under the 8192 cap is documented as policy behavior, not retuned",
]
V24E_FIXES=[
 "preload malformed-tool regression now exercises a committed post-</think> action while unclosed tool-like text is separately asserted to remain scratch reasoning",
 "PREFLIGHT initializes promotion_eligible_arms so recommendation construction cannot raise NameError",
]
V24D_FIXES=[
 "unclosed exploration reasoning cannot be parsed as a committed final action",
 "latest discarded incomplete reasoning is token-bounded and handed to terminal commit without shrinking the 4096-token output reserve",
 "invalid-finish/malformed/no-action retry exhaustion transitions to terminal commit instead of ending with reserve unused",
 "REPL startup exceptions become terminal infrastructure-failure records so Run All can continue to evidence bundling",
 "REPL execute infrastructure_error is propagated even if a later recovered candidate verifies",
 "terminal JSON raw_decode ignores surrounding control tokens without mutating control-token-looking literals inside solution_code",
 "GPU positive control verifies its generated identity program with the same fresh verifier",
 "Proxy48-ready and recommended arms require promotion_candidate, including compute gate and no-infrastructure conditions",
]
V24C_FIXES=[
 "verifier infrastructure propagation: launch/process/protocol/host verifier failures invalidate record + arm compute evidence",
 "terminal context hardening: tool feedback is token-fitted against the actual rendered finalizer prompt to preserve 4096 output tokens",
 "partial adaptive PILOT2 evidence cannot populate recommended_arm / provisional_best_valid_arm / matched comparison validity",
 "P0 terminal JSON parse: strip <|im_end|>-style control tokens before json.loads (Cell 6/7/8)",
 "P0 compute_smoke_gate: int(phase) crashed on the terminal_finalizer wave row (Cell 9)",
 "P1 agent-loop runtime accumulated across adaptive stages so Proxy-48 projection is not optimistic (Cell 8/9)",
 "P1 tool-call <parameter=code> payload dedent (Cell 6)",
 "P1 exploration_cap_v24 uses the current arm's think budget (Cell 8)",
 "P1 unclassified terminal stop reasons now counted and gate-blocking (Cell 9)",
 "P1 undefined _probe_go NameError in the PROBE report path (Cell 14)",
 "P2 engine output-count mismatch terminalized instead of looping to the deadline (Cell 8)",
 "P2 batch aborts terminalize every remaining task (coverage no longer silently incomplete) (Cell 8)",
 "P2 _phase2_failed_tids reset per wave (Cell 8)",
 "P2 adaptive pilot-in-SMOKE8 precheck before the 27B allocation (Cell 4)",
 "P2 prompts rendered only for ACTIVE tasks instead of all 120 evaluation tasks (Cell 4)",
 "P3 fresh_verify_solution always returns 'ok'; per-request prompt_tokens logged correctly; report labels fixed",
]
measured_only_on_gpu=["natural thinking length per arm","tool engagement / first tool turn","post-forced-close action mix",
    "final-program, syntax-valid, verifier-execution and train-exact rates","valid test execution and official exact tasks",
    "sec/output-token and projected wall-clock",
    "terminal positive-control backend behavior, finalizer completion/parse/verifier funnel, feedback presence in rendered terminal prompt"]
run_metadata={
 "experiment":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"task_set":TASK_SET,"accuracy_is_real":ACCURACY_IS_REAL,"run_id":RUN_ID,
 "arms":ARMS,"arm_config":ARM_CONFIG,"action_max_tokens":ACTION_MAX_TOKENS,"stop_thinking_sentence":STOP_THINKING_SENTENCE,
 "active_task_ids":ACTIVE_TASK_IDS,"proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,
 "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
 "prompt_contract_version":PROMPT_CONTRACT_VERSION,"seed":SEED,"model_path":str(MODEL_PATH),"model_type":model_cfg.get("model_type"),
 "runtime_stack":RUNTIME_STACK,"model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
 "engine_config_sha256":ENGINE_CONFIG_SHA256,"chat_template_sha256":template_sha,"tool_schema_sha256":TOOL_SCHEMA_SHA256,
 "agent_budget":AGENT_BUDGET,"agent_budget_sha256":AGENT_BUDGET_SHA256,"sampling":PIV_SAMPLING,"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
 "model_load_sec":model_load_sec,"warmup_sec":warmup_sec,"total_elapsed_sec":total_elapsed_sec,"max_model_len":MAX_MODEL_LEN,
 "frozen_xhigh_reference":FROZEN_XHIGH_REFERENCE,"verdict_thresholds":VERDICT_THRESHOLDS,
 "probe_approval":PROBE_APPROVAL,"is_probe":bool(IS_PROBE),"qwen_reasoning_effort_contract":QWEN_REASONING_EFFORT_CONTRACT,
 "act_fixed_setup_elapsed_sec":float(ACT_FIXED_SETUP_ELAPSED_SEC),"action_reserve_tokens":ACTION_RESERVE_TOKENS,
 "terminal_commit":{"enabled":TERMINAL_POLICY_ENABLED,"exploration_token_cap":EXPLORATION_TOKEN_CAP,"final_reserve_tokens":FINAL_RESERVE_TOKENS,"context_reserve_tokens":FINALIZER_CONTEXT_RESERVE_TOKENS,"finalizer_max_tokens":FINALIZER_MAX_TOKENS,"json_schema":FINALIZER_JSON_SCHEMA,"sampling":FINALIZER_SAMPLING,"positive_control":globals().get("TERMINAL_POSITIVE_CONTROL")},
 "v25_repair":{"enabled":REPAIR_POLICY_ENABLED,"rounds":REPAIR_ROUNDS,"feedback_max_tokens":REPAIR_FEEDBACK_MAX_TOKENS,"max_output_tokens":REPAIR_MAX_TOKENS,"seed_offset":REPAIR_SEED_OFFSET,"run_summary":globals().get("V25_REPAIR_RUN_SUMMARY"),"v24g_baseline_bundle_sha256":V24G_EVIDENCE_BUNDLE_SHA256,"budget_action_reserve_frozen":ACTION_RESERVE_TOKENS,"budget_preregistration":REPAIR_BUDGET_PREREGISTRATION},
 "adaptive_controller":{"enabled":RUN_PROFILE=="ADAPTIVE_SMOKE",
                        "pilot_task_ids":TERMINAL_PILOT_TASK_IDS,
                        "interface_gate":globals().get("ADAPTIVE_INTERFACE_GATE"),
                        "full_smoke_completed":globals().get("ADAPTIVE_SMOKE_COMPLETED"),
                        "stop_reason":globals().get("ADAPTIVE_STOP_REASON")},

 "per_arm":{a:{"coverage":ARM_RESULTS[a]["coverage"],"checkpoint_audit":ARM_RESULTS[a]["checkpoint_audit"],
              "output_weighted_exact":ARM_RESULTS[a]["raw_acc"],"train_perfect_program_rate":ARM_RESULTS[a]["train_perfect_rate"],
              "valid_grid_rate":ARM_RESULTS[a]["valid_rate"],"syntax_valid_program_rate":ARM_RESULTS[a]["syntax_valid_program_rate"],
               "verifier_execution_rate":ARM_RESULTS[a]["verifier_execution_rate"],"competition_output_exact_rate":ARM_RESULTS[a]["competition_output_exact_rate"],
               "all_queries_exact_task_rate":ARM_RESULTS[a]["all_queries_exact_task_rate"],
               "gate":ARM_RESULTS[a]["gate"],"verdict":ARM_RESULTS[a]["verdict"],
              "runtime":ARM_RESULTS[a]["runtime_info"]} for a in ARMS},
 "recommendation":RECOMMENDATION,"proven_by_preflight":proven_by_preflight,"measured_only_on_gpu":measured_only_on_gpu,
 "v24g_fixes":V24G_FIXES,
 "v24f_fixes":V24F_FIXES,
 "v24e_fixes":V24E_FIXES,
 "v24d_fixes":V24D_FIXES,
 "v24c_fixes":V24C_FIXES,
 "qwen_terminal_history_keeps_tool_calls":globals().get("QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS"),
 "source_verification":source_verification,
}
with open(WORK_ROOT/"run_metadata.json","w") as f: json.dump(run_metadata,f,indent=2,default=str)
report=[f"# {EXPERIMENT_ID} — {RUN_PROFILE}","",f"- Run ID: `{RUN_ID}` | arms: **{', '.join(ARMS)}** | tasks: **{len(ACTIVE_TASK_IDS)}** | accuracy is real: **{ACCURACY_IS_REAL}**",
        f"- Frozen reference: ACT-XHIGH (v1.7 SMOKE) tool_engagement=0.00, budget_death=1.00, 0.160 s/token","",
        "## PROVEN BY PREFLIGHT (CPU, fake model) — harness correctness only"]+[f"- {x}" for x in proven_by_preflight]+["",
        "## MEASURED ONLY ON REAL GPU — model behaviour"]+[f"- {x}" for x in measured_only_on_gpu]+["","## Arm comparison",""]
report.append("| arm | tool_eng | final_prog | syntax_valid | verifier_exec | train_exact | valid_test | competition_output_exact | strict_all_query_task_exact | budget_death | gen_tok/task | sec/tok | tier | promotion_candidate |")
report.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
for row in comparison_rows:
    def _f(v,fmt="{:.2f}"):
        try: return fmt.format(float(v))
        except Exception: return str(v)
    report.append(f"| {row['arm']} | {_f(row.get('tool_engagement_rate'))} | {_f(row.get('final_program_rate'))} | {_f(row.get('syntax_valid_program_rate'))} | {_f(row.get('verifier_execution_rate'))} | {_f(row.get('all_train_exact_rate'))} | {_f(row.get('valid_test_output_rate'))} | {_f(row.get('competition_output_exact_rate'),'{:.4f}')} | {_f(row.get('all_queries_exact_task_rate'))} | {_f(row.get('budget_death_rate'))} | {_f(row.get('generated_tokens_per_task_mean'),'{:.0f}')} | {_f(row.get('sec_per_output_token'),'{:.4f}')} | {row.get('tier_reached')} | {row.get('promotion_candidate')} |")
if IS_PROBE:
    report+=["",f"## PROBE canary ({PER_TURN_MAX_TOKENS} per-turn / {TOTAL_GENERATED_TOKEN_CAP} total tokens, {MAX_TOOL_CALLS} calls, {MAX_MODEL_TURNS} turns — NOT P0 budgets)","",
             f"- arms with activation: **{(globals().get('PROBE_VERDICT') or {}).get('smoke_go_arms')}**",
             "- projected Proxy-48 hours in the table above are for PROBE budgets and must not be read as P0 projections.",
             "- PROBE answers only: does the arm reach a tool call at all? Productivity/quality are measured in SMOKE_GPU."]
report+=["",f"## Recommendation","",f"- {RECOMMENDATION['decision']}",f"- ranked arms: {RECOMMENDATION['ranked_arms']}","",
         "## Files","- `arm_comparison.csv/json`, `matched_task_quality.csv`, `arm_deltas_vs_frozen_xhigh.json`, `recommendation.json`, `measurement_contract.json`, `v25_repair_run_summary.json`",
         "- per arm: `arms/<arm>/{task_quality.csv, repair_summary.csv/json, repair_feedback.jsonl, deferred_phase2_768_diagnostic.json, smoke_gate.json, measurement_report.json, verdict.json, agent_diagnostics.csv, task_output_scores.csv, raw_generations.jsonl, traces/, wave_metrics.jsonl}`"]
(WORK_ROOT/"REPORT_FOR_CHATGPT.md").write_text("\n".join(report),encoding="utf-8")
print("\n".join(report))



In [ ]:
# =====================================================================================
# CELL 15 — MANIFEST + BUNDLE (all arms)
# =====================================================================================
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()
compact_cols=["run_id","record_id","arm","task_id","status","candidate_status","stop_reason","seed","n_train","n_test",
    "repeated_tool_calls","stateful_reuse_events","natural_continuations","phase2_length_turns","stop_without_think_close",
    "model_turns","tool_calls","tool_errors","tool_timeouts","worker_restarts","source_retries","step_retries_total","budget_denials",
    "guard_rejections","salvaged_final_on_length","think_tokens_total","natural_think_closes","forced_think_closes","injected_stop_tokens_total",
    "first_tool_turn","tool_ok_count","tool_exception_count","unique_tool_code_hashes","generated_tokens","generated_tokens_observed_lower_bound","phase2_tokens_unknown","infrastructure_failure","infrastructure_errors","verifier_infrastructure_failure","verifier_infrastructure_errors","max_rendered_prompt_tokens","finalizer_started","finalizer_trigger_reason","finalizer_prompt_tokens","finalizer_output_tokens","finalizer_finish_reason","terminal_parse_status","terminal_render_method","terminal_prompt_sha256","feedback_present_in_rendered_prompt",
    "final_code_syntax_valid","verifier_invoked","verifier_executed","all_train_exact","test_execution_ok","any_test_output_ok","verifier_status","final_code_sha256","generation_contract_sha256",
    "record_stage","repair_eligible","repair_attempted","repair_status","repair_feedback_tokens","repair_feedback_sha256","repair_feedback_present_stripped","repair_feedback_marker_present","repair_prompt_tokens","repair_cap","repair_output_tokens","repair_finish_reason","candidate1_train_exact_count_reverified","candidate2_train_exact_count","candidate2_all_train_exact","repair_delta_train_exact_pairs","selected_candidate_index",
    "phase2_sampling_seeds","sampling_config_sha256","agent_budget_sha256","model_artifact_fingerprint_sha256","runtime_stack_sha256","engine_config_sha256","piv_source_commit"]
for arm in ARMS:
    lt=ARM_RESULTS[arm]["latest"]
    cols=[c for c in compact_cols if c in lt.columns]
    (lt[cols] if len(lt) else pd.DataFrame(columns=cols)).to_csv(ARM_RESULTS[arm]["dir"]/"generation_summary.csv",index=False)
manifest=[]
for p in sorted(WORK_ROOT.rglob('*')):
    if p.is_file() and p.name!="MANIFEST.csv": manifest.append({"file":str(p.relative_to(WORK_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(WORK_ROOT/"MANIFEST.csv",index=False)
ZIP_PATH=Path("/kaggle/working")/BUNDLE_NAME
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob('*')):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(WORK_ROOT)))
print("="*116); print(EXPERIMENT_ID,RUN_PROFILE,"PIPELINE COMPLETE"); print("Bundle:",ZIP_PATH); print("ZIP MB:",round(ZIP_PATH.stat().st_size/1024**2,2))
print("UPLOAD bundle + executed notebook (with outputs).")


## Run sequence — PIV-Q27-C1 v2.5 Candidate-First Verifier-Feedback Repair

Default remains:

```python
RUN_PROFILE = "ADAPTIVE_SMOKE"
```

Use one fresh Kaggle **4×L4**, Internet Off, Run All. The notebook runs the frozen v2.4G candidate-#1 path first, then exactly one structured verifier-feedback repair round inside the **same 12,288 generated-token ceiling**.

### v2.5 invariants

- Terminal/finalizer mechanics are frozen from v2.4G.
- LOW and BUDGET3072 both run; neither gates the other.
- Candidate #1 is fresh-reverified to obtain pair-level diagnostics.
- Repair feedback is training-side only and <=600 tokenizer tokens: dims, wrong-cell count + bounded mismatch sample, exception last line, and identity-output flag.
- Feedback transport requires canonical `strip()` hash + exact rendered-prompt presence; raw trailing whitespace is not an invalidation criterion.
- Repair call advertises no tools, uses `enable_thinking=False`, and emits the same structured JSON `solution_code`.
- Candidate #2 is fresh-verified; tests execute only after all training pairs are exact.
- Candidate #1 is retained if candidate #2 does not improve train-exact count.
- `ACTION_RESERVE_TOKENS=768` is deliberately unchanged. The known 768-token BUDGET late-action truncations are logged, not repaired here, to keep v2.5 repair-only.
- No Proxy48 from a first success: reproduce any non-zero train-exact + usable-test result once under the exact same configuration, then scale.

Expected bundle: `PIV_Q27_C1_V25_ADAPTIVE_SMOKE_BUNDLE.zip`
